# CHANRG Paper Track v5 — کنترل LM تصادفی + منحنی مقیاس + تحلیل حفظ‌کردن

**ادامهٔ v4.** v4 نشان داد: RNAfold ما با ادبیات هم‌تراز است (Δ≤0.014)، و سودِ LM تقریباً کاملاً درون‌توزیعی است (تعامل Test×OOD ≈ +0.136، CI [0.044, 0.205]، یک seed).
v5 این یافته را برای ادعاپذیری آماده می‌کند و «چرا» را می‌آزماید.

## تغییرات v5
| # | افزوده | چه می‌پرسد |
|---|---|---|
| 1 | **P2r + P5r:** arm `randlm+phys` (RiNALMo با وزن تصادفی) | سود LM از *دانش پیش‌آموزش* است یا فقط از یک ورودی پرابعاد؟ (C6 و C7) |
| 2 | **P5s + P10:** منحنی مقیاس (۱۵۰۰/۳۰۰۰/۶۰۰۰/۱۲۰۰۰) | آیا سود LM در ID با داده رشد می‌کند و در OOD نه؟ (تفاوت شیب با CI) |
| 3 | **P10:** تحلیل حفظ‌کردن | سود LM بر حسب تعداد نمونهٔ همان family در train (۰ = OOD) |
| 4 | **P9 (به‌روز):** تعامل برای سه تفکیک؛ Holm روی کل آزمون‌ها (C6/C7 هم) | — |

## ترتیب اجرا (همان سشن v4 اگر زنده است؛ وگرنه از اول)
1. سلول تنظیمات → **P0 را دوباره اجرا کنید** (plib به‌روز می‌شود). اگر سشن زنده است P1–P4 و seed 0 از P5 skip می‌شوند.
2. **P5** (seedهای ۱ و ۲ اجرا می‌شوند: ≈ ۱ ساعت).
3. **P2r → P5r** (≈ ۱۵–۲۵ دقیقه embedding + ≈ ۳۰ دقیقه آموزش برای ۳ seed).
4. **P5s** (≈ ۱–۱.۵ ساعت؛ برای کوتاه‌کردن `RNA_SCALE_SEEDS="0"`).
5. **P9 و P10** (CPU، چند دقیقه).
P6/P6b اختیاری‌اند.

**برای من بفرستید:** خروجی کامل P9 و P10 + جدول انتهای P5 و P5r + خروجی P5s (جدول آخر).
زمان‌ها تخمین‌اند؛ لاگ‌ها دقیق‌ترند. پیش‌فرض‌ها: ۳ seed برای arms اصلی و randlm، ۲ seed برای منحنی مقیاس.


## نسخهٔ Drive-backed (v6)

هر سلول خروجی‌اش را در Google Drive ذخیره می‌کند و اگر از قبل باشد از همان‌جا می‌خواند (سلول‌های PAR طولانی دوباره اجرا نمی‌شوند):

1. **سلول تنظیمات** → `DRIVE_RUN` (پوشهٔ اجرا در `MyDrive`) و `SAVE_EMB` (ذخیرهٔ embeddingها ≈۵GB؛ پیش‌فرض روشن).
2. **P0** → Drive mount می‌شود؛ `plib.py` و `drive.py` نوشته می‌شوند؛ `env.json` در Drive ذخیره می‌شود. P0 همیشه اجرا می‌شود (نصب/gateها).
3. هر سلول بعدی سه کار می‌کند: (۱) artifactهای مرحله را از Drive به `/content` بازیابی می‌کند؛ (۲) اگر مرحله قبلاً با **همین تنظیمات** کامل شده باشد
   (فینگرپرینت = `config_paper.json` + همهٔ متغیرهای `RNA_*` + نسخهٔ plib)، سلول **skip** می‌شود و لاگ ذخیره‌شدهٔ همان مرحله چاپ می‌شود (جدول‌ها و اعداد بدون اجرای دوباره دیده می‌شوند)؛
   (۳) در پایان — و حتی روی خطا یا قطع کوتی، اگر سلول تا آخر برسد — artifactها و لاگ به Drive آپلود و marker نوشته می‌شود.
4. ساختار پوشه: `MyDrive/chanrg_paper_v5/full/{artifacts,_done,logs}` (اجرای `quick` جدا است). مسیرهای داخل `artifacts` همان ساختار `/content/rna_paper_v4` است.

**چه چیزی در Drive نیست:** مدل RiNALMo-giga (کش HuggingFace) و دیتاست خام CHANRG — در هر سشن نو دوباره از HF دانلود می‌شوند (وقت/پهنای‌باند می‌خواهد، کوتی GPU نه).
embeddingها *هستند* چون گران‌ترین بخش محاسبه‌اند؛ برای خاموش‌کردن آپلودشان `SAVE_EMB=False` را در سلول تنظیمات بگذارید.

**اجبار به اجرای مجدد یک مرحله:** فایل marker آن مرحله (`.../full/_done/<stage>.json`) را در Drive پاک کنید، یا `DRIVE_RUN` را عوض کنید.
اگر کپ‌ها (`RNA_TRAIN_N` …) یا `RNA_EPOCHS/RNA_SEEDS/RNA_SCALE_*` عوض شوند، فینگرپرینت عوض می‌شود و مرحله‌های وابسته خودکار از نو اجرا می‌شوند.

**ادامه بعد از تموم شدن کوتی:** سلول تنظیمات → P0 → از همان مرحله‌ای که مانده‌اید ادامه دهید؛ مراحل کامل‌شده skip و نیمه‌کاره از آخرین checkpoint
(chunkهای P3، progressهای P2/P2r، checkpointهای per-seed در P5/P5r/P5s) ادامه می‌یابند.
سلول آخر («وضعیت Drive») خلاصهٔ مراحل کامل‌شده و فضای آزاد Drive را نشان می‌دهد.


In [ ]:
# ---------------- تنظیمات (قبل از P0 اجرا کنید) ----------------
import os
PROFILE = "lit"      # "lit" = هم‌تراز با ادبیات (پیشنهادی) | "v3" = مقیاس v3 | "quick" = دودی
# os.environ["HF_TOKEN"] = "hf_..."        # یا در Colab Secrets / /content/hf_token.txt
if PROFILE == "quick":
    os.environ["RNA_QUICK"] = "1"
elif PROFILE == "lit":
    for k, v in dict(RNA_TRAIN_N="12000", RNA_EPOCHS="3", RNA_UNET_BASE="32",
                     RNA_SEEDS="0,1,2",                       # v5: مرحلهٔ B (seed 0 اگر از قبل هست skip می‌شود)
                     RNA_SCALE_SIZES="1500,3000,6000", RNA_SCALE_SEEDS="0,1", RNA_RAND_SEED="0").items():
        os.environ.setdefault(k, v)
# ---- Drive (v6): artifactهای هر سلول در MyDrive ذخیره/بازیابی می‌شوند
DRIVE_RUN = "chanrg_paper_v5"     # پوشهٔ اجرا در MyDrive (اجرای quick جدا ذخیره می‌شود)
SAVE_EMB = True                   # False ⇒ embeddingها (≈۵GB هر کدام) آپلود نمی‌شوند (بازیابی انجام می‌شود)
for k, v in dict(RNA_DRIVE_RUN=DRIVE_RUN, RNA_SAVE_EMB="1" if SAVE_EMB else "0").items():
    os.environ.setdefault(k, v)
print({k: v for k, v in os.environ.items() if k.startswith("RNA_")})


## P0 — محیط + کتابخانهٔ مشترک
نصب، gate سخت GPU/دیسک/توکن، نوشتن `plib.py` و `drive.py`، mount درایو. این سلول همیشه اجرا می‌شود.


In [ ]:
# fmt: off
# ============================================================
# P0 — env + کتابخانهٔ مشترک (plib.py) + gate سخت GPU   [v3: + kvec/partial_spearman، CI تصحیح‌شدهٔ Bonferroni در bootstrap]
# runtime: GPU (T4) | ~5 min | artifact: /content/rna_plib/plib.py , AR/env.json
# هر سلول بعدی از plib استفاده می‌کند → seq_col/ss_col و offset و مدل‌ها فقط یک‌جا تعریف می‌شوند.
# متغیرهای محیطی: RNA_QUICK=1 (اجرای دودی کوچک، خروجی جدا در /content/rna_paper_quick)
#                 RNA_SKIP_PIP=1 ، RNA_ALLOW_CPU=1 ، RNA_FAKE_BACKBONE=1 (تست بدون دانلود giga)
# ============================================================
import json, os, pathlib, shutil, subprocess, sys

QUICK = os.environ.get("RNA_QUICK", "") == "1"
AR = pathlib.Path("/content/rna_paper_v4_quick" if QUICK else "/content/rna_paper_v4")
STAGE = "p0_env"
AR.mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/rna_plib").mkdir(parents=True, exist_ok=True)

PLIB = r'''# plib.py — کتابخانهٔ مشترک Paper Track v3 (توسط سلول P0 نوشته می‌شود)
# همهٔ منطق مشترک اینجاست تا سلول‌ها تکرار نشوند (باگ‌های v1: seq_col تعریف‌نشده، offset غلط، ss به‌جای seq).
import json, os, pathlib, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

QUICK = os.environ.get("RNA_QUICK", "") == "1"
FAKE = os.environ.get("RNA_FAKE_BACKBONE", "") == "1"      # smoke بدون دانلود giga
AR = pathlib.Path("/content/rna_paper_v4_quick" if QUICK else "/content/rna_paper_v4")
SPLITS = ["train", "val", "test", "gena", "genc", "genf"]
EVAL = ["test", "gena", "genc", "genf"]
OOD = ["gena", "genc", "genf"]
THR_GRID = [round(0.05 * k, 2) for k in range(1, 20)]       # 0.05 … 0.95 (v1: فقط تا 0.8)
DZ = 32
PW = 8.0
UNET_BASE = int(os.environ.get("RNA_UNET_BASE", "24"))   # v4: عرض U-Net (باید مضرب ۸ باشد؛ v3=24)
assert UNET_BASE % 8 == 0
DEV = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEV == "cuda"
PUBLISHED = {"rnafold_ood": 0.2886, "rinalmo_giga_test": 0.7579, "rinalmo_giga_ood": 0.2140}


def cfg():
    return json.load(open(AR / "config_paper.json"))


def gate_for(stage):
    def gate(ok, msg, hard=True):
        tag = "GATE: PASS" if ok else ("GATE: FAIL" if hard else "GATE: WARN")
        print(f"{tag} | {msg}", flush=True)
        AR.mkdir(parents=True, exist_ok=True)
        json.dump({"stage": stage, "pass": bool(ok), "msg": msg, "hard": hard}, open(AR / "last_gate.json", "w"))
        if not ok and hard:
            raise SystemExit(1)
    return gate


# ----------------------------------------------------------------- ساختار / توالی
_MATCH = {"(": ")", "[": "]", "{": "}", "<": ">"}
_CLOSE = {v: k for k, v in _MATCH.items()}
_TRI = {}


def tri(L):
    if L not in _TRI:
        _TRI[L] = np.triu_indices(L, 1)
    return _TRI[L]


def parse_pairs(ss):
    """{(i,j): نوع بازکننده}؛ '(' = کانونیک-سطح‌اول، بقیه = شبه‌گره/سطح‌بالاتر"""
    st = {k: [] for k in _MATCH}
    out = {}
    for i, ch in enumerate(ss):
        if ch in _MATCH:
            st[ch].append(i)
        elif ch in _CLOSE and st[_CLOSE[ch]]:
            out[(st[_CLOSE[ch]].pop(), i)] = _CLOSE[ch]
    return out


def pair_matrix(ss, L):
    t = np.zeros((L, L), dtype=bool)
    for (i, j) in parse_pairs(ss):
        if i < L and j < L:
            t[i, j] = t[j, i] = True
    return t


_LUT = np.full(256, 4, dtype=np.int64)
for _c, _k in zip("ACGU", range(4)):
    _LUT[ord(_c)] = _k
_CT = np.zeros((5, 5), dtype=np.float32)
_CT[0, 3] = _CT[3, 0] = 1.0; _CT[1, 2] = _CT[2, 1] = 1.0; _CT[2, 3] = _CT[3, 2] = 1.0   # AU, GC, GU


def seq_idx(seq):
    return _LUT[np.frombuffer(seq.encode(), dtype=np.uint8)]


def canon_matrix(seq):
    ix = seq_idx(seq)
    m = _CT[ix[:, None], ix[None, :]].copy()
    np.fill_diagonal(m, 0.0)
    return m


def norm_seq(s):
    return str(s).upper().replace("T", "U")


# ----------------------------------------------------------------- داده‌ی یک split
class Split:
    def __init__(self, name, emb=True, phys=True, emb_dir="emb_paper"):
        C = cfg()
        self.name = name
        df = pd.read_parquet(AR / "data_paper" / f"{name}.parquet").reset_index(drop=True)
        self.df = df
        self.seqs = [norm_seq(s) for s in df[C["seq_col"]]]
        self.ss = [str(s) for s in df[C["ss_col"]]]
        self.fam = [str(f) for f in df[C["family_col"]]]
        self.L = np.array([len(s) for s in self.seqs], dtype=np.int64)
        self._gt = {}
        self.emb = None
        if emb:
            z = np.load(AR / emb_dir / f"{name}_meta.npz")
            assert (z["lens"] == self.L).all(), f"{name}: طول embedding با parquet نمی‌خواند (P2 را دوباره بزنید)"
            self.emb = np.load(AR / emb_dir / f"{name}.npy", mmap_mode="r")
            self.roff = np.concatenate([[0], np.cumsum(self.L)[:-1]])          # offset به «سطر» (نه عنصر)
            assert self.emb.ndim == 2 and self.emb.shape[0] == int(self.L.sum())
            self.dim = int(z["dim"])
        self.bpp = self.mfe = None
        if phys:
            z = np.load(AR / "phys_paper" / f"phys_{name}.npz")
            assert (z["lens"] == self.L).all(), f"{name}: طول phys با parquet نمی‌خواند (P3 را دوباره بزنید)"
            self.mfe = z["mfe_flat"]
            self.bpp = z["bpp_flat"] if "bpp_flat" in z.files else None
            npair = self.L * (self.L - 1) // 2
            self.poff = np.concatenate([[0], np.cumsum(npair)[:-1]])

    def __len__(self):
        return len(self.L)

    def gt_flat(self, i):
        if i not in self._gt:
            L = int(self.L[i])
            self._gt[i] = pair_matrix(self.ss[i], L)[tri(L)]
        return self._gt[i]

    def h(self, i):
        o, L = int(self.roff[i]), int(self.L[i])
        return torch.from_numpy(np.asarray(self.emb[o:o + L], dtype=np.float32)).to(DEV)

    def onehot(self, i):
        ix = seq_idx(self.seqs[i])
        return torch.from_numpy(np.eye(5, dtype=np.float32)[ix]).to(DEV)

    def canon_t(self, i):
        return torch.from_numpy(canon_matrix(self.seqs[i])).to(DEV)

    def n_phys(self):
        return (1 if self.mfe is not None else 0) + (1 if self.bpp is not None else 0)

    def phys_t(self, i):
        L = int(self.L[i]); iu = tri(L); o = int(self.poff[i]); n = L * (L - 1) // 2
        mats = []
        if self.bpp is not None:
            B = np.zeros((L, L), np.float32); B[iu] = self.bpp[o:o + n]; B += B.T; mats.append(B)
        M = np.zeros((L, L), np.float32); M[iu] = self.mfe[o:o + n]; M += M.T; mats.append(M)
        return torch.from_numpy(np.stack(mats)).to(DEV)


# ----------------------------------------------------------------- مدل
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.GroupNorm(8, cout), nn.ReLU(inplace=True),
                                 nn.Conv2d(cout, cout, 3, padding=1), nn.GroupNorm(8, cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)


class PairUNet(nn.Module):
    def __init__(self, cin, base=24):
        super().__init__()
        self.inc = DoubleConv(cin, base)
        self.down1 = nn.Conv2d(base, base, 3, stride=2, padding=1)
        self.d1 = DoubleConv(base, base * 2)
        self.down2 = nn.Conv2d(base * 2, base * 2, 3, stride=2, padding=1)
        self.d2 = DoubleConv(base * 2, base * 4)
        self.up1 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.u1 = DoubleConv(base * 4, base * 2)
        self.up2 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.u2 = DoubleConv(base * 2, base)
        self.out = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(0)
        _, _, L, _ = x.shape
        L4 = (L + 3) // 4 * 4
        if L4 != L:
            x = nn.functional.pad(x, (0, L4 - L, 0, L4 - L))
        d0 = self.inc(x)
        d1 = self.d1(self.down1(d0))
        d2 = self.d2(self.down2(d1))
        u1 = self.u1(torch.cat([self.up1(d2), d1], 1))
        u2 = self.u2(torch.cat([self.up2(u1), d0], 1))
        return self.out(u2)[0, 0, :L, :L]


class Decoder(nn.Module):
    """ویژگی جفتی (outer-concat + |diff| + فاصله [+ مکملیت] [+ فیزیک]) → U-Net. in_dim: 1280/1024 برای LM یا 5 برای one-hot"""
    def __init__(self, in_dim, use_canon=True, n_phys=0):
        super().__init__()
        self.wz = nn.Linear(in_dim, DZ)
        self.use_canon, self.n_phys = use_canon, n_phys
        self.unet = PairUNet(3 * DZ + 2 + (1 if use_canon else 0) + n_phys, base=UNET_BASE)

    def forward(self, h, canon=None, phys=None):
        A = self.wz(h); L = A.shape[0]
        Ai = A.unsqueeze(1).expand(L, L, DZ); Aj = A.unsqueeze(0).expand(L, L, DZ)
        idx = torch.arange(L, device=h.device)
        d = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs().float()
        parts = [Ai, Aj, (Ai - Aj).abs(), torch.stack([torch.exp(-d / 60.0), 1.0 / (1.0 + d)], -1)]
        if self.use_canon:
            parts.append(canon.unsqueeze(-1))
        if self.n_phys:
            parts.append(phys.permute(1, 2, 0))
        return self.unet(torch.cat(parts, -1).permute(2, 0, 1))


ARMS = {  # lm: embedding RiNALMo | canon: کانال مکملیت | phys: bpp+mfe
    "unet+phys": dict(lm=True, canon=True, phys=True),
    "unet": dict(lm=True, canon=True, phys=False),
    "nolm+phys": dict(lm=False, canon=True, phys=True),
}


def build_arm(spec, lm_dim, sample_split):
    nph = sample_split.n_phys() if spec["phys"] else 0
    return Decoder(lm_dim if spec["lm"] else 5, spec["canon"], nph).to(DEV)


def arm_scores(model, spec, sp, i, hfn=None):
    if spec["lm"]:
        h = hfn(sp, i) if hfn is not None else sp.h(i)
    else:
        h = sp.onehot(i)
    canon = sp.canon_t(i) if spec["canon"] else None
    phys = sp.phys_t(i) if spec["phys"] else None
    with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
        s = model(h, canon, phys)
    return s.float()


def train_decoder(model, spec, tr, seed, n_seq, accum=1, hfn=None, extra_groups=None, log_every=2000, tag="",
                  lr=1e-3, clip=5.0):
    """n_seq توالی (با تکرار epoch) ؛ هر update = accum توالی. extra_groups: [{'params':..,'lr':..}] برای LoRA"""
    groups = [{"params": list(model.parameters()), "lr": lr}] + (extra_groups or [])
    opt = torch.optim.Adam(groups)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(PW, device=DEV))
    n = len(tr)
    order = np.concatenate([np.random.RandomState(seed + 100 * e).permutation(n) for e in range(n_seq // n + 1)])[:n_seq]
    model.train()
    lora_gn = []
    for u in range(n_seq // accum):
        opt.zero_grad(set_to_none=True)
        tot = 0.0
        for a in range(accum):
            i = int(order[u * accum + a]); L = int(tr.L[i]); iu = tri(L)
            s = arm_scores(model, spec, tr, i, hfn)
            tgt = torch.from_numpy(tr.gt_flat(i).astype(np.float32)).to(DEV)
            loss = lossf(s[iu[0], iu[1]], tgt) / accum
            scaler.scale(loss).backward(); tot += float(loss.detach())
        scaler.unscale_(opt)
        if extra_groups:
            lp = [p for g in extra_groups for p in g["params"]]
            lora_gn.append(float(torch.nn.utils.clip_grad_norm_(lp, float("inf"))))
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(opt); scaler.update()
        if (u + 1) % log_every == 0 or u == 0:
            extra = f" |g_lora|={lora_gn[-1]:.2e}" if lora_gn else ""
            print(f"    {tag} update {u+1}/{n_seq//accum} loss={tot:.4f}{extra}", flush=True)
    model.eval()
    return {"final_grad_norm_lora": lora_gn[-1] if lora_gn else None,
            "min_grad_norm_lora": min(lora_gn) if lora_gn else None}


# ----------------------------------------------------------------- ارزیابی
@torch.no_grad()
def collect(model, spec, sp, hfn=None):
    S, G = [], []
    for i in range(len(sp)):
        s = torch.sigmoid(arm_scores(model, spec, sp, i, hfn))
        s = ((s + s.T) / 2).cpu().numpy()
        S.append(s[tri(int(sp.L[i]))].astype(np.float16)); G.append(sp.gt_flat(i))
    return S, G


def counts_at(S, G, thr):
    tp = np.array([int(((s > thr) & g).sum()) for s, g in zip(S, G)])
    fp = np.array([int(((s > thr) & ~g).sum()) for s, g in zip(S, G)])
    fn = np.array([int((~(s > thr) & g).sum()) for s, g in zip(S, G)])
    return tp, fp, fn


def f1_of(tp, fp, fn):
    tp, fp, fn = float(np.sum(tp)), float(np.sum(fp)), float(np.sum(fn))
    return 2 * tp / max(2 * tp + fp + fn, 1.0)


def prf(tp, fp, fn):
    tp, fp, fn = float(np.sum(tp)), float(np.sum(fp)), float(np.sum(fn))
    return tp / max(tp + fp, 1.0), tp / max(tp + fn, 1.0), 2 * tp / max(2 * tp + fp + fn, 1.0)


def best_thr(S, G, grid=THR_GRID):
    best = (grid[0], -1.0)
    for t in grid:
        f = f1_of(*counts_at(S, G, t))
        if f > best[1]:
            best = (t, f)
    return best


def onepair_counts(S, G, Ls, thr):
    tp = np.zeros(len(S), int); fp = np.zeros(len(S), int); fn = np.zeros(len(S), int)
    for k, (s, g, L) in enumerate(zip(S, G, Ls)):
        L = int(L); iu = tri(L)
        cand = np.flatnonzero(s > thr)
        cand = cand[np.argsort(-s[cand].astype(np.float32), kind="stable")]
        used = np.zeros(L, bool); pk = np.zeros(len(s), bool)
        a_, b_ = iu[0][cand], iu[1][cand]
        for j, a, b in zip(cand, a_, b_):
            if not used[a] and not used[b]:
                used[a] = used[b] = True; pk[j] = True
        tp[k] = int((pk & g).sum()); fp[k] = int((pk & ~g).sum()); fn[k] = int((~pk & g).sum())
    return tp, fp, fn


def rank_metrics(S, G, max_n=30_000_000):
    from sklearn.metrics import average_precision_score, roc_auc_score
    s = np.concatenate(S).astype(np.float32); g = np.concatenate(G)
    if len(s) > max_n:
        sel = np.random.RandomState(0).choice(len(s), max_n, replace=False); s, g = s[sel], g[sel]
    if g.sum() == 0 or g.all():
        return None, None
    return round(float(roc_auc_score(g, s)), 4), round(float(average_precision_score(g, s)), 4)


def oracle_f1(S, G):
    best = (0.0, 0.5)
    for t in [round(x, 3) for x in np.linspace(0.02, 0.98, 25)]:
        f = f1_of(*counts_at(S, G, t))
        if f > best[0]:
            best = (f, t)
    return best


DIST_BINS = [(1, 4), (4, 10), (10, 30), (30, 100), (100, 10 ** 9)]


def category_counts(sp, S, thr):
    cat = {}
    for i, s in enumerate(S):
        L = int(sp.L[i]); iu = tri(L); g = sp.gt_flat(i); p = s > thr
        canon = canon_matrix(sp.seqs[i])[iu] > 0
        pkm = np.zeros((L, L), bool)
        for (a, b), k in parse_pairs(sp.ss[i]).items():
            if k != "(" and a < L and b < L:
                pkm[a, b] = True
        pk = pkm[iu]; d = iu[1] - iu[0]
        masks = [("canonical", canon), ("non_canonical", ~canon), ("pseudoknot", pk)] + \
                [(f"dist_{lo}-{hi if hi < 10**8 else 'inf'}", (d >= lo) & (d < hi)) for lo, hi in DIST_BINS]
        for key, m in masks:
            c = cat.setdefault(key, [0, 0, 0, 0])
            c[0] += int((p & g & m).sum()); c[1] += int((p & ~g & m).sum()); c[2] += int((~p & g & m).sum()); c[3] += int((g & m).sum())
    return {k: {"p": round(prf(c[0], c[1], c[2])[0], 4), "r": round(prf(c[0], c[1], c[2])[1], 4),
                "f1": round(prf(c[0], c[1], c[2])[2], 4), "n_gt": c[3]} for k, c in cat.items()}


def evaluate_arm(model, spec, splits, hfn=None, with_categories=True):
    """thr فقط روی val؛ سپس همهٔ splitهای ارزیابی. خروجی: (نتیجه، counts برای bootstrap)"""
    Sv, Gv = collect(model, spec, splits["val"], hfn)
    thr, vf1 = best_thr(Sv, Gv)
    res = {"thr": thr, "val_f1": round(vf1, 4), "splits": {}}
    cnt = {}
    for n in EVAL:
        sp = splits[n]
        S, G = collect(model, spec, sp, hfn)
        tp, fp, fn = counts_at(S, G, thr)
        tp1, fp1, fn1 = onepair_counts(S, G, sp.L, thr)
        p, r, f1 = prf(tp, fp, fn)
        of, ot = oracle_f1(S, G)
        auroc, auprc = rank_metrics(S, G)
        res["splits"][n] = {"f1": round(f1, 4), "p": round(p, 4), "r": round(r, 4),
                            "f1_onepair": round(f1_of(tp1, fp1, fn1), 4),
                            "f1_oracle": round(of, 4), "oracle_thr": ot, "auroc": auroc, "auprc": auprc}
        if with_categories:
            res["splits"][n]["categories"] = category_counts(sp, S, thr)
        for k, v in dict(tp=tp, fp=fp, fn=fn, tp1=tp1, fp1=fp1, fn1=fn1).items():
            cnt[f"{n}_{k}"] = v
        print(f"      eval {n}: f1={f1:.4f} (oracle {of:.4f}, one-pair {f1_of(tp1, fp1, fn1):.4f})", flush=True)
    res["ood_mean"] = round(float(np.mean([res["splits"][n]["f1"] for n in OOD])), 4)
    return res, cnt


# ----------------------------------------------------------------- backbone (RiNALMo یا toy برای smoke)
class _ToyAttn(nn.Module):
    def __init__(self, d):
        super().__init__(); self.query = nn.Linear(d, d); self.value = nn.Linear(d, d)


class _ToyLayer(nn.Module):
    def __init__(self, d):
        super().__init__(); self.attention = nn.Module(); self.attention.self = _ToyAttn(d); self.ff = nn.Linear(d, d)

    def forward(self, x):
        a = self.attention.self
        return x + torch.tanh(a.value(x) * torch.sigmoid(a.query(x))) + torch.tanh(self.ff(x))


class _ToyEnc(nn.Module):
    def __init__(self, d, n):
        super().__init__(); self.layer = nn.ModuleList([_ToyLayer(d) for _ in range(n)])


class ToyLM(nn.Module):
    def __init__(self, d=64, n=4):
        super().__init__()
        self.emb = nn.Embedding(8, d); self.encoder = _ToyEnc(d, n)

    def forward(self, input_ids=None, attention_mask=None, **kw):
        x = self.emb(input_ids)
        for l in self.encoder.layer:
            x = l(x)
        return type("O", (), {"last_hidden_state": x})()


class ToyTok:
    def __call__(self, seq, return_tensors="pt", add_special_tokens=True):
        ids = [1] + [2 + "ACGU".find(c) if c in "ACGU" else 6 for c in seq] + [7]
        t = torch.tensor([ids])
        return {"input_ids": t, "attention_mask": torch.ones_like(t)}


def load_backbone(train=False):
    """-> (model, tokenizer, name, dim). FAKE: ToyLM 64 بعدی"""
    if FAKE:
        torch.manual_seed(0)
        m = ToyLM().to(DEV)
        return m.eval(), ToyTok(), "toy-backbone", 64
    import transformers as tf
    for cls in (tf.AutoConfig, tf.AutoModel, tf.AutoTokenizer):
        orig = cls.register

        def safe(*a, _o=orig, **k):
            try:
                return _o(*a, **k)
            except ValueError:
                return None
        cls.register = safe
    from multimolecule import RnaTokenizer, RiNALMoModel
    last = None
    for name in ["multimolecule/rinalmo-giga", "multimolecule/rinalmo-mega"]:
        try:
            tok = RnaTokenizer.from_pretrained(name)
            try:
                model = RiNALMoModel.from_pretrained(name, dtype=torch.float16)
            except Exception:
                model = RiNALMoModel.from_pretrained(name, torch_dtype=torch.float16)
            dim = int(model.config.hidden_size)
            return model.to(DEV).eval(), tok, name, dim
        except Exception as e:
            last = e
            print(f"  {name} failed: {type(e).__name__}: {str(e)[:120]}", flush=True)
    raise RuntimeError(f"هیچ backbone لود نشد: {last}")


def load_random_backbone(seed=0):
    """v5: همان معماری RiNALMo-giga با وزن‌های «تصادفی» (کنترل: دانش پیش‌آموزش در برابر یک تبدیل پرابعاد تصادفی). -> (model, tok, name, dim)"""
    if FAKE:
        torch.manual_seed(1000 + seed)
        m = ToyLM().to(DEV)
        return m.eval(), ToyTok(), "toy-random", 64
    import transformers as tf
    for cls in (tf.AutoConfig, tf.AutoModel, tf.AutoTokenizer):
        orig = cls.register

        def safe(*a, _o=orig, **k):
            try:
                return _o(*a, **k)
            except ValueError:
                return None
        cls.register = safe
    from multimolecule import RnaTokenizer, RiNALMoModel
    name = "multimolecule/rinalmo-giga"
    tok = RnaTokenizer.from_pretrained(name)
    try:
        from multimolecule import RiNALMoConfig
        conf = RiNALMoConfig.from_pretrained(name)
    except Exception:
        _m = RiNALMoModel.from_pretrained(name)
        conf = _m.config
        del _m
    torch.manual_seed(seed)
    model = RiNALMoModel(conf)                      # وزن‌های تصادفی (init پیش‌فرض HF)
    model = model.half().to(DEV).eval()             # همان dtype مسیر پیش‌آموزش‌دیده
    return model, tok, f"RANDOM-INIT({name}, seed={seed})", int(conf.hidden_size)


def embed_tokens(model, tok, seq):
    """(L, dim) float32 — همان تابع برای کش P2 و LoRA در P6 (تا ورودی دو بازو یکسان باشد)"""
    L = len(seq)
    enc = tok(seq, return_tensors="pt", add_special_tokens=True)
    enc = {k: v.to(DEV) for k, v in enc.items()}
    with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
        o = model(**enc)
    h = o.last_hidden_state if hasattr(o, "last_hidden_state") else o[0]
    real = enc["attention_mask"][0].bool().nonzero().squeeze(-1)
    n = int(real.numel())
    take = real[1:1 + L] if n >= L + 2 else (real[1:] if n == L + 1 else real[:L])
    hh = h[0, take, :].float()
    if hh.shape[0] < L:
        hh = torch.cat([hh, hh[-1:].expand(L - hh.shape[0], -1)], 0)
    return hh[:L]


# ----------------------------------------------------------------- شباهت (4-mer) — v3: مشترک بین P1b و P7
def kvec(seq):
    ix = seq_idx(seq); v = np.zeros(256, np.float32)
    ok = ix < 4
    if len(ix) >= 4:
        code = ix[:-3] * 64 + ix[1:-2] * 16 + ix[2:-1] * 4 + ix[3:]
        good = ok[:-3] & ok[1:-2] & ok[2:-1] & ok[3:]
        np.add.at(v, code[good], 1.0)
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


def _rank(x):
    return pd.Series(x).rank().values


def partial_spearman(x, y, ctrl):
    R = np.column_stack([_rank(c) for c in ctrl] + [np.ones(len(x))])
    rx, ry = _rank(x), _rank(y)
    ex = rx - R @ np.linalg.lstsq(R, rx, rcond=None)[0]
    ey = ry - R @ np.linalg.lstsq(R, ry, rcond=None)[0]
    return float(np.corrcoef(ex, ey)[0, 1]) if ex.std() > 0 and ey.std() > 0 else float("nan")


# ----------------------------------------------------------------- bootstrap (واحد دلخواه) جفت‌شده
def fam_table(fams, tp, fp, fn):
    names = sorted(set(fams)); ix = {f: k for k, f in enumerate(names)}
    T = np.zeros((len(names), 3))
    for f, a, b, c in zip(fams, tp, fp, fn):
        T[ix[f]] += (a, b, c)
    return names, T


def paired_bootstrap(sys_a, sys_b, fams_by_split, n_boot=2000, seed=0, splits=None):
    """sys_x: {split: [ (tp,fp,fn)-arrays per seed ]}. fams_by_split = برچسب «واحد بازنمونه» هر توالی (v3: family یا clan/architecture). واحدها در هر split بازنمونه می‌شوند (یکسان برای دو سامانه
    و همهٔ seedها). آماره‌ها: Δ هر split و Δ میانگین-splitها (OOD و Test جدا)."""
    rng = np.random.RandomState(seed)
    splits = splits or EVAL
    W = {}
    for s in splits:
        names, _ = fam_table(fams_by_split[s], *([np.zeros(len(fams_by_split[s]))] * 3))
        W[s] = rng.multinomial(len(names), np.ones(len(names)) / len(names), size=n_boot).astype(float)

    def f1_boot(system, s):
        out = []
        for tpfpfn in system[s]:
            _, T = fam_table(fams_by_split[s], *tpfpfn)
            tp, fp, fn = W[s] @ T[:, 0], W[s] @ T[:, 1], W[s] @ T[:, 2]
            out.append(2 * tp / np.maximum(2 * tp + fp + fn, 1))
        return np.mean(out, 0)

    def point(system, s):
        return np.mean([f1_of(*t) for t in system[s]])

    res = {}
    D = {}
    for s in splits:
        da, db = f1_boot(sys_a, s), f1_boot(sys_b, s)
        D[s] = da - db
        d0 = point(sys_a, s) - point(sys_b, s)
        res[s] = {"delta": round(float(d0), 4), "ci95": [round(float(np.percentile(D[s], 2.5)), 4), round(float(np.percentile(D[s], 97.5)), 4)],
                  "ci_bonf3": [round(float(np.percentile(D[s], 100 * 0.05 / 3 / 2)), 4), round(float(np.percentile(D[s], 100 - 100 * 0.05 / 3 / 2)), 4)],
                  "n_units": int(len(set(fams_by_split[s]))), "n_families": int(len(set(fams_by_split[s])))}
    oods = [s for s in OOD if s in splits]
    if oods:
        d = np.mean([D[s] for s in oods], 0)
        d0 = np.mean([res[s]["delta"] for s in oods])
        res["ood_mean"] = {"delta": round(float(d0), 4), "ci95": [round(float(np.percentile(d, 2.5)), 4), round(float(np.percentile(d, 97.5)), 4)]}
    return res
'''

pathlib.Path("/content/rna_plib/plib.py").write_text(PLIB)

DPLIB = r'''# drive.py — همگام‌سازی artifactهای هر مرحله با Google Drive (توسط سلول P0 نوشته می‌شود؛ نسخهٔ v6)
# قرارداد هر سلول: D.stage(name, restore=<الگوی ورودی‌ها>, save=<الگوی خروجی‌ها>)
#   ۱) restore: فایل‌های موجود در Drive به /content کپی می‌شوند (اگر محلی نباشد یا کهنه باشد)
#   ۲) اگر marker مرحله با فینگرپرینت فعلی (config_paper.json + متغیرهای RNA_* + plib.py) بخواند، سلول skip
#      می‌شود و لاگ ذخیره‌شدهٔ آن مرحله چاپ می‌شود (نتیجه بدون اجرای دوباره دیده می‌شود)
#   ۳) در پایان — حتی روی خطا — خروجی‌ها و لاگ در Drive ذخیره و marker نوشته می‌شود (ادامهٔ نیمه‌کاره ممکن)
# ذخیره نمی‌شوند: مدل RiNALMo (کش HuggingFace) و دیتاست خام CHANRG — در هر سشن نو دوباره دانلود می‌شوند.
# مسیرها همه AR-نسبی‌اند؛ glob فقط در مؤلفهٔ پایانی مجاز است (مثلاً "emb_paper/*.npy").
import atexit, contextlib, glob, hashlib, json, os, pathlib, shutil, sys, time, traceback

SYNC_VERSION = "v6-1"
DRIVE_MNT = "/content/drive"


class StageSkipped(Exception):
    """سلول قبلاً کامل شده؛ IPython این را خطای واقعی نمی‌شمارد (سلول catch می‌کند)."""


def _quick():
    return os.environ.get("RNA_QUICK", "") == "1"


def AR():
    return pathlib.Path("/content/rna_paper_v4_quick" if _quick() else "/content/rna_paper_v4")


def drive_root():
    run = os.environ.get("RNA_DRIVE_RUN", "").strip() or "chanrg_paper_v5"
    return pathlib.Path(DRIVE_MNT) / "MyDrive" / run / ("quick" if _quick() else "full")


def _remote_dir():
    return drive_root() / "artifacts"


def _done_dir():
    return drive_root() / "_done"


def _log_dir():
    return drive_root() / "logs"


_MNT = None
_CHECKED = False


def mounted():
    """درایو را mount می‌کند (Colab)؛ اگر ممکن نبود None برمی‌گرداند و سلول‌ها محلی اجرا می‌شوند."""
    global _MNT, _CHECKED
    if _CHECKED:
        return _MNT
    _CHECKED = True
    if pathlib.Path(DRIVE_MNT, "MyDrive").exists():
        _MNT = DRIVE_MNT
    else:
        try:
            from google.colab import drive
            drive.mount(DRIVE_MNT)
            _MNT = DRIVE_MNT
            print(f"DRIVE: mounted → {drive_root()}", flush=True)
        except Exception as e:
            _MNT = None
            print(f"DRIVE: mount نشد ({type(e).__name__}: {str(e)[:120]}) — ادامهٔ محلی، بدون همگام‌سازی", flush=True)
    return _MNT


def _identical(a, b):
    """یکسان = اندازهٔ برابر و mtime ریموت کهنه‌تر نباشد (مقایسهٔ امن روی FUSE درایو)"""
    try:
        sa, sb = a.stat(), b.stat()
    except OSError:
        return False
    return sa.st_size == sb.st_size and sb.st_mtime >= sa.st_mtime - 86400


def _free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except OSError:
        return float("inf")


def _copy(src, dst, label):
    dst.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    tmp = dst.with_name(dst.name + ".part")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)
    sz = src.stat().st_size
    gb = f"{sz/1e9:.2f}GB" if sz > 1e9 else f"{sz/1e6:.1f}MB"
    print(f"  DRIVE {label} ({gb}, {time.time()-t0:.0f}s)", flush=True)


def _patterns(spec):
    return [p.strip() for p in str(spec).replace("\n", ";").split(";") if p.strip()]


def _local_files(pat):
    return sorted(pathlib.Path(f) for f in glob.glob(str(AR() / pat)))


def _remote_files(pat):
    if not mounted():
        return []
    return sorted(pathlib.Path(f) for f in glob.glob(str(_remote_dir() / pat)))


def _local_for(remote_file):
    return AR() / remote_file.relative_to(_remote_dir())


def pull(spec):
    if not mounted():
        return 0
    n = 0
    for pat in _patterns(spec):
        for rf in _remote_files(pat):
            lf = _local_for(rf)
            if lf.exists() and _identical(lf, rf):
                continue
            _copy(rf, lf, f"↓ {lf.relative_to(AR())}")
            n += 1
    print(f"DRIVE: بازیابی — {n} فایل از Drive" if n else "DRIVE: بازیابی — همهٔ فایل‌ها محلی بودند", flush=True)
    return n


def _save_ok(pat):
    rel = pat.replace("\\", "/").lstrip("./")
    return not rel.startswith("emb_") or os.environ.get("RNA_SAVE_EMB", "1") == "1"


def push(spec):
    if not mounted():
        return 0
    n = 0
    for pat in _patterns(spec):
        if not _save_ok(pat):
            print(f"DRIVE: «{pat}» آپلود نشد (RNA_SAVE_EMB=0)", flush=True)
            continue
        for lf in _local_files(pat):
            rf = _remote_dir() / lf.relative_to(AR())
            if rf.exists() and _identical(lf, rf):
                continue
            if lf.stat().st_size > _free_gb(DRIVE_MNT) * 1e9:
                print(f"DRIVE: ⚠ فضای Drive کافی نیست برای {lf.name} ({lf.stat().st_size/1e9:.1f}GB) — رد شد "
                      f"(RNA_SAVE_EMB=0 یا پوشهٔ قدیمی را پاک کنید)", flush=True)
                continue
            _copy(lf, rf, f"↑ {lf.relative_to(AR())}")
            n += 1
    print(f"DRIVE: ذخیره — {n} فایل در Drive" if n else "DRIVE: ذخیره — تغییرِ جدیدی نبود", flush=True)
    return n


def fingerprint():
    """اگر config یا هر RNA_* عوض شود، همهٔ markerها باطل می‌شوند (اجرای مجدد عمداً)."""
    h = hashlib.sha1()
    h.update(f"sync={SYNC_VERSION}".encode())
    for p in (AR() / "config_paper.json", pathlib.Path("/content/rna_plib/plib.py")):
        if p.exists():
            h.update(p.name.encode() + p.read_bytes())
    for k in sorted(k for k in os.environ if k.startswith("RNA_")):
        h.update(f"|{k}={os.environ[k]}".encode())
    return h.hexdigest()


def _marker(name):
    return _done_dir() / f"{name}.json"


def marker_ok(name, fp):
    try:
        return json.loads(_marker(name).read_text()).get("fp") == fp
    except Exception:
        return False


def write_marker(name, fp):
    _done_dir().mkdir(parents=True, exist_ok=True)
    p = _marker(name)
    tmp = p.with_name(p.name + ".part")
    tmp.write_text(json.dumps({"stage": name, "fp": fp, "ts": round(time.time()), "sync": SYNC_VERSION}, indent=2), encoding="utf-8")
    os.replace(tmp, p)


def _local_complete(*specs):
    for spec in specs:
        for pat in _patterns(spec):
            if _remote_files(pat) and not _local_files(pat):
                return False
    return True


def _show_log(name):
    lp = _log_dir() / f"{name}.log"
    if not lp.exists():
        return
    try:
        print(f"\n========== لاگ ذخیره‌شدهٔ «{name}» از Drive ==========", flush=True)
        sys.stdout.write(lp.read_text(encoding="utf-8", errors="replace"))
        print("========== پایان لاگ ==========\n", flush=True)
    except Exception as e:
        print(f"DRIVE: لاگ خوانده نشد — {type(e).__name__}: {str(e)[:120]}", flush=True)


class _Tee:
    """چاپ سلول هم به نوتبوک می‌رود، هم به فایل لاگ (برای ذخیره در Drive)."""
    def __init__(self, stream, f):
        self.stream, self.f = stream, f
    def write(self, x):
        self.stream.write(x)
        self.f.write(x)
        self.f.flush()
        return len(x)
    def flush(self):
        self.stream.flush()
        self.f.flush()
    def isatty(self):        # transformers/tqdm این را صدا می‌زنند (progress bar)
        try:
            return self.stream.isatty()
        except Exception:
            return False
    def fileno(self):
        try:
            return self.stream.fileno()
        except Exception:
            return 1          # stdout
    @property
    def encoding(self):
        return getattr(self.stream, "encoding", "utf-8")
    @property
    def errors(self):
        return getattr(self.stream, "errors", None)
    def writable(self):
        return True
    def readable(self):
        return False
    def seekable(self):
        return False
    def __getattr__(self, name):
        return getattr(self.stream, name)


@contextlib.contextmanager
def stage(name, restore="", save="", always=False):
    """پوشش هر سلول: بازیابی از Drive → skip اگر قبلاً کامل بود → اجرا → ذخیره + marker."""
    mnt = mounted()
    failed = {"v": True}    # فقط اگر بدنهٔ سلول واقعاً اجرا شد (yield رسید) False می‌شود
    log_local = AR() / "stage_logs" / f"{name}.log"
    try:
        if mnt:
            pull(restore)
        fp = fingerprint()   # بعد از بازیابی: config_paper.json ممکن است تازه از Drive آمیده باشد
        if mnt and not always and marker_ok(name, fp) and _local_complete(restore, save):
            print(f"### DRIVE: مرحلهٔ «{name}» قبلاً کامل شده — سلول skip می‌شود (fingerprint یکسان)", flush=True)
            _show_log(name)
            raise StageSkipped()
        log_local.parent.mkdir(parents=True, exist_ok=True)
        atexit.register(lambda: push(save))
        with open(log_local, "w", encoding="utf-8") as lf:
            with contextlib.redirect_stdout(_Tee(sys.stdout, lf)):
                failed["v"] = False   # اجرای واقعیِ سلول از این‌جا شروع می‌شود؛ قبل از آن هر خطا = ناتمام
                try:
                    yield
                except BaseException:
                    failed["v"] = True
                    sys.stdout.write("\n" + traceback.format_exc())
                    raise
    finally:
        if mnt:
            try:
                if log_local.exists():
                    _copy(log_local, _log_dir() / f"{name}.log", f"↑ log/{name}")
                push(save)
                if not failed["v"]:
                    write_marker(name, fingerprint())   # پس از اجرا: config تازه‌نوشته‌شده هم در فینگرپرینت نهفته است
            except Exception as e:
                print(f"DRIVE: ⚠ همگام‌سازی ناتمام — {type(e).__name__}: {str(e)[:150]}", flush=True)


def status():
    """خلاصهٔ وضعیت Drive: فضای آزاد + markerهای نوشته‌شده (سلول اختیاری انتهای نوتبوک)."""
    if not mounted():
        print("DRIVE: mount نیست")
        return
    print(f"DRIVE root: {drive_root()} | آزاد: {_free_gb(DRIVE_MNT):.1f}GB", flush=True)
    d = _done_dir()
    if not d.exists():
        print("marker: هنوز هیچ مرحله‌ای کامل نشده")
        return
    for m in sorted(d.glob("*.json")):
        try:
            j = json.loads(m.read_text())
            print(f"  ✅ {j['stage']:<22} fp={j['fp'][:10]}… ts={time.strftime('%Y-%m-%d %H:%M', time.localtime(j['ts']))}", flush=True)
        except Exception:
            print(f"  ⚠ {m.name} خوانده نشد")
'''

pathlib.Path("/content/rna_plib/drive.py").write_text(DPLIB)

# ---------------- Drive (v6): mount + همگام‌سازی artifactها ----------------
import importlib
import sys
sys.path.insert(0, "/content/rna_plib")
import drive as D
importlib.reload(D)   # فایل drive.py همین بالا نوشته شد → از دیسک بازخوانی می‌شود (کش کرنل قدیمی بی‌اثر)
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
with D.stage("p0_env", save="env.json, last_gate.json", always=True):



    def gate(ok, msg, hard=True):
        tag = "GATE: PASS" if ok else ("GATE: FAIL" if hard else "GATE: WARN")
        print(f"{tag} | {msg}", flush=True)
        json.dump({"stage": STAGE, "pass": bool(ok), "msg": msg, "hard": hard}, open(AR / "last_gate.json", "w"))
        if not ok and hard:
            raise SystemExit(1)


    print(f"### {STAGE} START | quick={QUICK}", flush=True)
    if os.environ.get("RNA_SKIP_PIP", "") != "1":
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "huggingface_hub", "pandas", "pyarrow",
                        "scikit-learn", "multimolecule", "peft", "ViennaRNA"], check=False, timeout=2400)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False, timeout=300)

    sys.path.insert(0, "/content/rna_plib")
    import importlib
    import torch
    import plib as P
    importlib.reload(P)

    cuda = torch.cuda.is_available()
    gate(cuda or os.environ.get("RNA_ALLOW_CPU", "") == "1",
         f"GPU: {torch.cuda.get_device_name(0)}" if cuda else
         ("CPU (فقط تست؛ RNA_ALLOW_CPU=1)" if os.environ.get("RNA_ALLOW_CPU") == "1" else "GPU لازم است — Runtime ▸ Change runtime type ▸ T4"))
    tok = os.environ.get("HF_TOKEN", "").strip()
    if not tok and pathlib.Path("/content/hf_token.txt").exists():
        tok = pathlib.Path("/content/hf_token.txt").read_text().strip(); os.environ["HF_TOKEN"] = tok
    if os.environ.get("RNA_CHANRG_DIR"):
        gate(True, f"داده‌ی محلی: {os.environ['RNA_CHANRG_DIR']}", hard=False)
    else:
        gate(bool(tok), "HF_TOKEN تنظیم است (CHANRG گیت‌شده است)" if tok else
             "HF_TOKEN نیست — در Colab Secrets بگذارید یا فایل /content/hf_token.txt بسازید", hard=False)
    try:
        import RNA
        gate(True, f"ViennaRNA {RNA.__version__}")
    except Exception as e:
        gate(False, f"ViennaRNA نصب نیست: {e}")
    free = shutil.disk_usage("/content").free / 1e9
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if cuda else 0
    env = {"gpu": torch.cuda.get_device_name(0) if cuda else "CPU", "vram_gb": round(vram, 1), "disk_free_gb": round(free, 1),
           "torch": torch.__version__, "quick": QUICK, "fake_backbone": P.FAKE}
    json.dump(env, open(AR / "env.json", "w"), indent=2)
    print(json.dumps(env, indent=2), flush=True)
    gate(free > 20, f"دیسک آزاد {free:.0f}GB (نیاز T4-config: حدود ۱۵GB)", hard=False)
    if cuda and vram < 14:
        print(f"  هشدار: VRAM={vram:.1f}GB (<14) — کپ‌ها را کم کنید", flush=True)
    print(f"### {STAGE} DONE", flush=True)


## P1 — داده
فیلتر طول 50..200 (بدون truncation)، train متوازن‌خانواده، eval تصادفی با cap.

In [ ]:
# fmt: off
# ============================================================
# P1 — داده‌ی paper-track (T4-friendly)
#  تغییرات اصلی نسبت به v1:
#   • طول 50..200 با «فیلتر» (نه truncate): هم‌سو با تحلیل کنترل‌شده‌ی طول CHANRG ،
#     RNAfold روی توالی کامل fold می‌شود، و هزینه‌ی U-Net/ViennaRNA کم می‌ماند.
#   • train: نمونه‌گیری متوازن خانواده ؛ val/eval: تصادفی با cap (توزیع طبیعی؛ bootstrap خانواده‌ای وزن را درست می‌کند)
#   • اندازه‌ها (پیش‌فرض T4): train 6000 | val 300 | test 1000 | genA 1000 | genC 1000 | genF 800
#   • RNA_QUICK=1: train 400 | val 100 | هر eval 150 — برای دود
# runtime: cpu | ~5-10 min | artifact: AR/data_paper/*.parquet + config_paper.json
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p1_data", restore="data_paper/*.parquet; config_paper.json",
                 save="data_paper/*.parquet; config_paper.json"):

        import json, os, pathlib, shutil, sys
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import pandas as pd
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p1_data"
        gate = P.gate_for(STAGE)
        Q = P.QUICK


        def envint(k, full, quick):
            return int(os.environ.get(k, quick if Q else full))


        REQ = {"train_n": envint("RNA_TRAIN_N", 6000, 400), "val_n": envint("RNA_VAL_N", 300, 100),
               "test_n": envint("RNA_TEST_N", 1000, 150), "gena_n": envint("RNA_GENA_N", 1000, 150),
               "genc_n": envint("RNA_GENC_N", 1000, 150), "genf_n": envint("RNA_GENF_N", 800, 100),
               "len_min": envint("RNA_LEN_MIN", 50, 50), "len_max": envint("RNA_LEN_MAX", 200, 200), "seed": 42}
        print(f"### {STAGE} START | quick={Q} | {REQ}", flush=True)

        cpath = AR / "config_paper.json"
        if cpath.exists():
            old = json.load(open(cpath))
            if all(old.get(k) == v for k, v in REQ.items()):
                print("ART_CHECK: config_paper.json با همین کپ‌ها موجود است — skip", flush=True)
                raise SystemExit(0)
            print("کپ‌ها عوض شده → خروجی‌های پایین‌دستی (emb/phys/artifacts) پاک می‌شوند", flush=True)
            for d in ["emb_paper", "phys_paper", "artifacts_paper", "data_paper"]:
                shutil.rmtree(AR / d, ignore_errors=True)

        ALIAS = {"train": "train", "validation": "val", "val": "val", "test": "test", "gena": "gena", "genc": "genc", "genf": "genf"}
        local = os.environ.get("RNA_CHANRG_DIR")
        frames = []
        for fname in ["train.parquet", "validation.parquet", "test.parquet"]:
            if local:
                p = str(pathlib.Path(local) / fname)
            else:
                from huggingface_hub import hf_hub_download
                p = hf_hub_download("multimolecule/chanrg", fname, repo_type="dataset", token=os.environ.get("HF_TOKEN") or None)
            d = pd.read_parquet(p)
            frames.append(d)
            print(f"  {fname}: {len(d)} rows", flush=True)
        full = pd.concat(frames, ignore_index=True)
        cols = list(full.columns)
        print("  ستون‌ها:", cols, flush=True)
        seq_col = next((c for c in cols if c.lower() in ("sequence", "seq")), None)
        ss_col = next((c for c in cols if "structure" in c.lower() or "dot" in c.lower()), None)
        fam_col = next((c for c in ["family", "clan", "architecture", "super_family"] if c in cols), None)
        split_col = next((c for c in cols if c.lower() == "split"), None)
        gate(all([seq_col, ss_col, fam_col, split_col]), f"cols: seq={seq_col} ss={ss_col} family={fam_col} split={split_col}")
        full["_split"] = full[split_col].astype(str).str.lower().map(ALIAS)
        have = set(full["_split"].dropna().unique())
        gate(set(ALIAS.values()) <= have, f"splits موجود: {sorted(have)} | مقادیر خام: {sorted(full[split_col].astype(str).unique())[:12]}")

        full["_len"] = full[seq_col].astype(str).str.len()
        ok = (full["_len"] >= REQ["len_min"]) & (full["_len"] <= REQ["len_max"]) & (full[ss_col].astype(str).str.len() == full["_len"])
        print(f"  بعد از فیلتر طول [{REQ['len_min']},{REQ['len_max']}] و seq==ss: {int(ok.sum())}/{len(full)}", flush=True)
        full = full[ok]


        def balanced(df, n, seed):
            if n >= len(df):
                return df.sample(frac=1, random_state=seed)
            quota = max(2, int(n / max(df[fam_col].nunique(), 1)))
            out = pd.concat([g.sample(min(len(g), quota), random_state=seed) for _, g in df.groupby(fam_col)])
            if len(out) > n:
                out = out.sample(n, random_state=seed)
            elif len(out) < n:
                rest = df.loc[~df.index.isin(out.index)]
                out = pd.concat([out, rest.sample(min(n - len(out), len(rest)), random_state=seed)])
            return out


        (AR / "data_paper").mkdir(parents=True, exist_ok=True)
        info = {}
        for name in ["train", "val", "test", "gena", "genc", "genf"]:
            d = full[full["_split"] == name]
            want = REQ[f"{name}_n"]
            sub = balanced(d, want, REQ["seed"]) if name == "train" else d.sample(min(want, len(d)), random_state=REQ["seed"])
            sub = sub[[seq_col, ss_col, fam_col]].copy()
            sub["orig_idx"] = sub.index.values
            sub.reset_index(drop=True).to_parquet(AR / "data_paper" / f"{name}.parquet")
            info[name] = {"total_after_filter": int(len(d)), "subset": int(len(sub)), "families": int(sub[fam_col].nunique()),
                          "len_mean": round(float(sub[seq_col].astype(str).str.len().mean()), 1)}
            print(f"  {name}: {info[name]}", flush=True)

        C = dict(REQ)
        C.update({"seq_col": seq_col, "ss_col": ss_col, "family_col": fam_col, "splits_info": info,
                  "dataset": "multimolecule/chanrg" if not local else f"local:{local}",
                  "protocol": "length filter (no truncation), micro-F1 over pairs",
                  "published": P.PUBLISHED})
        json.dump(C, open(cpath, "w"), indent=2)
        small = [k for k, v in info.items() if v["subset"] < 50]
        gate(not small, "همهٔ splitها ≥ ۵۰ توالی" if not small else f"splitهای کوچک: {small}", hard=False)
        few = {k: v["families"] for k, v in info.items() if k in P.OOD and v["families"] < 15}
        if few:
            print(f"  هشدار: خانوادهٔ کم در {few} → CI بوت‌استرپ پهن می‌شود", flush=True)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P1b — gate نشت + واحد بازنمونه (جدید در v3)
برچسب family/clan/architecture، اشتراک با train، توالی‌های یکسان، واحد بازنمونهٔ هر split. **بدون آموزش**؛ با artifactهای v2 سازگار است.

In [ ]:
# fmt: off
# ============================================================
# P1b — (جدید در v3) gate نشت + تعیین «واحد درست بازنمونه» برای هر split
#   مشکل v2: bootstrap فقط روی family بود، ولی GenA/GenC/GenF احتمالاً روی سطوح architecture/clan/family جدا شده‌اند.
#   اگر خانواده‌ها درون یک clan/architecture هم‌بسته باشند، «۱۰۲ یا ۲۸ خانوادهٔ مستقل» وجود ندارد و CI ها باریک‌تر از واقع‌اند.
#   این سلول (بدون آموزش و بدون تغییر دادهٔ P1):
#     1) برچسب‌های family/clan/architecture/super_family هر توالی را از دیتاست اصلی برمی‌گرداند (با orig_idx) و ذخیره می‌کند
#     2) برای هر split و هر سطح: اشتراک برچسب‌ها با train (کامل و زیرمجموعهٔ cap‌شده)
#     3) «واحد بازنمونه» = درشت‌ترین سطحی که با train کاملاً مجزاست (برای Test/val: family)
#     4) توالی‌های دقیقاً یکسان بین train و هر split ارزیابی (ماسک حذف برای تحلیل حساسیت در P7)
#     5) ماتریس 4-mer کل train فیلترشده (برای تحلیل شباهت در P7)
# runtime: CPU | ~3-6 min | artifact: artifacts_paper/leakage.json , leak_masks.npz , train_full_kmer.npy , data_paper/levels_{split}.parquet
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p1b_leakage", restore="data_paper/*.parquet; config_paper.json",
                 save="artifacts_paper/leakage.json; artifacts_paper/leak_masks.npz; artifacts_paper/train_full_kmer.npy; data_paper/levels_*.parquet"):

        import json, os, pathlib, sys
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import pandas as pd
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p1b_leakage"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        OUT = AR / "artifacts_paper"; OUT.mkdir(parents=True, exist_ok=True)
        print(f"### {STAGE} START", flush=True)

        LEVELS = ["family", "clan", "architecture", "super_family"]
        ALIAS = {"train": "train", "validation": "val", "val": "val", "test": "test", "gena": "gena", "genc": "genc", "genf": "genf"}
        local = os.environ.get("RNA_CHANRG_DIR")
        frames = []
        for fname in ["train.parquet", "validation.parquet", "test.parquet"]:
            if local:
                p = str(pathlib.Path(local) / fname)
            else:
                from huggingface_hub import hf_hub_download
                p = hf_hub_download("multimolecule/chanrg", fname, repo_type="dataset", token=os.environ.get("HF_TOKEN") or None)
            frames.append(pd.read_parquet(p))
        full = pd.concat(frames, ignore_index=True)          # همان ترتیب P1 → index = orig_idx
        cols = list(full.columns)
        split_col = next(c for c in cols if c.lower() == "split")
        seq_col, ss_col, fam_col = C["seq_col"], C["ss_col"], C["family_col"]
        full["_split"] = full[split_col].astype(str).str.lower().map(ALIAS)
        lv_cols = [l for l in LEVELS if l in cols]
        gate(fam_col in lv_cols and len(lv_cols) >= 2, f"ستون‌های سطح: {lv_cols} (family_col={fam_col})")
        print("  سطوح موجود:", lv_cols, flush=True)

        full["_len"] = full[seq_col].astype(str).str.len()
        ok = (full["_len"] >= C["len_min"]) & (full["_len"] <= C["len_max"]) & (full[ss_col].astype(str).str.len() == full["_len"])
        fullf = full[ok]


        def lab(df, lvl):
            """برچسب سطح؛ مقدار گمشده → «NA::family» (هر خانوادهٔ بدون برچسب، واحد خودش است)"""
            raw = df[lvl]
            s = raw.astype(str).str.strip()
            miss = raw.isna() | s.str.lower().isin(["", "nan", "none", "-", "<na>"])
            return s.where(~miss, "NA::" + df[fam_col].astype(str)), miss


        LAB, MISS = {}, {}
        for l in lv_cols:
            LAB[l], MISS[l] = lab(full, l)
        n_unique = {l: int(LAB[l][~MISS[l]].nunique()) for l in lv_cols}
        order = sorted(lv_cols, key=lambda l: n_unique[l])          # درشت → ریز
        print(f"  تعداد برچسب یکتا در کل دیتا: {n_unique} | ترتیب درشت→ریز: {order}", flush=True)

        # ---- زیرمجموعه‌های P1 و نگاشت به دیتای کامل
        sub = {n: pd.read_parquet(AR / "data_paper" / f"{n}.parquet").reset_index(drop=True) for n in P.SPLITS}
        for n in P.SPLITS:
            ix = sub[n]["orig_idx"].values
            same = [P.norm_seq(a) == P.norm_seq(b) for a, b in zip(full.loc[ix, seq_col], sub[n][seq_col])]
            gate(all(same), f"{n}: نگاشت orig_idx → دیتای کامل درست است ({len(ix)} توالی)")
            lv = pd.DataFrame({"orig_idx": ix})
            for l in lv_cols:
                lv[l] = LAB[l].loc[ix].values
            lv.to_parquet(AR / "data_paper" / f"levels_{n}.parquet")

        tr_idx_full = fullf.index[fullf["_split"] == "train"]
        tr_set_full = {l: set(LAB[l].loc[tr_idx_full]) for l in lv_cols}
        tr_set_cap = {l: set(LAB[l].loc[sub["train"]["orig_idx"].values]) for l in lv_cols}

        CHECK = ["val"] + P.EVAL
        stats = {}
        for n in CHECK:
            idx_full = fullf.index[fullf["_split"] == n]
            idx_cap = sub[n]["orig_idx"].values
            stats[n] = {}
            for l in lv_cols:
                sf, sc = set(LAB[l].loc[idx_full]), set(LAB[l].loc[idx_cap])
                stats[n][l] = {"n_units_full": len(sf), "n_units_cap": len(sc),
                               "shared_full": len(sf & tr_set_full[l]), "shared_cap": len(sc & tr_set_cap[l]),
                               "missing_frac": round(float(MISS[l].loc[idx_cap].mean()), 3)}

        # ---- واحد بازنمونه
        unit = {}
        for n in CHECK:
            if n in P.OOD:
                # v4: واحد = سطحِ کاملاً مجزا از train با «کمترین تعداد واحد» (حداقل ۵). v3 با فرض سلسله‌مراتب ثابت architecture را برای GenC (۲۸ واحد)
                # برمی‌گزید در حالی که clan فقط ۱۱ واحد داشت ⇒ CI باریک‌تر از واقع. GenC در مقاله «clanهای غایب در train» است.
                cand = [l for l in lv_cols if stats[n][l]["shared_full"] == 0 and stats[n][l]["n_units_cap"] >= 5]
                cand = sorted(cand, key=lambda l: (stats[n][l]["n_units_cap"], order.index(l)))
                unit[n] = cand[0] if cand else fam_col
            else:
                unit[n] = fam_col

        # ---- توالی‌های یکسان
        tr_seq_full = set(P.norm_seq(s) for s in fullf.loc[tr_idx_full, seq_col])
        tr_seq_cap = set(P.norm_seq(s) for s in sub["train"][seq_col])
        dup, keep = {}, {}
        for n in CHECK:
            ss = [P.norm_seq(s) for s in sub[n][seq_col]]
            d_full = np.array([s in tr_seq_full for s in ss]); d_cap = np.array([s in tr_seq_cap for s in ss])
            dup[n] = {"n": len(ss), "n_unique_seqs": len(set(ss)), "dup_vs_train_full": int(d_full.sum()), "dup_vs_train_cap": int(d_cap.sum())}
            keep[n] = ~d_cap
        np.savez(OUT / "leak_masks.npz", **{f"keep_{n}": keep[n] for n in CHECK})

        # ---- 4-mer کل train (برای تحلیل شباهت P7)
        Tk = np.stack([P.kvec(P.norm_seq(s)) for s in fullf.loc[tr_idx_full, seq_col]]).astype(np.float16)
        np.save(OUT / "train_full_kmer.npy", Tk)

        # ---- چاپ جدول
        print("\nsplit | سطح | واحد در cap | مشترک با train(cap) | مشترک با train(کامل)", flush=True)
        for n in CHECK:
            for l in order:
                s = stats[n][l]
                print(f"{n:>5} | {l:>12} | {s['n_units_cap']:>5} | {s['shared_cap']:>5} | {s['shared_full']:>5}" + ("   ← واحد بازنمونه" if unit[n] == l else ""), flush=True)
        print("\nتوالی‌های یکسان با train:", {n: (dup[n]["dup_vs_train_cap"], dup[n]["dup_vs_train_full"]) for n in CHECK}, "(cap, کامل)", flush=True)

        out = {"order_coarse_to_fine": order, "n_unique_levels": n_unique, "family_col": fam_col,
               "splits": {n: {"unit": unit[n], "levels": stats[n], "dups": dup[n]} for n in CHECK}}
        bad_cap = [n for n in P.OOD if stats[n][fam_col]["shared_cap"] > 0]
        bad_full = [n for n in P.OOD if stats[n][fam_col]["shared_full"] > 0]
        dup_total = int(sum(dup[n]["dup_vs_train_cap"] for n in P.EVAL))
        small = {n: stats[n][unit[n]]["n_units_cap"] for n in P.OOD if stats[n][unit[n]]["n_units_cap"] < 10}
        out["gates"] = {"ood_family_disjoint_cap": not bad_cap, "ood_family_disjoint_full": not bad_full, "dup_total_cap": dup_total,
                        "small_unit_splits": small}
        json.dump(out, open(OUT / "leakage.json", "w"), indent=2)

        gate(not bad_cap, "هر سه split تعمیم family-disjoint از train (زیرمجموعهٔ cap) هستند" if not bad_cap
             else f"نشت: خانوادهٔ مشترک با train در {bad_cap} — «OOD» بودن این splitها معتبر نیست")
        gate(not bad_full, "family-disjoint از train کامل" if not bad_full else f"خانوادهٔ مشترک با train کامل در {bad_full}", hard=False)
        gate(dup_total == 0, "هیچ توالی یکسانی بین train و splitهای ارزیابی نیست" if dup_total == 0
             else f"{dup_total} توالی ارزیابی دقیقاً در train هست — P7 تحلیل حساسیت بدون آن‌ها را هم می‌سازد", hard=False)
        gate(not small, "هر split OOD حداقل ۱۰ واحد بازنمونه دارد" if not small
             else f"واحدهای کم (CI پهن؛ نتیجهٔ per-split را محتاطانه بخوانید): {small}", hard=False)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P2 — embedding (giga fp16)
memmap، resume هر ۵۰۰ توالی، offset بر حسب سطر.

In [ ]:
# fmt: off
# ============================================================
# P2 — embedding RiNALMo-giga (fp16, batch=1، بدون truncation) — memmap + resume هر ۵۰۰ توالی
# فیکس‌های v1: offset بر حسب «سطر» (نه عنصر) ، assert درست ، بدون خواندن کل split در RAM ، seq_col از config
# runtime: GPU T4 | ~20-30 min (+ دانلود ~2.6GB) | artifact: emb_paper/{split}.npy (N_res × dim fp16) + {split}_meta.npz
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p2_embed", restore="data_paper/*.parquet; config_paper.json",
                 save="emb_paper/*.npy; emb_paper/*_meta.npz; emb_paper/embed_done.json; emb_paper/*.tmp.npy; emb_paper/*.progress"):

        import json, os, sys, time
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import torch
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p2_embed"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        EMB = AR / "emb_paper"; EMB.mkdir(parents=True, exist_ok=True)
        print(f"### {STAGE} START | device={P.DEV}", flush=True)

        done = EMB / "embed_done.json"
        if done.exists():
            d = json.load(open(done))
            if all((EMB / f"{s}.npy").exists() and (EMB / f"{s}_meta.npz").exists() for s in P.SPLITS):
                print("ART_CHECK: embed_done — skip", flush=True)
                raise SystemExit(0)

        model, tok, name, dim = P.load_backbone()
        print(f"backbone: {name} | dim={dim} | params={sum(p.numel() for p in model.parameters())/1e6:.0f}M", flush=True)
        import pandas as pd

        for split in P.SPLITS:
            fin, meta = EMB / f"{split}.npy", EMB / f"{split}_meta.npz"
            df = pd.read_parquet(AR / "data_paper" / f"{split}.parquet")
            seqs = [P.norm_seq(s) for s in df[C["seq_col"]]]
            lens = np.array([len(s) for s in seqs], dtype=np.int64)
            if fin.exists() and meta.exists() and (np.load(meta)["lens"] == lens).all():
                print(f"  {split}: موجود و معتبر — skip", flush=True); continue
            tmp, prog = EMB / f"{split}.tmp.npy", EMB / f"{split}.progress"
            total = int(lens.sum())
            start = 0
            if tmp.exists() and prog.exists():
                start = int(prog.read_text())
                mm = np.lib.format.open_memmap(tmp, mode="r+")
                if mm.shape != (total, dim):
                    start, mm = 0, np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(total, dim))
            else:
                mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(total, dim))
            roff = np.concatenate([[0], np.cumsum(lens)[:-1]])
            t0 = time.time()
            with torch.no_grad():
                for k in range(start, len(seqs)):
                    h = P.embed_tokens(model, tok, seqs[k])
                    assert h.shape[0] == lens[k], (split, k, h.shape, lens[k])
                    mm[roff[k]:roff[k] + lens[k]] = h.cpu().numpy().astype(np.float16)
                    if (k + 1) % 500 == 0 or k + 1 == len(seqs):
                        mm.flush(); prog.write_text(str(k + 1))
                        print(f"  {split}: {k+1}/{len(seqs)} | {time.time()-t0:.0f}s", flush=True)
            mm.flush(); del mm
            samp = np.load(tmp, mmap_mode="r")[:: max(1, total // 50)]
            gate(bool(np.isfinite(samp.astype(np.float32)).all()), f"{split}: embeddings متناهی", hard=True)
            os.replace(tmp, fin)
            np.savez(meta, lens=lens, dim=np.int32(dim))
            prog.unlink(missing_ok=True)
            print(f"  saved {split}: rows={total} dim={dim} ({total*dim*2/1e9:.2f}GB)", flush=True)

        json.dump({"model": name, "dim": dim, "batch": 1, "truncation": "none (length-filtered)",
                   "len_range": [C["len_min"], C["len_max"]]}, open(done, "w"), indent=2)
        gate(True, f"embeddings کامل | {name} dim={dim}")
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P3 — cache فیزیکی ViennaRNA
bpp + MFE، multiprocessing، chunk-resume.

In [ ]:
# fmt: off
# ============================================================
# P3 — cache فیزیکی ViennaRNA (bpp از partition function + MFE) — multiprocessing + chunk-resume
# v1: تک‌پردازه و فقط per-split atomic بود. اینجا هر ۴۰۰ توالی یک chunk ذخیره می‌شود (قطع شدن session فقط همان chunk را می‌سوزاند)
# runtime: CPU | L≤200: ~0.05-0.15 s/seq → حدود ۱۰-۳۰ min با ۲ هسته | artifact: phys_paper/phys_{split}.npz
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p3_phys", restore="data_paper/*.parquet; config_paper.json",
                 save="phys_paper/phys_*.npz; phys_paper/chunks/*"):

        import json, multiprocessing as mp, os, sys, time
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import pandas as pd
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p3_phys"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        PH = AR / "phys_paper"; (PH / "chunks").mkdir(parents=True, exist_ok=True)
        CHUNK = 400
        print(f"### {STAGE} START | cpus={os.cpu_count()}", flush=True)
        import RNA  # noqa: E402


        def _fold(seq):
            import RNA as R
            L = len(seq)
            fc = R.fold_compound(seq)
            ss, mfe = fc.mfe()
            fc.exp_params_rescale(mfe)
            fc.pf()
            iu = P.tri(L)
            b = np.array(fc.bpp(), dtype=np.float32)[1:, 1:][iu].astype(np.float16)
            m = np.zeros((L, L), dtype=np.uint8)
            for (i, j) in P.parse_pairs(ss):
                m[i, j] = 1
            return b, m[iu]


        nproc = max(1, min(4, os.cpu_count() or 2))
        t0 = time.time()
        for split in P.SPLITS:
            out = PH / f"phys_{split}.npz"
            df = pd.read_parquet(AR / "data_paper" / f"{split}.parquet")
            seqs = [P.norm_seq(s) for s in df[C["seq_col"]]]
            lens = np.array([len(s) for s in seqs], dtype=np.int64)
            if out.exists():
                z = np.load(out)
                if (z["lens"] == lens).all():
                    print(f"  {split}: معتبر — skip", flush=True); continue
            nch = (len(seqs) + CHUNK - 1) // CHUNK
            for c in range(nch):
                cf = PH / "chunks" / f"{split}_{c:04d}.npz"
                if cf.exists():
                    continue
                part = seqs[c * CHUNK:(c + 1) * CHUNK]
                with mp.get_context("fork").Pool(nproc) as pool:
                    res = pool.map(_fold, part, chunksize=8)
                tmp = PH / "chunks" / f"{split}_{c:04d}.tmp.npz"
                np.savez_compressed(tmp, bpp=np.concatenate([r[0] for r in res]), mfe=np.concatenate([r[1] for r in res]))
                os.replace(tmp, cf)
                print(f"  {split}: chunk {c+1}/{nch} | {time.time()-t0:.0f}s", flush=True)
            parts = [np.load(PH / "chunks" / f"{split}_{c:04d}.npz") for c in range(nch)]
            np.savez_compressed(PH / f"phys_{split}.tmp.npz", bpp_flat=np.concatenate([p["bpp"] for p in parts]),
                                mfe_flat=np.concatenate([p["mfe"] for p in parts]), lens=lens)
            os.replace(PH / f"phys_{split}.tmp.npz", out)
            for c in range(nch):
                (PH / "chunks" / f"{split}_{c:04d}.npz").unlink(missing_ok=True)
            print(f"  {split}: saved ({len(seqs)} seqs)", flush=True)

        for s in P.SPLITS:
            z = np.load(PH / f"phys_{s}.npz")
            npair = int((z["lens"] * (z["lens"] - 1) // 2).sum())
            gate(len(z["mfe_flat"]) == npair and len(z["bpp_flat"]) == npair, f"{s}: اندازهٔ cache درست ({npair} جفت)")
        gate(True, f"phys cache کامل | {time.time()-t0:.0f}s")
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P4 — baseline RNAfold + «گزارش اختلاف»
RNAfold ما در برابر عدد منتشرشده (فقط گزارش اختلاف، نه PASS)؛ counts برای bootstrap.

In [ ]:
# fmt: off
# ============================================================
# P4 — baseline فیزیکی RNAfold روی همان توالی‌ها + «گزارش اختلاف» با عدد منتشرشدهٔ CHANRG (v3: بدون PASS)
#  (v1: یک checkpoint رسمی با داده‌ی آموزشی نامعلوم/هم‌پوشان ارزیابی می‌شد؛ نشت و نسبت‌گیری بی‌معنی.
#   اینجا بازتولید = «RNAfold ما ≈ RNAfold منتشرشده» ؛ ارزان و بدون نشت. آستانه‌ی bpp فقط روی val.)
# runtime: CPU | ~1 min | artifact: artifacts_paper/ref_rnafold.json + counts_rnafold.npz
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p4_reference", restore="data_paper/*.parquet; phys_paper/phys_*.npz; config_paper.json",
                 save="artifacts_paper/ref_rnafold.json; artifacts_paper/counts_rnafold.npz"):

        import json, sys
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p4_reference"
        gate = P.gate_for(STAGE)
        OUT = AR / "artifacts_paper"; OUT.mkdir(exist_ok=True)
        print(f"### {STAGE} START", flush=True)
        sp = {n: P.Split(n, emb=False, phys=True) for n in ["val"] + P.EVAL}
        res = {"mfe": {}, "bpp": {}}
        cnt = {}
        # MFE
        for n in P.EVAL:
            s = sp[n]; S = []
            for i in range(len(s)):
                o, L = int(s.poff[i]), int(s.L[i]); m = L * (L - 1) // 2
                S.append(s.mfe[o:o + m].astype(np.float16))
            G = [s.gt_flat(i) for i in range(len(s))]
            tp, fp, fn = P.counts_at(S, G, 0.5)
            p, r, f1 = P.prf(tp, fp, fn)
            res["mfe"][n] = {"f1": round(f1, 4), "p": round(p, 4), "r": round(r, 4)}
            for k, v in dict(tp=tp, fp=fp, fn=fn).items():
                cnt[f"mfe_{n}_{k}"] = v
        # bpp با آستانه‌ی val
        def bpp_list(s):
            return [s.bpp[int(s.poff[i]):int(s.poff[i]) + int(s.L[i]) * (int(s.L[i]) - 1) // 2] for i in range(len(s))]
        Sv = bpp_list(sp["val"]); Gv = [sp["val"].gt_flat(i) for i in range(len(sp["val"]))]
        thr, vf = P.best_thr(Sv, Gv)
        res["bpp_thr"] = thr
        for n in P.EVAL:
            s = sp[n]; S = bpp_list(s); G = [s.gt_flat(i) for i in range(len(s))]
            tp, fp, fn = P.counts_at(S, G, thr)
            p, r, f1 = P.prf(tp, fp, fn)
            res["bpp"][n] = {"f1": round(f1, 4), "p": round(p, 4), "r": round(r, 4)}
            for k, v in dict(tp=tp, fp=fp, fn=fn).items():
                cnt[f"bpp_{n}_{k}"] = v
        for key in ["mfe", "bpp"]:
            res[key]["ood_mean"] = round(float(np.mean([res[key][n]["f1"] for n in P.OOD])), 4)
        np.savez(OUT / "counts_rnafold.npz", **cnt)
        pub = P.PUBLISHED["rnafold_ood"]
        res["published_rnafold_ood"] = pub
        res["note"] = ("عدد منتشرشده از پروتکل رسمی CHANRG (متریک و طول رسمی)؛ اینجا micro-F1 جفت‌ها روی توالی‌های 50..200 و زیرمجموعهٔ cap‌شده. "
                       "عدد منتشرشده را قبل از ارجاع در مقاله از خود مقاله تأیید کنید.")
        json.dump(res, open(OUT / "ref_rnafold.json", "w"), indent=2)
        print(json.dumps({k: res[k] for k in ["mfe", "bpp", "bpp_thr"]}, indent=2), flush=True)
        d = res["mfe"]["ood_mean"] - pub
        res["sanity_report"] = {"ours_mfe_ood": res["mfe"]["ood_mean"], "published": pub, "delta": round(d, 4),
                                "ours_bpp_ood": res["bpp"]["ood_mean"]}
        json.dump(res, open(OUT / "ref_rnafold.json", "w"), indent=2)
        # v3: این «گزارش اختلاف» است نه PASS/FAIL؛ تلورانس ±0.05 هم‌اندازهٔ اثرهای سنجیده‌شده بود و چیزی را تأیید نمی‌کرد.
        print(f"INFO | RNAfold-MFE OOD ما={res['mfe']['ood_mean']} در برابر منتشرشده {pub} → اختلاف {d:+.3f} "
              f"(طول/متریک/زیرمجموعه فرق دارد؛ فقط «هم‌مرتبه» بودن را نشان می‌دهد، نه هم‌ترازی پروتکل)", flush=True)
        # ---- v4: لنگر ادبیات. عدد مرجع صحیح = RNAfold در تحلیل length-matched 50..200 مقالهٔ CHANRG (arXiv:2603.22330)، نه OOD تمام‌طول (0.2886).
        LIT_RNAFOLD_LM = {"test": 0.32, "gena": 0.4193, "genc": 0.28, "genf": 0.28}   # GenA از متن؛ بقیه از heatmap با ۲ رقم اعشار
        def _macro(n, pre):
            tp, fp, fn = [cnt[f"{pre}_{n}_{k}"].astype(float) for k in ("tp", "fp", "fn")]
            has = (tp + fn) > 0
            return float(np.mean((2 * tp / np.maximum(2 * tp + fp + fn, 1e-9))[has]))
        anchor = {}
        print("\nلنگر ادبیات — RNAfold-MFE (ما: زیرمجموعهٔ cap، 50..200) در برابر منتشرشده (length-matched 50..200، splitهای کامل)", flush=True)
        for n in P.EVAL:
            mi, ma = res["mfe"][n]["f1"], _macro(n, "mfe")
            anchor[n] = {"micro": round(mi, 4), "macro": round(ma, 4), "published": LIT_RNAFOLD_LM[n],
                         "d_micro": round(mi - LIT_RNAFOLD_LM[n], 3), "d_macro": round(ma - LIT_RNAFOLD_LM[n], 3)}
            print(f"  {n:>5}: micro={mi:.3f} macro={ma:.3f} | منتشرشده={LIT_RNAFOLD_LM[n]:.3f} | Δmicro={mi-LIT_RNAFOLD_LM[n]:+.3f} Δmacro={ma-LIT_RNAFOLD_LM[n]:+.3f}", flush=True)
        res["anchor_lenmatched"] = anchor
        json.dump(res, open(OUT / "ref_rnafold.json", "w"), indent=2)
        worst = max(min(abs(a["d_micro"]), abs(a["d_macro"])) for a in anchor.values())
        print(f"INFO | بدترین |Δ| (بهترین تجمیع هر split) = {worst:.3f} → " + ("هم‌تراز با ادبیات (≤0.03)" if worst <= 0.03 else "فاصله > 0.03: split/طول/تجمیع را بازبینی کنید"), flush=True)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P5 — مدل‌های اصلی
unet+phys / unet / nolm+phys، ۳ seed، آستانه‌ی val، counts per-sequence.

In [ ]:
# fmt: off
# ============================================================
# P5 — مدل اصلی: giga frozen (کش P2) + U-Net + کانال‌های مکملیت/فیزیک ؛ سه arm × ۳ seed
#   unet+phys : LM + مکملیت + bpp/mfe     (مدل اصلی)
#   unet      : LM + مکملیت               (کنترل: سهم فیزیک)
#   nolm+phys : one-hot + مکملیت + فیزیک  (کنترل: سهم LM — کنترلی که در v1 جا افتاده بود)
# همه: همان U-Net (فقط ورودی wz فرق می‌کند) ، pw=8 ، 2 epoch ، آستانه فقط روی val (grid تا 0.95)،
# resume per (arm, seed) با fingerprint پروتکل. counts per-sequence برای bootstrap ذخیره می‌شود.
# runtime: GPU T4 | هر run حدود ۸-۱۵ min (برآورد؛ از لاگ seed اول تنظیم کنید) → ۹ run ≈ ۱.۵-۲.۵ h
# env: RNA_SEEDS="0,1,2" ، RNA_EPOCHS=2
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p5_main", restore="data_paper/*.parquet; emb_paper/*.npy; emb_paper/*_meta.npz; phys_paper/phys_*.npz; config_paper.json",
                 save="artifacts_paper/main_ckpt/*; artifacts_paper/results_main.json"):

        import json, os, sys, time
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import torch
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p5_main"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        SEEDS = [int(x) for x in os.environ.get("RNA_SEEDS", "0" if P.QUICK else "0,1,2").split(",")]
        EPOCHS = int(os.environ.get("RNA_EPOCHS", 1 if P.QUICK else 2))
        OUT = AR / "artifacts_paper"; CK = OUT / "main_ckpt"; CK.mkdir(parents=True, exist_ok=True)
        print(f"### {STAGE} START | seeds={SEEDS} epochs={EPOCHS} device={P.DEV} amp={P.AMP}", flush=True)

        splits = {n: P.Split(n) for n in P.SPLITS}
        dim = splits["train"].dim
        bb = json.load(open(AR / "emb_paper" / "embed_done.json"))["model"]
        PROTO = {"epochs": EPOCHS, "pw": P.PW, "n_train": len(splits["train"]), "thr_grid": P.THR_GRID,
                 "len": [C["len_min"], C["len_max"]], "backbone": bb, "n_phys": splits["train"].n_phys(), "unet_base": P.UNET_BASE}
        t0 = time.time()
        for arm, spec in P.ARMS.items():
            for seed in SEEDS:
                ck, cz = CK / f"{arm}_s{seed}.json", CK / f"{arm}_s{seed}.npz"
                if ck.exists() and cz.exists() and json.load(open(ck)).get("_proto") == PROTO:
                    print(f"[{arm} s{seed}] از checkpoint (پروتکل یکسان) — skip", flush=True); continue
                torch.manual_seed(seed); np.random.seed(seed)
                model = P.build_arm(spec, dim, splits["train"])
                npar = sum(p.numel() for p in model.parameters())
                print(f"[{arm} s{seed}] params={npar/1e3:.0f}K | شروع آموزش", flush=True)
                P.train_decoder(model, spec, splits["train"], seed, n_seq=EPOCHS * len(splits["train"]), accum=1, tag=f"{arm} s{seed}")
                res, cnt = P.evaluate_arm(model, spec, splits)
                res.update({"arm": arm, "seed": seed, "n_params": npar, "_proto": PROTO, "elapsed_s": round(time.time() - t0)})
                np.savez(cz, **cnt)
                json.dump(res, open(ck, "w"), indent=2)
                print(f"[{arm} s{seed}] val={res['val_f1']} thr={res['thr']} OOD={res['ood_mean']} Test={res['splits']['test']['f1']} "
                      f"GenC={res['splits']['genc']['f1']} | {time.time()-t0:.0f}s", flush=True)
                del model
                torch.cuda.empty_cache()

        # ---- تجمیع
        summary = {"protocol": PROTO, "seeds": SEEDS, "arms": {}}
        for arm in P.ARMS:
            rs = [json.load(open(CK / f"{arm}_s{s}.json")) for s in SEEDS if (CK / f"{arm}_s{s}.json").exists()]
            if not rs:
                continue
            summary["arms"][arm] = {
                "n_seeds": len(rs), "n_params": rs[0]["n_params"],
                "ood_mean": {"mean": round(float(np.mean([r["ood_mean"] for r in rs])), 4), "std": round(float(np.std([r["ood_mean"] for r in rs])), 4),
                             "per_seed": [r["ood_mean"] for r in rs]},
                "splits": {n: {k: {"mean": round(float(np.mean([r["splits"][n][k] for r in rs])), 4),
                                   "std": round(float(np.std([r["splits"][n][k] for r in rs])), 4)}
                               for k in ["f1", "p", "r", "f1_onepair", "f1_oracle", "auprc"]} for n in P.EVAL},
                "thr": [r["thr"] for r in rs]}
        json.dump(summary, open(OUT / "results_main.json", "w"), indent=2)
        print("\narm | Test | GenA | GenC | GenF | OOD mean (mean±std over seeds)", flush=True)
        for a, v in summary["arms"].items():
            s = v["splits"]
            print(f"{a:>10} | {s['test']['f1']['mean']:.3f} | {s['gena']['f1']['mean']:.3f} | {s['genc']['f1']['mean']:.3f} | "
                  f"{s['genf']['f1']['mean']:.3f} | {v['ood_mean']['mean']:.3f}±{v['ood_mean']['std']:.3f}", flush=True)
        edge = [a for a, v in summary["arms"].items() if any(t >= P.THR_GRID[-1] for t in v["thr"])]
        gate(len(summary["arms"]) == len(P.ARMS), f"همهٔ armها اجرا شدند ({len(SEEDS)} seed)" +
             (f" | آستانهٔ لبهٔ grid در {edge}" if edge else ""), hard=False)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P2r — embedding با LM «وزن‌تصادفی» (جدید در v5)
کنترلی که «سهم دانش پیش‌آموزش» را از «صرفاً ورودی ۱۲۸۰بُعدی» جدا می‌کند. اگر کل سودِ LM با LM تصادفی هم دیده شود، سود ID از دانش پیش‌آموزش نیست. ~۱۵–۲۵ دقیقه، ~۴.۵GB دیسک.
**قبل از اجرا P0 را دوباره بزنید** تا plib به‌روز (emb_dir و load_random_backbone) شود.


In [ ]:
# fmt: off
# ============================================================
# P2r (v5) — embedding با RiNALMo-giga «وزن تصادفی» (fp16، همان مسیر P2) — کنترل برای تفکیک «دانش پیش‌آموزش» از «ورودی پرابعاد»
# فیکس‌های v1: offset بر حسب «سطر» (نه عنصر) ، assert درست ، بدون خواندن کل split در RAM ، seq_col از config
# runtime: GPU T4 | ~15-25 min (config از کش HF؛ مدل تصادفی ساخته می‌شود) | artifact: emb_paper/{split}.npy (N_res × dim fp16) + {split}_meta.npz
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p2r_embed_random", restore="data_paper/*.parquet; config_paper.json",
                 save="emb_rand/*.npy; emb_rand/*_meta.npz; emb_rand/embed_done.json; emb_rand/*.tmp.npy; emb_rand/*.progress"):

        import json, os, sys, time
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import torch
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p2r_embed_random"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        EMB = AR / "emb_rand"; EMB.mkdir(parents=True, exist_ok=True)
        print(f"### {STAGE} START | device={P.DEV}", flush=True)

        done = EMB / "embed_done.json"
        if done.exists():
            d = json.load(open(done))
            if all((EMB / f"{s}.npy").exists() and (EMB / f"{s}_meta.npz").exists() for s in P.SPLITS):
                print("ART_CHECK: embed_done — skip", flush=True)
                raise SystemExit(0)

        RAND_SEED = int(os.environ.get("RNA_RAND_SEED", "0"))
        model, tok, name, dim = P.load_random_backbone(RAND_SEED)
        print(f"backbone: {name} | dim={dim} | params={sum(p.numel() for p in model.parameters())/1e6:.0f}M", flush=True)
        import pandas as pd

        for split in P.SPLITS:
            fin, meta = EMB / f"{split}.npy", EMB / f"{split}_meta.npz"
            df = pd.read_parquet(AR / "data_paper" / f"{split}.parquet")
            seqs = [P.norm_seq(s) for s in df[C["seq_col"]]]
            lens = np.array([len(s) for s in seqs], dtype=np.int64)
            if fin.exists() and meta.exists() and (np.load(meta)["lens"] == lens).all():
                print(f"  {split}: موجود و معتبر — skip", flush=True); continue
            tmp, prog = EMB / f"{split}.tmp.npy", EMB / f"{split}.progress"
            total = int(lens.sum())
            start = 0
            if tmp.exists() and prog.exists():
                start = int(prog.read_text())
                mm = np.lib.format.open_memmap(tmp, mode="r+")
                if mm.shape != (total, dim):
                    start, mm = 0, np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(total, dim))
            else:
                mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(total, dim))
            roff = np.concatenate([[0], np.cumsum(lens)[:-1]])
            t0 = time.time()
            with torch.no_grad():
                for k in range(start, len(seqs)):
                    h = P.embed_tokens(model, tok, seqs[k])
                    assert h.shape[0] == lens[k], (split, k, h.shape, lens[k])
                    mm[roff[k]:roff[k] + lens[k]] = h.cpu().numpy().astype(np.float16)
                    if (k + 1) % 500 == 0 or k + 1 == len(seqs):
                        mm.flush(); prog.write_text(str(k + 1))
                        print(f"  {split}: {k+1}/{len(seqs)} | {time.time()-t0:.0f}s", flush=True)
            mm.flush(); del mm
            samp = np.load(tmp, mmap_mode="r")[:: max(1, total // 50)]
            gate(bool(np.isfinite(samp.astype(np.float32)).all()), f"{split}: embeddings متناهی", hard=True)
            _sd = float(samp.astype(np.float32).std(0).mean())
            print(f"  {split}: میانگین std هر بُعد روی موقعیت‌ها = {_sd:.4f} (≈0 یعنی embedding فروریخته)", flush=True)
            gate(_sd > 1e-3, f"{split}: embedding تصادفی فروپاشیده نیست (std={_sd:.4f})", hard=False)
            os.replace(tmp, fin)
            np.savez(meta, lens=lens, dim=np.int32(dim))
            prog.unlink(missing_ok=True)
            print(f"  saved {split}: rows={total} dim={dim} ({total*dim*2/1e9:.2f}GB)", flush=True)

        json.dump({"model": name, "dim": dim, "batch": 1, "truncation": "none (length-filtered)",
                   "len_range": [C["len_min"], C["len_max"]]}, open(done, "w"), indent=2)
        gate(True, f"embeddings کامل | {name} dim={dim}")
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P5r — arm «randlm+phys» (جدید در v5)
همان آموزش/ارزیابی `unet+phys` با embedding تصادفی. `RNA_SEEDS` را مثل P5 بگذارید (مرحلهٔ B: `0,1,2`).


In [ ]:
# fmt: off
# ============================================================
# P5r — arm «randlm+phys»: همان decoder/ورودی‌ها/آموزش/ارزیابی unet+phys، ولی embeddingها از RiNALMo با وزن «تصادفی» (P2r)
#   سه‌گانهٔ تفکیک:  nolm+phys (one-hot ۵بُعدی) → randlm+phys (۱۲۸۰بُعدی تصادفی، بدون دانش) → unet+phys (پیش‌آموزش‌دیده)
#   C6 = unet+phys − randlm+phys  : سهم «دانش پیش‌آموزش»        C7 = randlm+phys − nolm+phys : سهم «ورودی پرابعاد تصادفی» (ظرفیت/ساختار ورودی)
# runtime: GPU T4 | هر seed ≈ همان زمان یک run از P5 (~۱۰ min) | artifact: main_ckpt/randlm+phys_s{seed}.{json,npz}
# resume per seed با fingerprint پروتکل (seed 0 را در حالت A اجرا کنید؛ بعد 0,1,2)
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p5r_randlm", restore="data_paper/*.parquet; emb_rand/*.npy; emb_rand/*_meta.npz; phys_paper/phys_*.npz; config_paper.json",
                 save="artifacts_paper/main_ckpt/randlm+phys_s*"):

        import json, os, sys, time
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import torch
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p5r_randlm"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        SEEDS = [int(x) for x in os.environ.get("RNA_SEEDS", "0" if P.QUICK else "0,1,2").split(",")]
        EPOCHS = int(os.environ.get("RNA_EPOCHS", 1 if P.QUICK else 2))
        OUT = AR / "artifacts_paper"; CK = OUT / "main_ckpt"; CK.mkdir(parents=True, exist_ok=True)
        print(f"### {STAGE} START | seeds={SEEDS} epochs={EPOCHS}", flush=True)

        splits = {n: P.Split(n, emb_dir="emb_rand") for n in P.SPLITS}
        dim = splits["train"].dim
        bb = json.load(open(AR / "emb_rand" / "embed_done.json"))["model"]
        PROTO = {"epochs": EPOCHS, "pw": P.PW, "n_train": len(splits["train"]), "thr_grid": P.THR_GRID,
                 "len": [C["len_min"], C["len_max"]], "backbone": bb, "n_phys": splits["train"].n_phys(), "unet_base": P.UNET_BASE}
        arm, spec = "randlm+phys", dict(P.ARMS["unet+phys"])
        t0 = time.time()
        for seed in SEEDS:
            ck, cz = CK / f"{arm}_s{seed}.json", CK / f"{arm}_s{seed}.npz"
            if ck.exists() and cz.exists() and json.load(open(ck)).get("_proto") == PROTO:
                print(f"[{arm} s{seed}] از checkpoint (پروتکل یکسان) — skip", flush=True); continue
            torch.manual_seed(seed); np.random.seed(seed)
            model = P.build_arm(spec, dim, splits["train"])
            npar = sum(p.numel() for p in model.parameters())
            print(f"[{arm} s{seed}] params={npar/1e3:.0f}K | شروع آموزش", flush=True)
            P.train_decoder(model, spec, splits["train"], seed, n_seq=EPOCHS * len(splits["train"]), accum=1, tag=f"{arm} s{seed}")
            res, cnt = P.evaluate_arm(model, spec, splits)
            res.update({"arm": arm, "seed": seed, "n_params": npar, "_proto": PROTO, "elapsed_s": round(time.time() - t0)})
            np.savez(cz, **cnt)
            json.dump(res, open(ck, "w"), indent=2)
            print(f"[{arm} s{seed}] val={res['val_f1']} thr={res['thr']} OOD={res['ood_mean']} Test={res['splits']['test']['f1']} "
                  f"GenC={res['splits']['genc']['f1']} | {time.time()-t0:.0f}s", flush=True)
            del model
            torch.cuda.empty_cache()

        rs = [json.load(open(CK / f"{arm}_s{s}.json")) for s in SEEDS if (CK / f"{arm}_s{s}.json").exists()]
        gate(len(rs) == len(SEEDS), f"{arm}: {len(rs)}/{len(SEEDS)} seed کامل", hard=False)
        if rs:
            mean = lambda n: float(np.mean([r["splits"][n]["f1"] for r in rs]))
            print(f"\n{arm} | Test {mean('test'):.3f} | GenA {mean('gena'):.3f} | GenC {mean('genc'):.3f} | GenF {mean('genf'):.3f} | OOD {np.mean([r['ood_mean'] for r in rs]):.3f}", flush=True)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P5s — منحنی مقیاس (جدید در v5)
دو arm (`unet+phys`, `nolm+phys`) روی ۱۵۰۰/۳۰۰۰/۶۰۰۰ توالی (+ نقطهٔ ۱۲۰۰۰ از P5). برای تفسیر «ID با داده بالا می‌رود، OOD نه».


In [ ]:
# fmt: off
# ============================================================
# P5s — منحنی مقیاس: همان decoder و همان arms (unet+phys, nolm+phys) روی زیرمجموعه‌های تو‌در‌توی تصادفی از train (۱۵۰۰/۳۰۰۰/۶۰۰۰ + نقطهٔ ۱۲۰۰۰ از P5)
#   epoch ثابت (⇒ تعداد update با N بزرگ می‌شود)، آستانه روی همان val، eval بدون categories. خروجی P10: شیب «به‌ازای دو برابر شدن داده» برای ID و OOD.
# runtime: GPU T4 | ≈ ۱۵-۲۵ min به‌ازای هر (arm, seed) برای هر سه اندازه (برآورد) → ۲ arm × ۲ seed ≈ ۱-۱.۵ h
# env: RNA_SCALE_SIZES="1500,3000,6000"  RNA_SCALE_SEEDS="0,1"
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p5s_scale", restore="data_paper/*.parquet; emb_paper/*.npy; emb_paper/*_meta.npz; phys_paper/phys_*.npz; config_paper.json",
                 save="artifacts_paper/scale_ckpt/*"):

        import json, os, sys, time
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import torch
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p5s_scale"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        SIZES = [int(x) for x in os.environ.get("RNA_SCALE_SIZES", "300" if P.QUICK else "1500,3000,6000").split(",")]
        SEEDS = [int(x) for x in os.environ.get("RNA_SCALE_SEEDS", "0" if P.QUICK else "0,1").split(",")]
        EPOCHS = int(os.environ.get("RNA_EPOCHS", 1 if P.QUICK else 2))
        ARMS_S = ["unet+phys", "nolm+phys"]
        OUT = AR / "artifacts_paper"; SCK = OUT / "scale_ckpt"; SCK.mkdir(parents=True, exist_ok=True)
        print(f"### {STAGE} START | sizes={SIZES} seeds={SEEDS} epochs={EPOCHS}", flush=True)

        splits = {n: P.Split(n) for n in P.SPLITS}
        dim = splits["train"].dim
        bb = json.load(open(AR / "emb_paper" / "embed_done.json"))["model"]
        perm = np.random.RandomState(123).permutation(len(splits["train"]))     # زیرمجموعه‌های تو‌در‌تو: perm[:N]


        class _Sub:
            """نمای «N توالی» از train (همان API که train_decoder/build_arm لازم دارد)"""
            def __init__(self, sp, ix):
                self.sp, self.ix = sp, np.asarray(ix)
                self.L = sp.L[self.ix]
                self.fam = [sp.fam[int(i)] for i in self.ix]
            def __len__(self): return len(self.ix)
            def gt_flat(self, i): return self.sp.gt_flat(int(self.ix[i]))
            def h(self, i): return self.sp.h(int(self.ix[i]))
            def onehot(self, i): return self.sp.onehot(int(self.ix[i]))
            def canon_t(self, i): return self.sp.canon_t(int(self.ix[i]))
            def phys_t(self, i): return self.sp.phys_t(int(self.ix[i]))
            def n_phys(self): return self.sp.n_phys()


        t0 = time.time()
        for N in SIZES:
            assert N <= len(splits["train"]), f"N={N} > اندازهٔ train"
            sub = _Sub(splits["train"], perm[:N])
            print(f"--- N={N} | family در زیرمجموعه: {len(set(sub.fam))} | میانگین طول {float(sub.L.mean()):.0f}", flush=True)
            PROTO = {"epochs": EPOCHS, "N": N, "pw": P.PW, "thr_grid": P.THR_GRID, "backbone": bb, "unet_base": P.UNET_BASE, "perm_seed": 123}
            for arm in ARMS_S:
                spec = P.ARMS[arm]
                for seed in SEEDS:
                    ck, cz = SCK / f"{arm}_n{N}_s{seed}.json", SCK / f"{arm}_n{N}_s{seed}.npz"
                    if ck.exists() and cz.exists() and json.load(open(ck)).get("_proto") == PROTO:
                        print(f"[{arm} N={N} s{seed}] از checkpoint — skip", flush=True); continue
                    torch.manual_seed(seed); np.random.seed(seed)
                    model = P.build_arm(spec, dim, sub)
                    P.train_decoder(model, spec, sub, seed, n_seq=EPOCHS * N, accum=1, tag=f"{arm} N={N} s{seed}", log_every=5000)
                    res, cnt = P.evaluate_arm(model, spec, splits, with_categories=False)
                    res.update({"arm": arm, "seed": seed, "N": N, "_proto": PROTO, "elapsed_s": round(time.time() - t0)})
                    np.savez(cz, **cnt); json.dump(res, open(ck, "w"), indent=2)
                    print(f"[{arm} N={N} s{seed}] val={res['val_f1']} thr={res['thr']} Test={res['splits']['test']['f1']} OOD={res['ood_mean']} | {time.time()-t0:.0f}s", flush=True)
                    del model
                    torch.cuda.empty_cache()

        print("\nN | arm | Test | OOD (میانگین seedها)", flush=True)
        for N in SIZES:
            for arm in ARMS_S:
                rs = [json.load(open(f)) for f in sorted(SCK.glob(f"{arm}_n{N}_s*.json"))]
                if rs: print(f"{N:>6} | {arm:<10} | {np.mean([r['splits']['test']['f1'] for r in rs]):.3f} | {np.mean([r['ood_mean'] for r in rs]):.3f} ({len(rs)} seed)", flush=True)
        gate(True, "منحنی مقیاس ذخیره شد (تفسیر در P10)", hard=False)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

> **اختیاری در v5:** اگر P6/P6b را در v4 اجرا کرده‌اید و سشن زنده است، رد شوند (skip می‌شوند). برای سشن نو لازم نیستند مگر بخواهید C5 را دوباره ببینید.

## P6 — LoRA در برابر frozen (matched)
در یک سلول، با gate گرادیان و هم‌ارزی اولیه. v3: ۳ seed (seedهای قبلی از checkpoint).

In [ ]:
# fmt: off
# ============================================================
# P6 — H4 در رژیم decoder قوی: LoRA در برابر frozen «matched» داخل همین سلول
#   v1: frozen نداشت، decoder بدون مکملیت، 2000 توالی در برابر 24000، thr ثابت 0.5، و ss به‌جای seq به tokenizer می‌رفت.
#   اینجا: هر seed هر دو بازو با همان decoder (unet+phys) ، همان init/ترتیب داده/تعداد توالی، آستانه‌ی val ، همان ارزیابی.
# LoRA: r=8 روی query/value چهار لایهٔ آخر ؛ giga fp16 + adapter fp32 ؛ gate: |g_lora|>0 ، ‖B‖>0 ، هم‌ارزی اولیه با کش
# runtime: GPU T4 | هر seed حدود ۲۰-۳۵ min (frozen ≈۳ + LoRA ≈۱۵-۳۰؛ برآورد) | artifact: results_lora.json
# env: RNA_LORA_STEPS=1500 (هر step = ۲ توالی) ، RNA_LORA_SEEDS="0,1,2"  (v3: ۳ seed؛ seedهای 0,1 از v2 با همان PROTO skip می‌شوند)
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p6_lora", restore="data_paper/*.parquet; emb_paper/*.npy; emb_paper/*_meta.npz; phys_paper/phys_*.npz; config_paper.json",
                 save="artifacts_paper/lora_ckpt/*; artifacts_paper/results_lora.json"):

        import json, os, re, sys, time
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import torch
        import torch.nn as nn
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p6_lora"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        SEEDS = [int(x) for x in os.environ.get("RNA_LORA_SEEDS", "0" if P.QUICK else "0,1,2").split(",")]
        STEPS = int(os.environ.get("RNA_LORA_STEPS", 40 if P.QUICK else 1500))
        ACC, RANK, LORA_LR = 2, 8, 2e-4
        OUT = AR / "artifacts_paper"; OUT.mkdir(exist_ok=True)
        CK = OUT / "lora_ckpt"; CK.mkdir(exist_ok=True)
        print(f"### {STAGE} START | seeds={SEEDS} steps={STEPS}×{ACC} device={P.DEV}", flush=True)
        from peft import LoraConfig, get_peft_model

        splits = {n: P.Split(n) for n in P.SPLITS}
        dim = splits["train"].dim
        spec = P.ARMS["unet+phys"]
        PROTO = {"steps": STEPS, "acc": ACC, "rank": RANK, "lora_lr": LORA_LR, "pw": P.PW, "n_train": len(splits["train"]),
                 "thr_grid": P.THR_GRID, "backbone": json.load(open(AR / "emb_paper" / "embed_done.json"))["model"]}
        t0 = time.time()
        all_res = {"protocol": PROTO, "seeds": {}}
        for seed in SEEDS:
            ck = CK / f"seed{seed}.json"
            if ck.exists() and json.load(open(ck)).get("_proto") == PROTO:
                print(f"[seed {seed}] checkpoint — skip", flush=True)
                all_res["seeds"][str(seed)] = json.load(open(ck)); continue
            rec = {"_proto": PROTO, "seed": seed}
            # ---------------- frozen matched
            torch.manual_seed(seed); np.random.seed(seed)
            dec = P.build_arm(spec, dim, splits["train"])
            P.train_decoder(dec, spec, splits["train"], seed, n_seq=STEPS * ACC, accum=ACC, tag=f"frozen s{seed}", log_every=500)
            rf, cf = P.evaluate_arm(dec, spec, splits, with_categories=False)
            np.savez(CK / f"counts_frozen_matched_s{seed}.npz", **cf); rec["frozen"] = rf
            print(f"[seed {seed}] FROZEN: OOD={rf['ood_mean']} Test={rf['splits']['test']['f1']} thr={rf['thr']} | {time.time()-t0:.0f}s", flush=True)
            del dec

            # ---------------- LoRA matched (همان init decoder و همان ترتیب داده)
            base, tok, name, d2 = P.load_backbone()
            assert d2 == dim, (d2, dim)
            lin = [n for n, m in base.named_modules() if isinstance(m, nn.Linear)]
            idx = [(n, int(re.search(r"\.(\d+)\.", n).group(1))) for n in lin if re.search(r"\.(\d+)\.", n)]
            gate(len(idx) > 0, "لایه‌های شماره‌دار پیدا شد")
            mx = max(i for _, i in idx)
            cands = [n for n, i in idx if re.search(r"(query|value|q_proj|v_proj)", n, re.I) and i >= mx - 3]
            gate(len(cands) > 0, f"{len(cands)} ماژول LoRA در لایه‌های {mx-3}..{mx}")
            torch.manual_seed(seed)
            pm = get_peft_model(base, LoraConfig(r=RANK, lora_alpha=2 * RANK, lora_dropout=0.05, target_modules=cands, bias="none")).to(P.DEV)
            lora_params = [p for p in pm.parameters() if p.requires_grad]
            print(f"  adapter: {sum(p.numel() for p in lora_params)/1e3:.1f}K پارامتر", flush=True)

            def hfn(sp, i, _pm=pm, _tok=tok):
                return P.embed_tokens(_pm, _tok, sp.seqs[i])

            # هم‌ارزی اولیه با کش (B=0)
            pm.eval()
            with torch.no_grad():
                rel = []
                for i in range(3):
                    a, b = hfn(splits["val"], i), splits["val"].h(i)
                    rel.append(float((a - b).abs().max() / b.abs().mean().clamp(min=1e-6)))
            gate(max(rel) < 0.05, f"هم‌ارزی اولیهٔ LoRA با کش: max rel diff={max(rel):.4f} (کوچک = ورودی دو بازو یکسان)", hard=False)

            torch.manual_seed(seed); np.random.seed(seed)
            dec = P.build_arm(spec, dim, splits["train"])
            pm.train()
            diag = P.train_decoder(dec, spec, splits["train"], seed, n_seq=STEPS * ACC, accum=ACC, tag=f"lora s{seed}", log_every=250,
                                   hfn=lambda sp, i: (pm.train(), hfn(sp, i))[1], extra_groups=[{"params": lora_params, "lr": LORA_LR}])
            pm.eval()
            bnorm = float(sum(p.detach().float().norm() for n, p in pm.named_parameters() if "lora_B" in n))
            gate(diag["min_grad_norm_lora"] is not None and diag["final_grad_norm_lora"] > 0 and bnorm > 0,
                 f"LoRA واقعاً آموزش دید | |g| نهایی={diag['final_grad_norm_lora']:.2e} ‖B‖={bnorm:.3f}")
            rl, cl = P.evaluate_arm(dec, spec, splits, hfn=hfn, with_categories=False)
            np.savez(CK / f"counts_lora_matched_s{seed}.npz", **cl)
            rec["lora"] = rl; rec["lora_diag"] = {**diag, "B_norm": round(bnorm, 4)}
            d_ood = rl["ood_mean"] - rf["ood_mean"]; d_test = rl["splits"]["test"]["f1"] - rf["splits"]["test"]["f1"]
            rec["delta"] = {"ood": round(d_ood, 4), "test": round(d_test, 4)}
            print(f"[seed {seed}] LoRA: OOD={rl['ood_mean']} Test={rl['splits']['test']['f1']} thr={rl['thr']} | ΔOOD={d_ood:+.4f} ΔTest={d_test:+.4f} | {time.time()-t0:.0f}s", flush=True)
            json.dump(rec, open(ck, "w"), indent=2)
            all_res["seeds"][str(seed)] = rec
            del pm, base, dec
            torch.cuda.empty_cache()

        d = [v["delta"]["ood"] for v in all_res["seeds"].values()]
        all_res["delta_ood_per_seed"] = d
        all_res["delta_ood_mean"] = round(float(np.mean(d)), 4)
        json.dump(all_res, open(OUT / "results_lora.json", "w"), indent=2)
        gate(True, f"LoRA − frozen (matched): ΔOOD per seed={d} (فقط با bootstrap خانواده‌ای P7 قابل‌تفسیر است؛ seed-std نویز نمونه‌گیری را نمی‌گیرد)", hard=False)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P6b — کنترل ظرفیت: adapter هم‌پارامتر (جدید در v3)
منظور: جدا کردن «ترمیم بازنمایی» از «ظرفیت آموزش‌پذیر بیشتر».

In [ ]:
# fmt: off
# ============================================================
# P6b — (جدید در v3) کنترل ظرفیت برای LoRA: adapter هم‌پارامتر روی embeddingهای frozen
#   پرسش: «LoRA بازنمایی را ترمیم می‌کند» یا فقط «۱۶۴ هزار پارامتر آموزش‌پذیر بیشتر کمک می‌کند»؟
#   arm: h' = h + Up(GELU(Down(h)))  روی embedding کش‌شدهٔ giga (backbone دست نمی‌خورد)،
#        bottleneck طوری که تعداد پارامتر = LoRA (r=8 روی ۸ ماژول q/v → 163,840 پارامتر)؛ Up با صفر شروع می‌شود (مثل B=0 در LoRA).
#   همان decoder (unet+phys)، همان init/ترتیب داده/تعداد توالی/lr/آستانهٔ val که در frozen و LoRA سلول P6.
#   اگر LoRA − adapter ≈ 0 → اثر LoRA را نمی‌توان به «ترمیم بازنمایی» نسبت داد.
# runtime: GPU T4 | هر seed حدود ۲-۴ min | پیش‌نیاز: P6 (counts_frozen/lora برای همان seedها)
# env: RNA_LORA_SEEDS ، RNA_LORA_STEPS (باید با P6 یکی باشد)
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p6b_adapter", restore="data_paper/*.parquet; emb_paper/*.npy; emb_paper/*_meta.npz; phys_paper/phys_*.npz; artifacts_paper/lora_ckpt/*; config_paper.json",
                 save="artifacts_paper/adapter_ckpt/*; artifacts_paper/results_adapter.json"):

        import json, os, sys, time
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import torch
        import torch.nn as nn
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p6b_adapter"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        SEEDS = [int(x) for x in os.environ.get("RNA_LORA_SEEDS", "0" if P.QUICK else "0,1,2").split(",")]
        STEPS = int(os.environ.get("RNA_LORA_STEPS", 40 if P.QUICK else 1500))
        ACC, ADAPTER_LR, LORA_PARAMS = 2, 2e-4, 163_840
        OUT = AR / "artifacts_paper"
        LK = OUT / "lora_ckpt"; CK = OUT / "adapter_ckpt"; CK.mkdir(parents=True, exist_ok=True)
        print(f"### {STAGE} START | seeds={SEEDS} steps={STEPS}×{ACC} device={P.DEV}", flush=True)

        splits = {n: P.Split(n) for n in P.SPLITS}
        dim = splits["train"].dim
        spec = P.ARMS["unet+phys"]
        RANK = max(1, min(dim, round(LORA_PARAMS / (2 * dim))))
        PROTO = {"steps": STEPS, "acc": ACC, "adapter_rank": RANK, "adapter_lr": ADAPTER_LR, "pw": P.PW, "n_train": len(splits["train"]),
                 "thr_grid": P.THR_GRID, "backbone": json.load(open(AR / "emb_paper" / "embed_done.json"))["model"]}


        class Adapter(nn.Module):
            def __init__(self, d, r):
                super().__init__()
                self.down = nn.Linear(d, r, bias=False); self.up = nn.Linear(r, d, bias=False)
                nn.init.zeros_(self.up.weight)

            def forward(self, h):
                return h + self.up(nn.functional.gelu(self.down(h)))


        t0 = time.time()
        res_all = {"protocol": PROTO, "seeds": {}}
        for seed in SEEDS:
            ck = CK / f"seed{seed}.json"
            if ck.exists() and json.load(open(ck)).get("_proto") == PROTO:
                print(f"[seed {seed}] checkpoint — skip", flush=True)
                res_all["seeds"][str(seed)] = json.load(open(ck)); continue
            lj = LK / f"seed{seed}.json"
            gate(lj.exists() and (LK / f"counts_frozen_matched_s{seed}.npz").exists(), f"seed {seed}: خروجی P6 (frozen/LoRA) موجود است")
            lrec = json.load(open(lj))
            gate(lrec["_proto"]["steps"] == STEPS and lrec["_proto"]["acc"] == ACC, f"seed {seed}: STEPS/ACC با P6 یکی است (matched)")
            torch.manual_seed(seed); np.random.seed(seed)
            dec = P.build_arm(spec, dim, splits["train"])          # همان init decoder مثل frozen/LoRA
            adp = Adapter(dim, RANK).to(P.DEV)
            n_adp = sum(p.numel() for p in adp.parameters())
            print(f"  adapter: rank={RANK} پارامتر={n_adp/1e3:.1f}K (LoRA: {LORA_PARAMS/1e3:.1f}K)", flush=True)
            gate(abs(n_adp - LORA_PARAMS) / LORA_PARAMS < 0.02 or dim != 1280, f"تعداد پارامتر adapter ≈ LoRA ({n_adp} در برابر {LORA_PARAMS})")

            def hfn(sp, i, _a=adp):
                return _a(sp.h(i))

            diag = P.train_decoder(dec, spec, splits["train"], seed, n_seq=STEPS * ACC, accum=ACC, tag=f"adapter s{seed}", log_every=500,
                                   hfn=hfn, extra_groups=[{"params": list(adp.parameters()), "lr": ADAPTER_LR}])
            adp.eval()
            unorm = float(adp.up.weight.detach().float().norm())
            gate(diag["min_grad_norm_lora"] is not None and diag["final_grad_norm_lora"] > 0 and unorm > 0,
                 f"adapter واقعاً آموزش دید | |g| نهایی={diag['final_grad_norm_lora']:.2e} ‖Up‖={unorm:.3f}")
            r, cnt = P.evaluate_arm(dec, spec, splits, hfn=hfn, with_categories=False)
            np.savez(CK / f"counts_adapter_matched_s{seed}.npz", **cnt)
            rec = {"_proto": PROTO, "seed": seed, "adapter": r, "adapter_diag": {**diag, "Up_norm": round(unorm, 4), "n_params": n_adp},
                   "ref": {"frozen_ood": lrec["frozen"]["ood_mean"], "lora_ood": lrec["lora"]["ood_mean"]}}
            rec["delta"] = {"adapter_minus_frozen_ood": round(r["ood_mean"] - lrec["frozen"]["ood_mean"], 4),
                            "lora_minus_adapter_ood": round(lrec["lora"]["ood_mean"] - r["ood_mean"], 4)}
            json.dump(rec, open(ck, "w"), indent=2)
            res_all["seeds"][str(seed)] = rec
            print(f"[seed {seed}] adapter OOD={r['ood_mean']} | frozen {lrec['frozen']['ood_mean']} | LoRA {lrec['lora']['ood_mean']} | "
                  f"adapter−frozen={rec['delta']['adapter_minus_frozen_ood']:+.4f} LoRA−adapter={rec['delta']['lora_minus_adapter_ood']:+.4f} | {time.time()-t0:.0f}s", flush=True)
            del dec, adp
            torch.cuda.empty_cache()

        res_all["delta_per_seed"] = {s: v["delta"] for s, v in res_all["seeds"].items()}
        json.dump(res_all, open(OUT / "results_adapter.json", "w"), indent=2)
        gate(True, f"کنترل ظرفیت: {len(res_all['seeds'])} seed | تفسیر فقط با bootstrap P7 (LoRA − adapter و adapter − frozen)", hard=False)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P7 — آمار (v3)
bootstrap جفت‌شده با واحد درست + حساسیت (family، بدون duplicate) + CI تصحیح‌شده + شباهت با کنترل RNAfold.

In [ ]:
# fmt: off
# ============================================================
# P7 (v3) — آمار: bootstrap «جفت‌شده» با واحد درست + تحلیل حساسیت + شباهت با کنترل RNAfold
#   تغییرات نسبت به v2:
#   • واحد بازنمونه = واحد تعریف‌کنندهٔ split از P1b (GenA/GenC/GenF: معمولاً architecture/clan/family)؛ Test: family
#   • حساسیت ۱: همان آمار با واحد family (v2) ؛ اگر وضعیت ادعا عوض شود، P8 آن را «حساس به واحد» علامت می‌زند
#   • حساسیت ۲: حذف توالی‌های ارزیابی که دقیقاً در train هستند (فقط اگر وجود داشته باشند)
#   • CI تصحیح‌شده برای ۳ split (Bonferroni: 98.33%) کنار CI95؛ ادعای هر split OOD با CI تصحیح‌شده سنجیده می‌شود
#   • مقایسه‌های جدید: adapter هم‌پارامتر (P6b) در برابر frozen و در برابر LoRA
#   • محور شباهت: همان چارک‌ها برای RNAfold-MFE هم + CI بوت‌استرپ برای «تفاضل ρ مدل − ρ RNAfold» (+ شباهت به کل train)
# runtime: CPU | ~4-8 min | artifact: artifacts_paper/analysis_paper_v3.json
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p7_analysis", restore="config_paper.json; artifacts_paper/leakage.json; artifacts_paper/leak_masks.npz; artifacts_paper/train_full_kmer.npy; artifacts_paper/main_ckpt/*; artifacts_paper/counts_rnafold.npz; artifacts_paper/lora_ckpt/*; artifacts_paper/adapter_ckpt/*; data_paper/*.parquet; data_paper/levels_*.parquet; phys_paper/phys_*.npz",
                 save="artifacts_paper/analysis_paper_v3.json"):

        import json, os, sys
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import pandas as pd
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p7_analysis"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        OUT = AR / "artifacts_paper"
        CK, LK, AK = OUT / "main_ckpt", OUT / "lora_ckpt", OUT / "adapter_ckpt"
        NB = 300 if P.QUICK else 2000
        print(f"### {STAGE} START | n_boot={NB}", flush=True)
        gate((OUT / "leakage.json").exists() and (OUT / "leak_masks.npz").exists(), "خروجی P1b موجود است (leakage.json, leak_masks.npz)")
        leak = json.load(open(OUT / "leakage.json"))
        masks = np.load(OUT / "leak_masks.npz")
        meta = {n: P.Split(n, emb=False, phys=False) for n in ["train"] + P.EVAL}


        def level_col(n, level):
            return pd.read_parquet(AR / "data_paper" / f"levels_{n}.parquet")[level].astype(str).tolist()


        unit_level = {n: leak["splits"][n]["unit"] for n in P.EVAL}
        units_primary = {n: level_col(n, unit_level[n]) for n in P.EVAL}
        units_family = {n: list(meta[n].fam) for n in P.EVAL}
        keep = {n: masks[f"keep_{n}"].astype(bool) for n in P.EVAL}
        for n in P.EVAL:
            gate(len(units_primary[n]) == len(meta[n].L) == len(keep[n]), f"{n}: طول برچسب واحد/ماسک با split یکی است")
        any_dup = any((~keep[n]).any() for n in P.EVAL)
        unit_info = {n: {"unit": unit_level[n], "n_units": int(len(set(units_primary[n]))), "n_families": int(len(set(units_family[n])))} for n in P.EVAL}
        print("واحد بازنمونه:", unit_info, flush=True)


        def sysdict(files, prefix=""):
            seeds = []
            for f in files:
                z = np.load(f)
                seeds.append({n: (z[f"{prefix}{n}_tp"], z[f"{prefix}{n}_fp"], z[f"{prefix}{n}_fn"]) for n in P.EVAL})
            return {n: [s[n] for s in seeds] for n in P.EVAL}


        def apply_keep(system, km):
            return {n: [tuple(a[km[n]] for a in t) for t in system[n]] for n in system}


        systems = {}
        for arm in P.ARMS:
            fs = sorted(CK.glob(f"{arm}_s*.npz"))
            if fs:
                systems[arm] = sysdict(fs)
        ref = OUT / "counts_rnafold.npz"
        if ref.exists():
            systems["rnafold_mfe"] = sysdict([ref], "mfe_")
            systems["rnafold_bpp"] = sysdict([ref], "bpp_")
        fz, lo, ad = sorted(LK.glob("counts_frozen_matched_s*.npz")), sorted(LK.glob("counts_lora_matched_s*.npz")), sorted(AK.glob("counts_adapter_matched_s*.npz"))
        if fz and lo:
            systems["frozen_matched"] = sysdict(fz); systems["lora_matched"] = sysdict(lo)
        if ad:
            systems["adapter_matched"] = sysdict(ad)
        print("سامانه‌ها (تعداد seed):", {k: len(v["test"]) for k, v in systems.items()}, flush=True)

        COMPS = [("unet+phys", "rnafold_mfe", "مدل اصلی − RNAfold-MFE"),
                 ("unet+phys", "rnafold_bpp", "مدل اصلی − RNAfold-bpp(thr val)"),
                 ("nolm+phys", "rnafold_mfe", "بدون LM − RNAfold-MFE"),
                 ("unet+phys", "nolm+phys", "سهم LM (با − بدون)"),
                 ("unet+phys", "unet", "سهم فیزیک (با − بدون)"),
                 ("lora_matched", "frozen_matched", "LoRA − frozen (matched)"),
                 ("adapter_matched", "frozen_matched", "adapter هم‌پارامتر − frozen"),
                 ("lora_matched", "adapter_matched", "LoRA − adapter هم‌پارامتر (کنترل ظرفیت)")]


        def run_boot(units, km=None):
            out = {}
            for a, b, title in COMPS:
                if a in systems and b in systems:
                    A, Bs, u = systems[a], systems[b], units
                    if km is not None:
                        A, Bs = apply_keep(A, km), apply_keep(Bs, km)
                        u = {n: [x for x, k in zip(units[n], km[n]) if k] for n in units}
                    r = P.paired_bootstrap(A, Bs, u, n_boot=NB)
                    r["title"] = title; r["n_seeds_a"] = len(A["test"]); r["n_seeds_b"] = len(Bs["test"])
                    out[f"{a}__vs__{b}"] = r
            return out


        MIN_UNITS = 5   # از پیش اعلام‌شده: با کمتر از ۵ واحد، bootstrap واحد-محور CI معتبر نمی‌دهد (چند ترکیب بیشتر ندارد)


        def cell(r, s):
            ci = r[s]["ci_bonf3"] if s in P.OOD else r[s]["ci95"]
            if unit_info[s]["n_units"] < MIN_UNITS:
                return f"{r[s]['delta']:+.3f}‡"
            return f"{r[s]['delta']:+.3f}{'*' if ci[0] > 0 else ('†' if ci[1] < 0 else '')}"


        def show(tag, boot):
            print(f"\n--- {tag}  (* = CI>0 ، † = CI<0 ، ‡ = کمتر از ۵ واحد بازنمونه ⇒ CI معتبر نیست؛ OOD با CI تصحیح‌شدهٔ Bonferroni×3، Test با CI95) ---", flush=True)
            for k, r in boot.items():
                o = r["ood_mean"]
                print(f"{r['title']}: " + " ".join(f"{s}:{cell(r, s)}" for s in P.EVAL) + f" | OOD-mean(توصیفی) {o['delta']:+.3f} [{o['ci95'][0]:+.3f},{o['ci95'][1]:+.3f}]", flush=True)


        boot_unit = run_boot(units_primary); show("واحد اصلی (واحد تعریف‌کنندهٔ split)", boot_unit)
        boot_family = run_boot(units_family); show("حساسیت: واحد family (v2)", boot_family)
        boot_nodup = None
        if any_dup:
            boot_nodup = run_boot(units_primary, keep); show("حساسیت: بدون توالی‌های یکسان با train", boot_nodup)
        else:
            print("\n(توالی یکسانی با train نیست → تحلیل حساسیت حذف duplicate لازم نیست)", flush=True)

        # ---------------- محور شباهت (4-mer کسینوسی) با کنترل طول/GC؛ مدل اصلی در برابر RNAfold-MFE
        def per_seq_f1(tp, fp, fn):
            tp, fp, fn = [np.asarray(x, float) for x in (tp, fp, fn)]
            return 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9)


        def smax_of(Q, T, chunk=250):
            return np.concatenate([(Q[i:i + chunk] @ T.T).max(1) for i in range(0, len(Q), chunk)])


        sim = {}
        if "unet+phys" in systems and "rnafold_mfe" in systems:
            Tsub = np.stack([P.kvec(s) for s in meta["train"].seqs])
            kf = OUT / "train_full_kmer.npy"
            Tfull = np.load(kf).astype(np.float32) if kf.exists() else None
            rng = np.random.RandomState(1)
            NBR = 200 if P.QUICK else 500
            for n in P.EVAL:
                sp = meta[n]
                tp = np.mean([t[0] for t in systems["unet+phys"][n]], 0); fp = np.mean([t[1] for t in systems["unet+phys"][n]], 0)
                fn = np.mean([t[2] for t in systems["unet+phys"][n]], 0)
                f1m = per_seq_f1(tp, fp, fn)
                f1r = per_seq_f1(*systems["rnafold_mfe"][n][0])
                has = (tp + fn) > 0
                Q = np.stack([P.kvec(q) for q in sp.seqs])
                gc = np.array([np.mean([c in "GC" for c in q]) for q in sp.seqs]); Ls = sp.L.astype(float)
                groups = {}
                for ix in np.flatnonzero(has):
                    groups.setdefault(units_primary[n][ix], []).append(ix)
                groups = [np.array(v) for v in groups.values()]

                def block(sm):
                    q = np.quantile(sm[has], [0.25, 0.5, 0.75]); b = np.digitize(sm, q)
                    qf = lambda f: [round(float(f[has & (b == k)].mean()), 3) if (has & (b == k)).any() else None for k in range(4)]
                    qm, qr = qf(f1m), qf(f1r)
                    ds = []
                    for _ in range(NBR):
                        ii = np.concatenate([groups[k] for k in rng.randint(0, len(groups), len(groups))])
                        ds.append(P.partial_spearman(sm[ii], f1m[ii], [Ls[ii], gc[ii]]) - P.partial_spearman(sm[ii], f1r[ii], [Ls[ii], gc[ii]]))
                    ds = np.array(ds); ds = ds[np.isfinite(ds)]
                    rm, rr = P.partial_spearman(sm[has], f1m[has], [Ls[has], gc[has]]), P.partial_spearman(sm[has], f1r[has], [Ls[has], gc[has]])
                    return {"rho_model_raw": round(float(pd.Series(sm[has]).corr(pd.Series(f1m[has]), method="spearman")), 3),
                            "rho_model_ctrl": round(rm, 3),
                            "rho_rnafold_raw": round(float(pd.Series(sm[has]).corr(pd.Series(f1r[has]), method="spearman")), 3),
                            "rho_rnafold_ctrl": round(rr, 3),
                            "rho_diff_ctrl": round(rm - rr, 3),
                            "rho_diff_ctrl_ci95": [round(float(np.percentile(ds, 2.5)), 3), round(float(np.percentile(ds, 97.5)), 3)] if len(ds) > 20 else None,
                            "q_model": qm, "q_rnafold": qr,
                            "q_delta": [None if (a is None or b_ is None) else round(a - b_, 3) for a, b_ in zip(qm, qr)]}

                sim[n] = {"n": int(has.sum()), "unit": unit_level[n], "n_units": len(groups), "len_mean": round(float(Ls.mean()), 1),
                          "train_subset": block(smax_of(Q, Tsub))}
                if Tfull is not None:
                    sim[n]["train_full"] = block(smax_of(Q, Tfull))
                b = sim[n]["train_subset"]
                print(f"similarity {n} (n={sim[n]['n']}, واحد={unit_level[n]}×{len(groups)}): ρ_ctrl مدل={b['rho_model_ctrl']} RNAfold={b['rho_rnafold_ctrl']} "
                      f"Δρ={b['rho_diff_ctrl']} CI={b['rho_diff_ctrl_ci95']} | F1 چارک مدل={b['q_model']} RNAfold={b['q_rnafold']}", flush=True)

        json.dump({"bootstrap_unit": boot_unit, "bootstrap_family": boot_family, "bootstrap_unit_nodup": boot_nodup, "similarity": sim,
                   "units": unit_info, "n_boot": NB, "min_units": MIN_UNITS,
                   "note": "ستاره: CI پایین‌تر از صفر نیست (OOD: Bonferroni×3). OOD-mean فقط توصیفی (میانگین هم‌وزن سه split)."},
                  open(OUT / "analysis_paper_v3.json", "w"), indent=2)
        gate(bool(boot_unit), f"{len(boot_unit)} مقایسهٔ جفت‌شده (واحد اصلی) + حساسیت family" + (" + بدون duplicate" if boot_nodup else "") if boot_unit
             else "هیچ مقایسه‌ای ممکن نبود (P4/P5 را اجرا کنید)", hard=False)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P8 — گزارش و gateهای حکم (v3)
جدول ادعا ↔ شاهد به‌تفکیک split؛ خروجی `report_paper_v3.md`. حکم نهایی دستی است.

In [ ]:
# fmt: off
# ============================================================
# P8 (v3) — گزارش ادعا ↔ شاهد «به‌تفکیک split» + gateهای حکم (حکم نهایی دستی است)
#   تغییرات نسبت به v2:
#   • هر ادعا (C1–C5) برای هر split جدا سنجیده می‌شود؛ OOD-mean فقط توصیفی است
#   • وضعیت‌ها: ✅ مثبت معنادار | 🔻 منفی معنادار | ⬜ تأییدنشده ؛ برای هر split OOD با CI تصحیح‌شدهٔ Bonferroni×3
#   • وابستگی به متریک: F1@thr(val) در برابر F1@oracle و AUPRC برای C2/C3 (قاعدهٔ از پیش‌اعلام‌شده در متن)
#   • حساسیت به واحد بازنمونه (family در برابر واحد تعریف‌کنندهٔ split) و به توالی‌های تکراری
#   • sanity P4 فقط «گزارش اختلاف» است، نه PASS
#   • حکم: gateها خودکار، ولی جملهٔ نهایی «پایهٔ کامل» را فقط نویسنده می‌نویسد (بخش ۶)
# runtime: any | ~1 min | artifact: artifacts_paper/report_paper_v3.md
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p8_report", restore="config_paper.json; env.json; emb_paper/embed_done.json; artifacts_paper/*.json; artifacts_paper/adapter_ckpt/*",
                 save="artifacts_paper/report_paper_v3.md"):

        import json, sys
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p8_report"
        gate = P.gate_for(STAGE)
        A = AR / "artifacts_paper"


        def load(p):
            try:
                return json.load(open(p))
            except Exception:
                return None


        cfg = P.cfg()
        emb, env = load(AR / "emb_paper" / "embed_done.json"), load(AR / "env.json")
        ref, main, lora, ana, leak = (load(A / "ref_rnafold.json"), load(A / "results_main.json"), load(A / "results_lora.json"),
                                      load(A / "analysis_paper_v3.json"), load(A / "leakage.json"))
        n_adapter = len(list((A / "adapter_ckpt").glob("seed*.json")))
        n_main = len(main["seeds"]) if main else 0
        n_lora = len(lora["seeds"]) if lora else 0
        L = ["# CHANRG Paper Track v3 — گزارش", ""]

        # ---------------------------------------------------------------- پروتکل
        L.append("## پروتکل")
        if emb:
            L.append(f"- backbone `{emb['model']}` (dim {emb['dim']}) | طول {cfg['len_min']}..{cfg['len_max']} با فیلتر (بدون truncation) | micro-F1 روی جفت‌ها")
        L.append("- زیرمجموعه‌ها: " + ", ".join(f"{k} {v['subset']} ({v['families']} خانواده)" for k, v in cfg["splits_info"].items()))
        if env:
            L.append(f"- سخت‌افزار: {env['gpu']} | VRAM {env['vram_gb']}GB" + (" | **QUICK**" if env.get("quick") else ""))
        L.append(f"- seedها: مدل اصلی {n_main} | LoRA/frozen {n_lora} | adapter هم‌پارامتر {n_adapter}")
        L.append("- این گزارش فقط بر پایهٔ اجرای v3/v2 است؛ نتایج pilot (mega، ۲۵۰۰ توالی تصادفی، truncate) با آن ادغام **نمی‌شوند**.")
        L.append("")

        # ---------------------------------------------------------------- واحد بازنمونه و نشت
        L.append("## ۱) واحد بازنمونه و نشت (P1b)")
        if leak and ana:
            L.append("| split | واحد بازنمونه | #واحد (cap) | #family | مشترک با train (family، cap) | توالی یکسان با train (cap) |")
            L.append("|---|---|---:|---:|---:|---:|")
            for n in P.EVAL:
                s = leak["splits"][n]; u = s["unit"]; fam = leak["family_col"]
                L.append(f"| {n} | {u} | {s['levels'][u]['n_units_cap']} | {s['levels'][fam]['n_units_cap']} | {s['levels'][fam]['shared_cap']} | {s['dups']['dup_vs_train_cap']} |")
            L.append(f"- ترتیب سطوح (درشت→ریز): {leak['order_coarse_to_fine']} | تعداد برچسب یکتا: {leak['n_unique_levels']}")
            L.append("- واحد (v4) = سطحِ کاملاً مجزا از train با کمترین تعداد واحد (≥۵)؛ CIهای این گزارش روی همین واحد بازنمونه می‌شوند.")
        else:
            L.append("_P1b/P7 اجرا نشده._")
        L.append("")

        # ---------------------------------------------------------------- sanity
        L.append("## ۲) RNAfold ما در برابر عدد منتشرشده (گزارش اختلاف، نه تأیید)")
        if ref:
            L.append("| | Test | GenA | GenC | GenF | OOD mean |")
            L.append("|---|---:|---:|---:|---:|---:|")
            for k, nm in [("mfe", "RNAfold-MFE"), ("bpp", f"RNAfold-bpp (thr val={ref['bpp_thr']})")]:
                r = ref[k]
                L.append(f"| {nm} | {r['test']['f1']:.3f} | {r['gena']['f1']:.3f} | {r['genc']['f1']:.3f} | {r['genf']['f1']:.3f} | {r['ood_mean']:.3f} |")
            dd = ref["mfe"]["ood_mean"] - ref["published_rnafold_ood"]
            L.append(f"- منتشرشده (OOD mean): {ref['published_rnafold_ood']} | ما: {ref['mfe']['ood_mean']:.3f} | **اختلاف {dd:+.3f}** — {ref['note']}")
            L.append("- اختلاف هم‌اندازهٔ اثرهای سنجیده‌شده است؛ اعداد را با جدول رسمی CHANRG مقایسه نکنید (طول/متریک/زیرمجموعه فرق دارد).")
        else:
            L.append("_P4 اجرا نشده._")
        L.append("")

        # ---------------------------------------------------------------- مدل‌ها
        L.append("## ۳) مدل‌های اصلی (LM frozen)")
        if main:
            L.append("| arm | #params | Test | GenA | GenC | GenF | OOD mean±std(seed) | AUPRC OOD-mean |")
            L.append("|---|---:|---:|---:|---:|---:|---|---:|")
            for a, v in main["arms"].items():
                s = v["splits"]
                au = sum(s[n]["auprc"]["mean"] for n in P.OOD) / 3
                L.append(f"| {a} | {v['n_params']/1e3:.0f}K | {s['test']['f1']['mean']:.3f} | {s['gena']['f1']['mean']:.3f} | {s['genc']['f1']['mean']:.3f} | "
                         f"{s['genf']['f1']['mean']:.3f} | {v['ood_mean']['mean']:.3f}±{v['ood_mean']['std']:.3f} | {au:.3f} |")
            L.append("- ±std بین seedها فقط نویز آموزش است؛ استنباط از CI خانواده/واحدی (بخش ۴) انجام می‌شود. OOD mean توصیفی است (هر split وزن برابر).")
            ex = main["arms"].get("unet+phys")
            if ex:
                L.append("- آستانه/قید (unet+phys): " + " | ".join(
                    f"{n}: F1 {ex['splits'][n]['f1']['mean']:.3f} → oracle {ex['splits'][n]['f1_oracle']['mean']:.3f}, one-pair {ex['splits'][n]['f1_onepair']['mean']:.3f}" for n in P.EVAL))
        else:
            L.append("_P5 اجرا نشده._")
        L.append("")

        # ---------------------------------------------------------------- ادعاها per-split
        L.append("## ۴) ادعا ↔ شاهد، به‌تفکیک split")
        L.append("قاعدهٔ از پیش‌اعلام‌شده: ✅ اگر حد پایین CI > 0 ؛ 🔻 اگر حد بالا < 0 ؛ وگرنه ⬜ (تأییدنشده؛ «اثبات نبود اثر» نیست). "
                 "اگر split کمتر از ۵ واحد بازنمونه داشته باشد، CI واحد-محور معتبر نیست و وضعیت ⬜ (واحد کم) می‌ماند؛ ستون «family» فقط حساسیت است و جایگزین ادعا نمی‌شود. "
                 "CI برای Test = ۹۵٪؛ برای هر split OOD = ۹۸٫۳۳٪ (Bonferroni×3). بازنمونه روی واحد جدول ۱؛ ستون «family» همان آمار با واحد v2.")
        L.append("")
        B = (ana or {}).get("bootstrap_unit") or {}
        BF = (ana or {}).get("bootstrap_family") or {}
        BN = (ana or {}).get("bootstrap_unit_nodup")
        MIN_UNITS = (ana or {}).get("min_units", 5)
        claims, flips = {}, []


        def ci_of(r, s):
            return r[s]["ci_bonf3"] if s in P.OOD else r[s]["ci95"]


        def st(r, s):
            ci = ci_of(r, s)
            return "✅" if ci[0] > 0 else ("🔻" if ci[1] < 0 else "⬜")


        def seed_metric(arm, seed, split, key):
            d = load(A / "main_ckpt" / f"{arm}_s{seed}.json")
            return d["splits"][split].get(key) if d else None


        def metric_deltas(a, b, split):
            out = {}
            for key in ["f1", "f1_oracle", "auprc"]:
                ds = []
                for sd in (main["seeds"] if main else []):
                    x, y = seed_metric(a, sd, split, key), seed_metric(b, sd, split, key)
                    if x is not None and y is not None:
                        ds.append(x - y)
                out[key] = (float(np.mean(ds)), int(sum(d > 0 for d in ds)), len(ds)) if ds else None
            return out


        def claim_block(cid, key, splits, title):
            r = B.get(key)
            if not r:
                L.append(f"**{cid} — {title}**: _داده نیست_"); L.append(""); return
            L.append(f"**{cid} — {title}** ({r['title']}؛ seed {r['n_seeds_a']}/{r['n_seeds_b']})")
            L.append("| split | واحد (n) | Δ | CI مورد استفاده | وضعیت | با واحد family | بدون توالی تکراری |")
            L.append("|---|---|---:|---|---|---|---|")
            for s in splits:
                ci = ci_of(r, s); u = ana["units"][s]
                sf = st(BF[key], s) if key in BF else "—"
                sn = st(BN[key], s) if (BN and key in BN) else "—"
                few = u["n_units"] < MIN_UNITS
                mark = "⬜ (واحد کم)" if few else st(r, s)
                claims[(cid, s)] = mark
                if (not few) and ((sf not in ("—", mark)) or (sn not in ("—", mark))):
                    flips.append(f"{cid}/{s}")
                cis = "نامعتبر (<۵ واحد)" if few else f"[{ci[0]:+.3f}, {ci[1]:+.3f}]"
                L.append(f"| {s} | {u['unit']} ({u['n_units']}) | {r[s]['delta']:+.3f} | {cis} | {mark} | {sf} | {sn} |")
            o = r["ood_mean"]
            L.append(f"- توصیفی: OOD-mean Δ={o['delta']:+.3f} [{o['ci95'][0]:+.3f}, {o['ci95'][1]:+.3f}] (CI95؛ برای ادعا استفاده نمی‌شود)")
            L.append("")


        def metric_block(a, b, splits):
            L.append("| split | ΔF1 @thr(val) | ΔF1 @oracle | ΔAUPRC | seed مثبت AUPRC | نتیجه |")
            L.append("|---|---:|---:|---:|---|---|")
            for s in splits:
                m = metric_deltas(a, b, s)
                if not m["f1"]:
                    continue
                au = m["auprc"]
                if au is None:
                    tag = "AUPRC ندارد"
                else:
                    rank_ok = au[0] > 0.01 and au[1] == au[2]
                    tag = "رتبه‌بندی بهتر" if rank_ok else ("منفی در AUPRC" if au[0] < -0.01 else "AUPRC تقریباً بی‌تغییر ⇒ اثر وابسته به آستانه/متریک")
                L.append(f"| {s} | {m['f1'][0]:+.3f} | {m['f1_oracle'][0]:+.3f} | " + (f"{au[0]:+.3f}" if au else "—") + f" | " + (f"{au[1]}/{au[2]}" if au else "—") + f" | {tag} |")
            L.append("- قاعدهٔ از پیش‌اعلام‌شده: «رتبه‌بندی بهتر» فقط اگر میانگین ΔAUPRC > ۰٫۰۱ و همهٔ seedها مثبت باشند؛ در غیر این‌صورت F1 معنادار را «وابسته به آستانه/متریک» بخوانید. میانگین seedها، بدون CI.")
            L.append("")


        claim_block("C1", "unet+phys__vs__nolm+phys", ["test"], "LM در داده‌ی درون‌توزیع (Test) کمک می‌کند")
        claim_block("C2", "unet+phys__vs__nolm+phys", P.OOD, "LM در OOD کمک می‌کند (هر split جدا)")
        claim_block("C3", "unet+phys__vs__unet", ["test"] + P.OOD, "ورودی فیزیکی کمک می‌کند (هر split جدا)")
        if main:
            L.append("وابستگی C3 (و C2) به متریک:")
            L.append("")
            L.append("*C3: unet+phys − unet*"); L.append("")
            metric_block("unet+phys", "unet", ["test"] + P.OOD)
            L.append("*C2: unet+phys − nolm+phys*"); L.append("")
            metric_block("unet+phys", "nolm+phys", ["test"] + P.OOD)
        claim_block("C4a", "unet+phys__vs__rnafold_mfe", ["test"] + P.OOD, "مدل اصلی از RNAfold-MFE بهتر است")
        claim_block("C4b", "unet+phys__vs__rnafold_bpp", ["test"] + P.OOD, "مدل اصلی از RNAfold-bpp(thr val) بهتر است")
        claim_block("C5a", "lora_matched__vs__frozen_matched", ["test"] + P.OOD, "LoRA (matched) از frozen بهتر است")
        claim_block("C5b", "adapter_matched__vs__frozen_matched", ["test"] + P.OOD, "کنترل: adapter هم‌پارامتر از frozen بهتر است")
        claim_block("C5c", "lora_matched__vs__adapter_matched", ["test"] + P.OOD, "LoRA از adapter هم‌پارامتر بهتر است (اثر «ترمیم بازنمایی» جدا از ظرفیت)")
        L.append("- تعبیر C5: اگر C5a ✅ ولی C5c ⬜ باشد، فقط «ظرفیت آموزش‌پذیر بیشتر کمک می‌کند» قابل‌ادعاست، نه «LoRA بازنمایی را ترمیم می‌کند».")
        L.append("")

        # ---------------------------------------------------------------- LoRA / adapter خلاصه
        if lora:
            L.append("### LoRA / frozen / adapter — خلاصهٔ هر seed (OOD-mean توصیفی)")
            for s, v in lora["seeds"].items():
                a = load(A / "adapter_ckpt" / f"seed{s}.json")
                L.append(f"- seed {s}: frozen {v['frozen']['ood_mean']} | LoRA {v['lora']['ood_mean']} (‖B‖={v['lora_diag']['B_norm']}) | " +
                         (f"adapter {a['adapter']['ood_mean']} (‖Up‖={a['adapter_diag']['Up_norm']})" if a else "adapter: اجرا نشده"))
            L.append("")

        # ---------------------------------------------------------------- شباهت
        L.append("### محور شباهت (4-mer به train cap‌شده؛ کنترل طول+GC) — مدل اصلی در برابر RNAfold-MFE")
        sim = (ana or {}).get("similarity") or {}
        if sim:
            L.append("| split | n | ρ مدل | ρ RNAfold | Δρ [CI95] | F1 چارک مدل (کم→زیاد شباهت) | F1 چارک RNAfold |")
            L.append("|---|---:|---:|---:|---|---|---|")
            for n, v in sim.items():
                b = v["train_subset"]
                L.append(f"| {n} | {v['n']} | {b['rho_model_ctrl']} | {b['rho_rnafold_ctrl']} | {b['rho_diff_ctrl']} {b['rho_diff_ctrl_ci95']} | {b['q_model']} | {b['q_rnafold']} |")
            full_ok = all("train_full" in v for v in sim.values())
            if full_ok:
                L.append("- حساسیت: شباهت به **کل** train فیلترشده — Δρ [CI]: " + " | ".join(
                    f"{n}: {v['train_full']['rho_diff_ctrl']} {v['train_full']['rho_diff_ctrl_ci95']}" for n, v in sim.items()))
            L.append("- ρ منفی در مدل ولی نه در RNAfold (Δρ منفی و CI شامل صفر نباشد) = افت وابسته به شباهت **مخصوص مدل** (نشانهٔ shortcut احتمالی)؛ "
                     "اگر RNAfold هم همین الگو را دارد، اثر مربوط به دشواری ذاتی نمونه‌هاست و نه یادگیری مدل. علامت‌ها را به‌تفکیک split بخوانید.")
        else:
            L.append("_P7 اجرا نشده._")
        L.append("")

        # ---------------------------------------------------------------- gateهای حکم
        L.append("## ۵) gateهای حکم (خودکار)")
        g = leak["gates"] if leak else {}
        gates = [
            ("تعمیم OOD از نظر family از train مجزاست (cap)", bool(g.get("ood_family_disjoint_cap"))),
            ("هیچ توالی ارزیابی دقیقاً در train نیست (یا تحلیل حساسیت دارد)", g.get("dup_total_cap", 1) == 0 or bool(ana and ana.get("bootstrap_unit_nodup"))),
            ("هر split OOD حداقل ۱۰ واحد بازنمونه دارد", bool(leak) and not g.get("small_unit_splits")),
            (f"seed مدل اصلی ≥ ۳ (الان {n_main})", n_main >= 3),
            (f"seed LoRA/frozen ≥ ۳ (الان {n_lora})", n_lora >= 3),
            (f"کنترل adapter هم‌پارامتر با ≥ ۳ seed (الان {n_adapter})", n_adapter >= 3),
            ("وضعیت هیچ ادعایی با عوض‌کردن واحد/حذف duplicate تغییر نمی‌کند", bool(claims) and not flips),
            ("اجرای کامل (نه QUICK)", bool(env) and not env.get("quick")),
        ]
        for t, ok in gates:
            L.append(f"- {'✅' if ok else '⬜'} {t}")
        if flips:
            L.append(f"- ادعاهای حساس به واحد/duplicate: {', '.join(flips)}")
        auto_ok = all(ok for _, ok in gates)
        L.append("")
        L.append(f"### حکم خودکار: **{'gateها گذشتند — فقط ادعاهای ✅ بخش ۴ نوشتنی‌اند؛ تأیید دستی لازم است' if auto_ok else 'ناقص — gateهای ⬜ بالا را ببینید'}**")
        L.append("- این حکم «پایهٔ تجربی کامل» نیست؛ جمله‌های مقاله را فقط از وضعیت per-split بخش ۴ بنویسید.")
        L.append("")

        # ---------------------------------------------------------------- خلاصهٔ نوشتنی
        L.append("## ۶) خلاصهٔ per-split برای نوشتن (نویسنده بازبینی کند)")
        for cid in sorted({c for c, _ in claims}):
            L.append(f"- {cid}: " + " | ".join(f"{s} {m}" for (c, s), m in claims.items() if c == cid))
        L.append("- هر ⬜ را «تأییدنشده» بنویسید؛ هر 🔻 را به‌عنوان اثر منفی گزارش کنید، نه نادیده بگیرید.")
        L.append("")
        L.append("## ۷) محدودیت‌ها (کنار هر عدد بماند)")
        L.append(f"- زیرمجموعهٔ cap‌شده (train {cfg['train_n']}) و طول {cfg['len_min']}..{cfg['len_max']}؛ اعداد با جدول رسمی CHANRG مستقیماً قابل‌مقایسه نیستند.")
        L.append("- LM در armهای P5 frozen است؛ LoRA فقط در P6 و بودجهٔ کوچک (۱۵۰۰×۲ توالی)؛ کنترل ظرفیت فقط یک adapter bottleneck است (نه head عمیق‌تر یا ظرفیت دیگر).")
        L.append("- فیزیک: ViennaRNA با پارامترهای پیش‌فرض (RNAfold تنها baseline فیزیکی؛ BPfold/EternaFold بررسی نشده).")
        L.append("- CI ها ۸ مقایسه × ۴ split را تصحیح نمی‌کنند (فقط Bonferroni×3 درون هر ادعا برای splitهای OOD)؛ ادعاهای ✅ مرزی را محتاطانه بخوانید.")
        L.append("- GenF واحدهای کم دارد؛ آستانه روی val درون‌توزیع تنظیم می‌شود؛ AUPRC/oracle بدون CI و فقط میانگین seed هستند.")
        L.append("- F1@oracle فقط تشخیصی است (آستانه روی خود split انتخاب می‌شود).")
        report = "\n".join(L)
        print(report, flush=True)
        (A / "report_paper_v3.md").write_text(report, encoding="utf-8")
        gate(bool(main and ref and ana and leak), "گزارش ساخته شد" if (main and ref and ana and leak) else "بخشی از artifactها ناقص است", hard=False)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P9 — آمار v5 + مقایسه با ادبیات
CPU. بعد از P5 (+P5r). Holm، بوت‌استرپ واحد×seed، تعامل برای سه تفکیک: LM / پیش‌آموزش / ورودی تصادفی. خروجی: `artifacts_paper/analysis_v5.json`.


In [ ]:
# fmt: off
# ============================================================
# P9 (v5) — آمار هم‌تراز با ادبیات: متریک per-sequence + micro، بوت‌استرپ «واحد × seed»، Holm، قاعدهٔ هم‌علامتی seedها،
#           شکاف/retention/تعامل (LM × split) و جدول مقایسه با اعداد منتشرشدهٔ CHANRG (length-matched 50..200)
# runtime: CPU | ~3-8 min | بدون وابستگی به plib/torch | artifact: artifacts_paper/analysis_v5.json
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p9_analysis", restore="config_paper.json; artifacts_paper/leakage.json; artifacts_paper/*.json; artifacts_paper/main_ckpt/*; artifacts_paper/counts_rnafold.npz; artifacts_paper/scale_ckpt/*; artifacts_paper/lora_ckpt/*; artifacts_paper/adapter_ckpt/*; data_paper/levels_*.parquet",
                 save="artifacts_paper/analysis_v5.json"):

        import json, os, pathlib, re
        import numpy as np
        import pandas as pd

        AR = pathlib.Path(os.environ.get("RNA_AR", "/content/rna_paper_v4_quick" if os.environ.get("RNA_QUICK", "") == "1" else "/content/rna_paper_v4"))
        OUT = AR / "artifacts_paper"
        EVAL, OOD = ["test", "gena", "genc", "genf"], ["gena", "genc", "genf"]
        QUICKM = os.environ.get("RNA_QUICK", "") == "1"
        NB = 300 if QUICKM else 4000
        MIN_UNITS = 5
        rng = np.random.RandomState(7)

        # ---- اعداد منتشرشده (arXiv:2603.22330). length-matched 50..200: GenA از متن (۴ رقم)، Test/GenC/GenF از heatmap (۲ رقم اعشار)
        LIT_LM = {"RNAfold": {"test": 0.32, "gena": 0.4193, "genc": 0.28, "genf": 0.28},
                  "RiNALMo-giga-U-Net (FM، fine-tune کامل)": {"test": 0.7832, "gena": 0.3565, "genc": 0.1525, "genf": 0.2298},
                  "BPfold (DL)": {"test": 0.45, "gena": 0.5687, "genc": 0.40, "genf": 0.28},
                  "EternaFold (SD)": {"test": 0.33, "gena": 0.4608, "genc": 0.29, "genf": 0.29}}
        LIT_FULL = {"RNAfold": {"test": 0.3013, "ood": 0.2886}, "RiNALMo-giga-U-Net": {"test": 0.7579, "gena": 0.2509, "genc": 0.1651, "genf": 0.2260, "ood": 0.2140},
                    "BPfold": {"test": 0.4065, "ood": 0.3608}, "EternaFold": {"test": 0.3189, "ood": 0.3064}}

        leak = json.load(open(OUT / "leakage.json"))
        meta_fam = {}
        units = {}
        for n in EVAL:
            lv = pd.read_parquet(AR / "data_paper" / f"levels_{n}.parquet")
            u = leak["splits"][n]["unit"]
            units[n] = lv[u].astype(str).tolist()
            meta_fam[n] = len(units[n])
        unit_info = {n: {"unit": leak["splits"][n]["unit"], "n_units": len(set(units[n])), "n_seq": len(units[n])} for n in EVAL}
        print("واحد بازنمونه:", unit_info, flush=True)
        IDX = {n: {u: k for k, u in enumerate(sorted(set(units[n])))} for n in EVAL}
        UID = {n: np.array([IDX[n][u] for u in units[n]]) for n in EVAL}


        def load_sys(files, prefix=""):
            """-> {seed_key: {split: (tp,fp,fn)}}"""
            d = {}
            for f in files:
                m = re.search(r"_s(\d+)\.npz$", str(f)); key = int(m.group(1)) if m else "det"
                z = np.load(f)
                d[key] = {n: tuple(z[f"{prefix}{n}_{k}"].astype(float) for k in ("tp", "fp", "fn")) for n in EVAL}
            return d


        SYS = {}
        for arm in ["unet+phys", "unet", "nolm+phys", "randlm+phys"]:
            fs = sorted((OUT / "main_ckpt").glob(f"{arm}_s*.npz"))
            if fs: SYS[arm] = load_sys(fs)
        if (OUT / "counts_rnafold.npz").exists():
            SYS["rnafold_mfe"] = load_sys([OUT / "counts_rnafold.npz"], "mfe_")
            SYS["rnafold_bpp"] = load_sys([OUT / "counts_rnafold.npz"], "bpp_")
        for name, pat in [("frozen_matched", "lora_ckpt/counts_frozen_matched_s*.npz"), ("lora_matched", "lora_ckpt/counts_lora_matched_s*.npz"),
                          ("adapter_matched", "adapter_ckpt/counts_adapter_matched_s*.npz")]:
            fs = sorted(OUT.glob(pat))
            if fs: SYS[name] = load_sys(fs)
        print("سامانه‌ها (تعداد seed):", {k: len(v) for k, v in SYS.items()}, flush=True)
        _ns = min([len(v) for k, v in SYS.items() if not k.startswith("rnafold")] or [0])
        if _ns < 3:
            print(f"WARN | کمتر از ۳ seed ({_ns}) — بوت‌استرپ seed و قاعدهٔ هم‌علامتی بی‌اعتبارند؛ این اجرا فقط برای تصمیمِ «مرحلهٔ A» است نه ادعا.", flush=True)


        # ---- جدول واحد×آماره برای هر (سامانه، seed، split، متریک)
        def unit_table(tpfpfn, n, metric):
            tp, fp, fn = tpfpfn; U = len(IDX[n]); T = np.zeros((U, 3))
            if metric == "micro":
                for j, v in enumerate((tp, fp, fn)): np.add.at(T[:, j], UID[n], v)
                return T                                   # ستون‌ها: tp, fp, fn
            has = (tp + fn) > 0
            f = np.where(has, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), 0.0)
            np.add.at(T[:, 0], UID[n], f); np.add.at(T[:, 1], UID[n], has.astype(float))
            return T                                       # ستون‌ها: sumF1، n_valid


        def stat_from(Wt, metric):
            """Wt: (B,3) یا (B,2) جمع‌های وزن‌دار → مقدار متریک"""
            if metric == "micro":
                return 2 * Wt[:, 0] / np.maximum(2 * Wt[:, 0] + Wt[:, 1] + Wt[:, 2], 1.0)
            return Wt[:, 0] / np.maximum(Wt[:, 1], 1.0)


        # وزن‌های بازنمونهٔ واحدها (مشترک بین همهٔ مقایسه‌ها) و انتخاب seed (با جای‌گذاری)
        W = {n: rng.multinomial(len(IDX[n]), np.ones(len(IDX[n])) / len(IDX[n]), size=NB).astype(float) for n in EVAL}
        JBIG = rng.randint(0, 10 ** 6, size=(NB, 16))     # شاخص‌های تصادفی؛ با mod روی تعداد seed استفاده می‌شود


        def boot_system(arm, n, metric):
            """-> (point، boot_matrix (B,)) با بازنمونهٔ واحد و seed. سامانهٔ «det» (RNAfold) seed ندارد."""
            sd = SYS[arm]; keys = sorted(k for k in sd if k != "det") or ["det"]
            per = []
            for k in keys:
                T = unit_table(sd[k][n], n, metric); per.append(stat_from(W[n] @ T, metric))
            P = np.stack(per)                                   # (S,B)
            pts = np.mean([stat_from(unit_table(sd[k][n], n, metric).sum(0, keepdims=True), metric)[0] for k in keys])
            S = len(keys)
            if S == 1: return float(pts), P[0], keys
            J = JBIG[:, :S] % S                                 # (B,S)
            return float(pts), np.take_along_axis(P, J.T, 0).mean(0), keys


        def per_seed_points(arm, n, metric):
            sd = SYS[arm]; keys = sorted(k for k in sd if k != "det") or ["det"]
            return {k: float(stat_from(unit_table(sd[k][n], n, metric).sum(0, keepdims=True), metric)[0]) for k in keys}


        def pval(d):
            b = len(d); return float(min(1.0, 2 * (1 + min((d <= 0).sum(), (d >= 0).sum())) / (b + 1)))


        COMPS = [("unet+phys", "rnafold_mfe", "C4a مدل اصلی − RNAfold-MFE"), ("unet+phys", "nolm+phys", "C1/C2 سهم LM"),
                 ("unet+phys", "unet", "C3 سهم فیزیک"), ("nolm+phys", "rnafold_mfe", "C4c بدون LM − RNAfold-MFE"),
                 ("lora_matched", "frozen_matched", "C5a LoRA − frozen"), ("adapter_matched", "frozen_matched", "C5b adapter − frozen"),
                 ("lora_matched", "adapter_matched", "C5c LoRA − adapter"),
                 ("unet+phys", "randlm+phys", "C6 دانش پیش‌آموزش (LM − LM تصادفی)"), ("randlm+phys", "nolm+phys", "C7 ورودی پرابعاد تصادفی − بدون LM")]


        def holm(ps):
            o = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
            for r, i in enumerate(o):
                run = max(run, (m - r) * ps[i]); adj[i] = min(1.0, run)
            return adj


        def run_family(metric):
            rows = []
            for a, b, title in COMPS:
                if a not in SYS or b not in SYS: continue
                for n in EVAL:
                    if unit_info[n]["n_units"] < MIN_UNITS:
                        rows.append(dict(comp=title, split=n, a=a, b=b, valid=False)); continue
                    pa, Ba, ka = boot_system(a, n, metric); pb, Bb, kb = boot_system(b, n, metric)
                    d = Ba - Bb; d0 = pa - pb
                    sa, sb = per_seed_points(a, n, metric), per_seed_points(b, n, metric)
                    common = [k for k in sa if k in sb] or list(sa)
                    sg = [sa[k] - (sb[k] if k in sb else list(sb.values())[0]) for k in common]
                    rows.append(dict(comp=title, split=n, a=a, b=b, valid=True, delta=round(d0, 4), ci95=[round(float(np.percentile(d, 2.5)), 4), round(float(np.percentile(d, 97.5)), 4)],
                                     p=pval(d), seeds_pos=int(sum(x > 0 for x in sg)), seeds_neg=int(sum(x < 0 for x in sg)), n_seeds=len(sg)))
            ok = [r for r in rows if r["valid"]]
            adj = holm(np.array([r["p"] for r in ok])) if ok else []
            for r, ap in zip(ok, adj):
                r["p_holm"] = round(float(ap), 4)
                pos = r["delta"] > 0 and r["seeds_pos"] == r["n_seeds"]; neg = r["delta"] < 0 and r["seeds_neg"] == r["n_seeds"]
                r["status"] = "✅" if (ap < 0.05 and pos) else ("🔻" if (ap < 0.05 and neg) else "⬜")
                r["p"] = round(r["p"], 4)
            return rows, len(ok)


        res = {"unit_info": unit_info, "n_boot": NB, "rule": "✅/🔻 فقط اگر p_Holm<0.05 (روی کل خانوادهٔ آزمون‌ها) و Δ در همهٔ seedهای مشترک هم‌علامت باشد؛ CI = بوت‌استرپ واحد×seed"}
        for metric in ["macro", "micro"]:
            rows, m = run_family(metric)
            res[f"tests_{metric}"] = rows; res[f"n_tests_{metric}"] = m
            print(f"\n=== آزمون‌ها ({metric}-F1) | m={m} آزمون | Holm روی همهٔ آن‌ها ===", flush=True)
            for r in rows:
                if not r["valid"]: print(f"{r['comp']:<34} {r['split']:<5} ‡ (<{MIN_UNITS} واحد)", flush=True); continue
                print(f"{r['comp']:<34} {r['split']:<5} Δ={r['delta']:+.3f} CI95[{r['ci95'][0]:+.3f},{r['ci95'][1]:+.3f}] p={r['p']:.3f} pHolm={r['p_holm']:.3f} "
                      f"seeds+{r['seeds_pos']}/−{r['seeds_neg']}/{r['n_seeds']} {r['status']}", flush=True)

        # ---------------- شکاف، retention و تعامل (Test در برابر OOD؛ نمونه‌های مستقل ⇒ بازنمونهٔ مستقل)
        print("\n=== شکاف و retention (macro و micro) ===", flush=True)
        res["gap"] = {}
        for metric in ["macro", "micro"]:
            res["gap"][metric] = {}
            for arm in ["unet+phys", "unet", "nolm+phys", "randlm+phys", "rnafold_mfe"]:
                if arm not in SYS: continue
                pt, Bt, _ = boot_system(arm, "test", metric)
                r = {"test": round(pt, 4)}
                oo_pts, oo_B = [], []
                for n in OOD:
                    p_, B_, _ = boot_system(arm, n, metric)
                    oo_pts.append(p_); oo_B.append(B_)
                    ret = B_ / np.maximum(Bt, 1e-9)
                    r[n] = {"f1": round(p_, 4), "retention": round(p_ / max(pt, 1e-9), 3), "retention_ci95": [round(float(np.percentile(ret, 2.5)), 3), round(float(np.percentile(ret, 97.5)), 3)],
                            "gap_test_minus_ood": round(pt - p_, 4)}
                om, oB = float(np.mean(oo_pts)), np.mean(oo_B, 0)
                r["ood_mean"] = {"f1": round(om, 4), "retention": round(om / max(pt, 1e-9), 3), "retention_ci95": [round(float(np.percentile(oB / np.maximum(Bt, 1e-9), 2.5)), 3), round(float(np.percentile(oB / np.maximum(Bt, 1e-9), 97.5)), 3)],
                                 "gap_test_minus_ood": round(pt - om, 4), "gap_ci95": [round(float(np.percentile(Bt - oB, 2.5)), 4), round(float(np.percentile(Bt - oB, 97.5)), 4)]}
                res["gap"][metric][arm] = r
                print(f"[{metric}] {arm:<12} Test {pt:.3f} | " + " ".join(f"{n}:{r[n]['f1']:.3f}" for n in OOD) +
                      f" | OODmean {om:.3f} retention {r['ood_mean']['retention']:.2f} {r['ood_mean']['retention_ci95']} gap {r['ood_mean']['gap_test_minus_ood']:+.3f} {r['ood_mean']['gap_ci95']}", flush=True)

        res["interaction"] = {}
        INTER = [("LM", "unet+phys", "nolm+phys", "سهم LM (با − بدون)"), ("pretrain", "unet+phys", "randlm+phys", "سهم دانش پیش‌آموزش (LM − LM تصادفی)"),
                 ("randfeat", "randlm+phys", "nolm+phys", "سهم ورودی پرابعاد تصادفی (LM تصادفی − بدون LM)")]
        for key, A_, B_, title in INTER:
            if A_ not in SYS or B_ not in SYS: continue
            print(f"\n=== تعامل: {title} — آیا در OOD کمتر از Test است؟  [Δ(Test) − Δ(OOD)] ===", flush=True)
            res["interaction"][key] = {}
            for metric in ["macro", "micro"]:
                pa, Ba, _ = boot_system(A_, "test", metric); pb, Bb, _ = boot_system(B_, "test", metric)
                dT, dT0 = Ba - Bb, pa - pb
                res["interaction"][key][metric] = {}
                for n in OOD + ["ood_mean"]:
                    if n == "ood_mean":
                        dl = [boot_system(A_, s_, metric) for s_ in OOD]; dn = [boot_system(B_, s_, metric) for s_ in OOD]
                        dO = np.mean([x[1] - y[1] for x, y in zip(dl, dn)], 0); dO0 = np.mean([x[0] - y[0] for x, y in zip(dl, dn)])
                    else:
                        qa, Qa, _ = boot_system(A_, n, metric); qb, Qb, _ = boot_system(B_, n, metric); dO, dO0 = Qa - Qb, qa - qb
                    ix = dT - dO
                    e = {"delta_test": round(dT0, 4), "delta_ood": round(float(dO0), 4), "interaction": round(float(dT0 - dO0), 4),
                         "ci95": [round(float(np.percentile(ix, 2.5)), 4), round(float(np.percentile(ix, 97.5)), 4)], "p": round(pval(ix), 4)}
                    res["interaction"][key][metric][n] = e
                    print(f"[{metric}] {n:<8}: Δ(Test)={e['delta_test']:+.3f} Δ(OOD)={e['delta_ood']:+.3f} تعامل={e['interaction']:+.3f} CI95{e['ci95']} p={e['p']}", flush=True)
        res["interaction_LM"] = res["interaction"].get("LM", {})

        # ---------------- هم‌ترازی با ادبیات
        print("\n=== مقایسه با ادبیات (length-matched 50..200؛ انتشار: مدل کامل/سمپل کامل — ما: زیرمجموعه) ===", flush=True)
        lit_rows = []
        def line(name, f):
            om = np.mean([f[n] for n in OOD]); return f"{name:<44} Test {f['test']:.3f} GenA {f['gena']:.3f} GenC {f['genc']:.3f} GenF {f['genf']:.3f} | OODmean {om:.3f} retention {om / f['test']:.2f}"
        for nm, f in LIT_LM.items(): print("منتشرشده:", line(nm, f), flush=True)
        for metric in ["macro", "micro"]:
            for arm in ["unet+phys", "unet", "nolm+phys", "randlm+phys", "rnafold_mfe"]:
                if arm in SYS:
                    f = {n: boot_system(arm, n, metric)[0] for n in EVAL}; lit_rows.append({"metric": metric, "arm": arm, **f})
                    print(f"ما [{metric}]:", line(arm, f), flush=True)
        res["literature"] = {"lenmatched": LIT_LM, "fullsplit": LIT_FULL, "ours": lit_rows,
                             "caveats": "اعداد Test/GenC/GenF منتشرشده از heatmap با ۲ رقم اعشارند؛ منتشرشده روی splitهای کامل و ما روی زیرمجموعهٔ cap؛ تجمیع رسمی (per-sequence یا micro) را در مقاله تأیید کنید — هر دو گزارش می‌شود."}
        chk = {}
        if "rnafold_mfe" in SYS:
            for metric in ["macro", "micro"]:
                d = {n: boot_system("rnafold_mfe", n, metric)[0] - LIT_LM["RNAfold"][n] for n in EVAL}
                chk[metric] = {n: round(v, 3) for n, v in d.items()}
                print(f"RNAfold ما − منتشرشده [{metric}]: {chk[metric]}  (|Δ|≤0.03 ⇒ پایپ‌لاین و داده هم‌تراز با ادبیات)", flush=True)
        res["rnafold_anchor"] = chk
        fm = res["gap"].get("macro", {}).get("unet+phys")
        if fm:
            id_ok = fm["test"] >= 0.60; ret = fm["ood_mean"]["retention"]
            res["repro_flags"] = {"fm_id_f1_macro": fm["test"], "fm_id_reaches_0.60": bool(id_ok), "fm_retention": ret, "fm_retention_below_0.5": bool(ret < 0.5)}
            print(f"\nINFO | آیا «پدیدهٔ ادبیات» بازتولید شد؟ ID(F1)={fm['test']:.3f} (انتشار ≈0.78 با fine-tune کامل) → {'بله' if id_ok else 'خیر'} | retention={ret:.2f} (انتشار ≈0.31) → "
                  f"{'فروپاشی OOD دیده می‌شود' if ret < 0.5 else 'فروپاشی OOD دیده نمی‌شود (مدل به‌قدر کافی ID را حفظ نکرده یا decoder/آموزش کوچک است)'}", flush=True)
        json.dump(res, open(OUT / "analysis_v5.json", "w"), indent=2, ensure_ascii=False)
        print("\nذخیره شد:", OUT / "analysis_v5.json", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P10 — مقیاس‌پذیری + حفظ‌کردن (جدید در v5)
بعد از P5s (و P5r برای ستون pretrain). CPU. خروجی: `analysis_v5_scale_exposure.json`.


In [ ]:
# fmt: off
# fmt: off
# ============================================================
# (بخش مشترک با P9): متریک per-sequence + micro، بوت‌استرپ «واحد × seed»، Holm، قاعدهٔ هم‌علامتی seedها،
#           شکاف/retention/تعامل (LM × split) و جدول مقایسه با اعداد منتشرشدهٔ CHANRG (length-matched 50..200)
# runtime: CPU | ~3-8 min | بدون وابستگی به plib/torch | artifact: artifacts_paper/analysis_v4.json , report_v4.md
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p10_scale_exposure", restore="config_paper.json; artifacts_paper/leakage.json; artifacts_paper/*.json; artifacts_paper/main_ckpt/*; artifacts_paper/counts_rnafold.npz; artifacts_paper/scale_ckpt/*; artifacts_paper/lora_ckpt/*; artifacts_paper/adapter_ckpt/*; data_paper/levels_*.parquet",
                 save="artifacts_paper/analysis_v5_scale_exposure.json"):

        import json, os, pathlib, re
        import numpy as np
        import pandas as pd

        AR = pathlib.Path(os.environ.get("RNA_AR", "/content/rna_paper_v4_quick" if os.environ.get("RNA_QUICK", "") == "1" else "/content/rna_paper_v4"))
        OUT = AR / "artifacts_paper"
        EVAL, OOD = ["test", "gena", "genc", "genf"], ["gena", "genc", "genf"]
        QUICKM = os.environ.get("RNA_QUICK", "") == "1"
        NB = 300 if QUICKM else 4000
        MIN_UNITS = 5
        rng = np.random.RandomState(7)

        # ---- اعداد منتشرشده (arXiv:2603.22330). length-matched 50..200: GenA از متن (۴ رقم)، Test/GenC/GenF از heatmap (۲ رقم اعشار)
        LIT_LM = {"RNAfold": {"test": 0.32, "gena": 0.4193, "genc": 0.28, "genf": 0.28},
                  "RiNALMo-giga-U-Net (FM، fine-tune کامل)": {"test": 0.7832, "gena": 0.3565, "genc": 0.1525, "genf": 0.2298},
                  "BPfold (DL)": {"test": 0.45, "gena": 0.5687, "genc": 0.40, "genf": 0.28},
                  "EternaFold (SD)": {"test": 0.33, "gena": 0.4608, "genc": 0.29, "genf": 0.29}}
        LIT_FULL = {"RNAfold": {"test": 0.3013, "ood": 0.2886}, "RiNALMo-giga-U-Net": {"test": 0.7579, "gena": 0.2509, "genc": 0.1651, "genf": 0.2260, "ood": 0.2140},
                    "BPfold": {"test": 0.4065, "ood": 0.3608}, "EternaFold": {"test": 0.3189, "ood": 0.3064}}

        leak = json.load(open(OUT / "leakage.json"))
        meta_fam = {}
        units = {}
        for n in EVAL:
            lv = pd.read_parquet(AR / "data_paper" / f"levels_{n}.parquet")
            u = leak["splits"][n]["unit"]
            units[n] = lv[u].astype(str).tolist()
            meta_fam[n] = len(units[n])
        unit_info = {n: {"unit": leak["splits"][n]["unit"], "n_units": len(set(units[n])), "n_seq": len(units[n])} for n in EVAL}
        print("واحد بازنمونه:", unit_info, flush=True)
        IDX = {n: {u: k for k, u in enumerate(sorted(set(units[n])))} for n in EVAL}
        UID = {n: np.array([IDX[n][u] for u in units[n]]) for n in EVAL}


        def load_sys(files, prefix=""):
            """-> {seed_key: {split: (tp,fp,fn)}}"""
            d = {}
            for f in files:
                m = re.search(r"_s(\d+)\.npz$", str(f)); key = int(m.group(1)) if m else "det"
                z = np.load(f)
                d[key] = {n: tuple(z[f"{prefix}{n}_{k}"].astype(float) for k in ("tp", "fp", "fn")) for n in EVAL}
            return d


        SYS = {}
        for arm in ["unet+phys", "unet", "nolm+phys", "randlm+phys"]:
            fs = sorted((OUT / "main_ckpt").glob(f"{arm}_s*.npz"))
            if fs: SYS[arm] = load_sys(fs)
        if (OUT / "counts_rnafold.npz").exists():
            SYS["rnafold_mfe"] = load_sys([OUT / "counts_rnafold.npz"], "mfe_")
            SYS["rnafold_bpp"] = load_sys([OUT / "counts_rnafold.npz"], "bpp_")
        for name, pat in [("frozen_matched", "lora_ckpt/counts_frozen_matched_s*.npz"), ("lora_matched", "lora_ckpt/counts_lora_matched_s*.npz"),
                          ("adapter_matched", "adapter_ckpt/counts_adapter_matched_s*.npz")]:
            fs = sorted(OUT.glob(pat))
            if fs: SYS[name] = load_sys(fs)
        print("سامانه‌ها (تعداد seed):", {k: len(v) for k, v in SYS.items()}, flush=True)
        _ns = min([len(v) for k, v in SYS.items() if not k.startswith("rnafold")] or [0])
        if _ns < 3:
            print(f"WARN | کمتر از ۳ seed ({_ns}) — بوت‌استرپ seed و قاعدهٔ هم‌علامتی بی‌اعتبارند؛ این اجرا فقط برای تصمیمِ «مرحلهٔ A» است نه ادعا.", flush=True)


        # ---- جدول واحد×آماره برای هر (سامانه، seed، split، متریک)
        def unit_table(tpfpfn, n, metric):
            tp, fp, fn = tpfpfn; U = len(IDX[n]); T = np.zeros((U, 3))
            if metric == "micro":
                for j, v in enumerate((tp, fp, fn)): np.add.at(T[:, j], UID[n], v)
                return T                                   # ستون‌ها: tp, fp, fn
            has = (tp + fn) > 0
            f = np.where(has, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), 0.0)
            np.add.at(T[:, 0], UID[n], f); np.add.at(T[:, 1], UID[n], has.astype(float))
            return T                                       # ستون‌ها: sumF1، n_valid


        def stat_from(Wt, metric):
            """Wt: (B,3) یا (B,2) جمع‌های وزن‌دار → مقدار متریک"""
            if metric == "micro":
                return 2 * Wt[:, 0] / np.maximum(2 * Wt[:, 0] + Wt[:, 1] + Wt[:, 2], 1.0)
            return Wt[:, 0] / np.maximum(Wt[:, 1], 1.0)


        # وزن‌های بازنمونهٔ واحدها (مشترک بین همهٔ مقایسه‌ها) و انتخاب seed (با جای‌گذاری)
        W = {n: rng.multinomial(len(IDX[n]), np.ones(len(IDX[n])) / len(IDX[n]), size=NB).astype(float) for n in EVAL}
        JBIG = rng.randint(0, 10 ** 6, size=(NB, 16))     # شاخص‌های تصادفی؛ با mod روی تعداد seed استفاده می‌شود


        def boot_system(arm, n, metric):
            """-> (point، boot_matrix (B,)) با بازنمونهٔ واحد و seed. سامانهٔ «det» (RNAfold) seed ندارد."""
            sd = SYS[arm]; keys = sorted(k for k in sd if k != "det") or ["det"]
            per = []
            for k in keys:
                T = unit_table(sd[k][n], n, metric); per.append(stat_from(W[n] @ T, metric))
            P = np.stack(per)                                   # (S,B)
            pts = np.mean([stat_from(unit_table(sd[k][n], n, metric).sum(0, keepdims=True), metric)[0] for k in keys])
            S = len(keys)
            if S == 1: return float(pts), P[0], keys
            J = JBIG[:, :S] % S                                 # (B,S)
            return float(pts), np.take_along_axis(P, J.T, 0).mean(0), keys


        def per_seed_points(arm, n, metric):
            sd = SYS[arm]; keys = sorted(k for k in sd if k != "det") or ["det"]
            return {k: float(stat_from(unit_table(sd[k][n], n, metric).sum(0, keepdims=True), metric)[0]) for k in keys}


        def pval(d):
            b = len(d); return float(min(1.0, 2 * (1 + min((d <= 0).sum(), (d >= 0).sum())) / (b + 1)))



        # ============================================================
        # P10 (v5) — مقیاس‌پذیری (ID در برابر OOD) + تحلیل «حفظ‌کردن» (سود LM بر حسب تعداد نمونهٔ همان family در train)
        # بعد از P5s اجرا شود. CPU، بدون torch. artifact: analysis_v5_scale_exposure.json
        # ============================================================
        import pandas as pd
        cfgp = json.load(open(AR / "config_paper.json"))
        N_MAIN = int(cfgp["train_n"])
        SC = OUT / "scale_ckpt"
        for arm in ["unet+phys", "nolm+phys"]:
            if arm in SYS: SYS[f"{arm}@{N_MAIN}"] = SYS[arm]
        sizes = set([N_MAIN])
        for f in sorted(SC.glob("*_n*_s*.npz")) if SC.exists() else []:
            m = re.match(r"(.+)_n(\d+)_s(\d+)\.npz$", f.name)
            arm, N = m.group(1), int(m.group(2))
            SYS.setdefault(f"{arm}@{N}", {}).update(load_sys([f])); sizes.add(N)
        sizes = sorted(sizes)
        have = [N for N in sizes if f"unet+phys@{N}" in SYS and f"nolm+phys@{N}" in SYS]
        print("اندازه‌های train با هر دو arm:", have, {N: len(SYS[f"unet+phys@{N}"]) for N in have}, flush=True)
        res10 = {"sizes": have}


        def slope_ci(xs, Ys):
            """Ys: لیست آرایه‌های (B,) به‌ازای هر x → شیب OLS روی log2(N) با CI بوت‌استرپ"""
            x = np.log2(np.array(xs, float)); xc = x - x.mean(); Y = np.stack(Ys)
            sl = (xc[:, None] * (Y - Y.mean(0))).sum(0) / (xc ** 2).sum()
            return sl


        if len(have) >= 3:
            for metric in ["macro"]:
                print(f"\n=== منحنی مقیاس ({metric}-F1) — N = تعداد توالی train (family-تصادفی تو‌در‌تو)، epoch ثابت ===", flush=True)
                curve = {}
                for arm in ["unet+phys", "nolm+phys"]:
                    curve[arm] = {}
                    for N in have:
                        t = boot_system(f"{arm}@{N}", "test", metric)
                        oo = [boot_system(f"{arm}@{N}", s_, metric) for s_ in OOD]
                        curve[arm][N] = {"test": t, "ood": (float(np.mean([x[0] for x in oo])), np.mean([x[1] for x in oo], 0)), "oo_each": {s_: oo[k][0] for k, s_ in enumerate(OOD)}}
                    print(f"{arm:<10} " + " | ".join(f"N={N}: Test {curve[arm][N]['test'][0]:.3f} OOD {curve[arm][N]['ood'][0]:.3f}" for N in have), flush=True)
                gainT = [curve["unet+phys"][N]["test"][1] - curve["nolm+phys"][N]["test"][1] for N in have]
                gainO = [curve["unet+phys"][N]["ood"][1] - curve["nolm+phys"][N]["ood"][1] for N in have]
                g0T = [curve["unet+phys"][N]["test"][0] - curve["nolm+phys"][N]["test"][0] for N in have]
                g0O = [curve["unet+phys"][N]["ood"][0] - curve["nolm+phys"][N]["ood"][0] for N in have]
                print("سود LM (با − بدون):  " + " | ".join(f"N={N}: Test {a:+.3f} OOD {b:+.3f}" for N, a, b in zip(have, g0T, g0O)), flush=True)
                x = np.log2(np.array(have, float)); xc = x - x.mean()
                pt = lambda ys: float((xc * (np.array(ys) - np.mean(ys))).sum() / (xc ** 2).sum())
                rows = {}
                for name, Ys, pts in [("F1_test(LM)", [curve["unet+phys"][N]["test"][1] for N in have], [curve["unet+phys"][N]["test"][0] for N in have]),
                                      ("F1_OOD(LM)", [curve["unet+phys"][N]["ood"][1] for N in have], [curve["unet+phys"][N]["ood"][0] for N in have]),
                                      ("F1_test(noLM)", [curve["nolm+phys"][N]["test"][1] for N in have], [curve["nolm+phys"][N]["test"][0] for N in have]),
                                      ("F1_OOD(noLM)", [curve["nolm+phys"][N]["ood"][1] for N in have], [curve["nolm+phys"][N]["ood"][0] for N in have]),
                                      ("gain_LM_test", gainT, g0T), ("gain_LM_OOD", gainO, g0O)]:
                    sl = slope_ci(have, Ys); rows[name] = {"slope_per_doubling": round(pt(pts), 4), "ci95": [round(float(np.percentile(sl, 2.5)), 4), round(float(np.percentile(sl, 97.5)), 4)]}
                    print(f"  شیب به‌ازای دو برابر شدن داده — {name:<14}: {rows[name]['slope_per_doubling']:+.4f}  CI95{rows[name]['ci95']}", flush=True)
                d = slope_ci(have, gainT) - slope_ci(have, gainO)
                rows["slope_diff_gain_test_minus_ood"] = {"value": round(pt(g0T) - pt(g0O), 4), "ci95": [round(float(np.percentile(d, 2.5)), 4), round(float(np.percentile(d, 97.5)), 4)], "p": round(pval(d), 4)}
                print(f"  ★ تفاوت شیب سود LM (Test − OOD) = {rows['slope_diff_gain_test_minus_ood']['value']:+.4f} CI95{rows['slope_diff_gain_test_minus_ood']['ci95']} p={rows['slope_diff_gain_test_minus_ood']['p']}", flush=True)
                print("    (مثبت و معنادار ⇒ با بزرگ‌تر شدن داده، سود LM در ID بیشتر از OOD رشد می‌کند.)", flush=True)
                res10["scale"] = {"curve_points": {a: {str(N): {"test": round(v["test"][0], 4), "ood": round(v["ood"][0], 4)} for N, v in c.items()} for a, c in curve.items()}, "slopes": rows}
        else:
            print("WARN | کمتر از ۳ اندازه برای منحنی مقیاس (P5s را اجرا کنید).", flush=True)

        # ---------------- حفظ‌کردن: سود (LM − بدون LM) بر حسب تعداد توالی‌های همان family در train (cap)
        print("\n=== حفظ‌کردن: سود LM بر حسب «تعداد نمونهٔ همان family در train» (۰ = OOD) ===", flush=True)
        famc = cfgp["family_col"]
        trf = pd.read_parquet(AR / "data_paper" / "train.parquet")[famc].astype(str)
        cnt = trf.value_counts()
        tef = pd.read_parquet(AR / "data_paper" / "test.parquet")[famc].astype(str)
        expo = tef.map(cnt).fillna(0).astype(int).values
        print("توزیع مواجهه در Test:", dict(pd.Series(expo).value_counts().sort_index()), flush=True)


        def perseq_f1(arm, n):
            sd = SYS[arm]; ks = sorted(k for k in sd if k != "det") or ["det"]; Fs = []
            for k in ks:
                tp, fp, fn = sd[k][n]; has = (tp + fn) > 0
                Fs.append(np.where(has, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), np.nan))
            return np.nanmean(np.stack(Fs), 0)


        def expo_bins(e):
            u = np.unique(e); top = u.max()
            bins = [(f"{int(v)}", e == v) for v in u if v < min(top, 3)]
            bins.append((f"≥{int(min(top, 3))}", e >= min(top, 3)))
            return bins


        res10["exposure"] = {}
        for key, A_, B_ in [("LM", "unet+phys", "nolm+phys"), ("pretrain", "unet+phys", "randlm+phys")]:
            if A_ not in SYS or B_ not in SYS: continue
            g = perseq_f1(A_, "test") - perseq_f1(B_, "test"); ok = ~np.isnan(g)
            rows = []
            for lab, msk in expo_bins(expo):
                mk = msk & ok; U = len(IDX["test"])
                su = np.bincount(UID["test"][mk], weights=g[mk], minlength=U); nu = np.bincount(UID["test"][mk], minlength=U).astype(float)
                Bm = (W["test"] @ su) / np.maximum(W["test"] @ nu, 1.0)
                rows.append((lab, int(mk.sum()), float(g[mk].mean()), Bm))
            # OOD (مواجهه ۰)
            gO = []
            for n in OOD:
                gn = perseq_f1(A_, n) - perseq_f1(B_, n); ok_n = ~np.isnan(gn); U = len(IDX[n])
                su = np.bincount(UID[n][ok_n], weights=gn[ok_n], minlength=U); nu = np.bincount(UID[n][ok_n], minlength=U).astype(float)
                gO.append((float(gn[ok_n].mean()), (W[n] @ su) / np.maximum(W[n] @ nu, 1.0), int(ok_n.sum())))
            rows.append(("0 (OOD)", sum(x[2] for x in gO), float(np.mean([x[0] for x in gO])), np.mean([x[1] for x in gO], 0)))
            res10["exposure"][key] = {}
            print(f"-- سود {key}: {A_} − {B_}", flush=True)
            for lab, n_, m_, Bm in rows:
                ci = [round(float(np.percentile(Bm, 2.5)), 4), round(float(np.percentile(Bm, 97.5)), 4)]
                res10["exposure"][key][lab] = {"n_seq": n_, "mean_gain": round(m_, 4), "ci95": ci}
                print(f"   مواجهه {lab:<8} n={n_:<5} سود={m_:+.3f} CI95{ci}", flush=True)
            if len(rows) >= 3:                  # حداقل دو سطح مواجههٔ ID + OOD
                lo, hi = rows[0], rows[-2]      # کمترین مواجههٔ ID در برابر بیشترین
                d = hi[3] - lo[3]
                res10["exposure"][key]["trend_high_minus_low"] = {"value": round(hi[2] - lo[2], 4), "ci95": [round(float(np.percentile(d, 2.5)), 4), round(float(np.percentile(d, 97.5)), 4)], "p": round(pval(d), 4)}
                print(f"   روند (مواجههٔ {hi[0]} − {lo[0]}) = {hi[2]-lo[2]:+.3f} CI95{res10['exposure'][key]['trend_high_minus_low']['ci95']} p={res10['exposure'][key]['trend_high_minus_low']['p']}", flush=True)
        print("\nنکته: دامنهٔ مواجهه در Test کوتاه است (train متوازن‌شده ≈ ۱–۴ نمونه برای هر family)؛ منحنی مقیاس و تفاضل با OOD مواجههٔ مؤثرتر را می‌سنجند.", flush=True)
        json.dump(res10, open(OUT / "analysis_v5_scale_exposure.json", "w"), indent=2, ensure_ascii=False)
        print("ذخیره شد:", OUT / "analysis_v5_scale_exposure.json", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

In [ ]:
# fmt: off
# ============================================================
# وضعیت Drive — خلاصهٔ مراحل کامل‌شده + فضای آزاد (اختیاری)
# ============================================================
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)
D.status()


---
# بخش تکمیلی v7 — backbone دوم، LoRA مقیاس کامل، دوز–پاسخ، آزمون ضربی، جدول‌های گزارش
ترتیب اجرا: **P2b → P5x → P6L (اختیاری) → P12a → P12b**؛ سپس P10d را برای capهای جدید دوباره اجرا کنید. همهٔ سلول‌ها Drive-backed و resume‌پذیرند.


## P2b — embedding backbone دوم (RNA-FM پیش‌فرض)


In [ ]:
# fmt: off
# ============================================================
# P2b — embedding با backbone دوم (پیش‌فرض: RNA-FM؛ جایگزین: ERNIE-RNA) از multimolecule — برای ادعای عمومی‌تر. فقط decoder روی آن آموزش می‌بیند (P5x: bb2_full)
# فیکس‌های v1: offset بر حسب «سطر» (نه عنصر) ، assert درست ، بدون خواندن کل split در RAM ، seq_col از config
# runtime: GPU T4 | RNA-FM ≈ ۸-۱۵ min (مدل کوچک‌تر از giga؛ تخمین) | artifact: emb_b2/{split}.npy (N_res × dim fp16) + {split}_meta.npz + embed_done.json
# env: RNA_BB2="multimolecule/rnafm"  (یا "multimolecule/ernierna" — توجه: مجوز ERNIE-RNA در HF برابر AGPL-3.0 است)
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)   # در کرنل قدیمی ممکن است ماژول کش‌شدهٔ باگی مانده باشد → از فایل تازه بازخوانی می‌شود
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p2b_embed_backbone2", restore="data_paper/*.parquet; config_paper.json",
                 save="emb_b2/*.npy; emb_b2/*_meta.npz; emb_b2/embed_done.json; emb_b2/*.tmp.npy; emb_b2/*.progress"):

        import json, os, sys, time
        sys.path.insert(0, "/content/rna_plib")
        import importlib
        import numpy as np
        import torch
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p2b_embed_backbone2"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        EMB = AR / "emb_b2"; EMB.mkdir(parents=True, exist_ok=True)
        print(f"### {STAGE} START | device={P.DEV}", flush=True)

        done = EMB / "embed_done.json"
        if done.exists():
            d = json.load(open(done))
            if all((EMB / f"{s}.npy").exists() and (EMB / f"{s}_meta.npz").exists() for s in P.SPLITS):
                print("ART_CHECK: embed_done — skip", flush=True)
                raise SystemExit(0)

        BB2 = os.environ.get("RNA_BB2", "multimolecule/rnafm")

        def load_bb2(name):
            """backbone دوم از multimolecule (fp16). -> (model, tok, name, dim)"""
            import transformers as tf
            for cls in (tf.AutoConfig, tf.AutoModel, tf.AutoTokenizer):
                orig = cls.register

                def safe(*a, _o=orig, **k):
                    try:
                        return _o(*a, **k)
                    except ValueError:
                        return None
                cls.register = safe
            import multimolecule
            from multimolecule import RnaTokenizer
            tok = RnaTokenizer.from_pretrained(name)
            key = name.split("/")[-1].replace(".", "-").split("-")[0].lower()
            cls_name = {"rnafm": "RnaFmModel", "mrnafm": "RnaFmModel", "ernierna": "ErnieRnaModel", "rinalmo": "RiNALMoModel"}.get(key)
            M = getattr(multimolecule, cls_name) if cls_name and hasattr(multimolecule, cls_name) else tf.AutoModel
            try:
                model = M.from_pretrained(name, dtype=torch.float16)
            except Exception:
                model = M.from_pretrained(name, torch_dtype=torch.float16)
            return model.to(P.DEV).eval(), tok, name, int(model.config.hidden_size)

        model, tok, name, dim = load_bb2(BB2)
        print(f"backbone: {name} | dim={dim} | params={sum(p.numel() for p in model.parameters())/1e6:.0f}M", flush=True)
        import pandas as pd

        for split in P.SPLITS:
            fin, meta = EMB / f"{split}.npy", EMB / f"{split}_meta.npz"
            df = pd.read_parquet(AR / "data_paper" / f"{split}.parquet")
            seqs = [P.norm_seq(s) for s in df[C["seq_col"]]]
            lens = np.array([len(s) for s in seqs], dtype=np.int64)
            if fin.exists() and meta.exists() and (np.load(meta)["lens"] == lens).all():
                print(f"  {split}: موجود و معتبر — skip", flush=True); continue
            tmp, prog = EMB / f"{split}.tmp.npy", EMB / f"{split}.progress"
            total = int(lens.sum())
            start = 0
            if tmp.exists() and prog.exists():
                start = int(prog.read_text())
                mm = np.lib.format.open_memmap(tmp, mode="r+")
                if mm.shape != (total, dim):
                    start, mm = 0, np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(total, dim))
            else:
                mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(total, dim))
            roff = np.concatenate([[0], np.cumsum(lens)[:-1]])
            t0 = time.time()
            with torch.no_grad():
                for k in range(start, len(seqs)):
                    h = P.embed_tokens(model, tok, seqs[k])
                    assert h.shape[0] == lens[k], (split, k, h.shape, lens[k])
                    mm[roff[k]:roff[k] + lens[k]] = h.cpu().numpy().astype(np.float16)
                    if (k + 1) % 500 == 0 or k + 1 == len(seqs):
                        mm.flush(); prog.write_text(str(k + 1))
                        print(f"  {split}: {k+1}/{len(seqs)} | {time.time()-t0:.0f}s", flush=True)
            mm.flush(); del mm
            samp = np.load(tmp, mmap_mode="r")[:: max(1, total // 50)]
            gate(bool(np.isfinite(samp.astype(np.float32)).all()), f"{split}: embeddings متناهی", hard=True)
            _sd = float(samp.astype(np.float32).std(0).mean())
            print(f"  {split}: میانگین std هر بُعد روی موقعیت‌ها = {_sd:.4f}", flush=True)
            gate(_sd > 1e-3, f"{split}: embedding فروپاشیده نیست (std={_sd:.4f})", hard=False)
            os.replace(tmp, fin)
            np.savez(meta, lens=lens, dim=np.int32(dim))
            prog.unlink(missing_ok=True)
            print(f"  saved {split}: rows={total} dim={dim} ({total*dim*2/1e9:.2f}GB)", flush=True)

        json.dump({"model": name, "dim": dim, "batch": 1, "truncation": "none (length-filtered)",
                   "len_range": [C["len_min"], C["len_max"]]}, open(done, "w"), indent=2)
        gate(True, f"embeddings کامل | {name} dim={dim}")
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست

## P5x — آزمایش‌های تکمیلی (backbone دوم، LM تصادفی زیر سقف، seed سوم، سقف‌های ۳/۵۰)
با `RNA_XP_JOBS` انتخاب کنید؛ برآورد زمان قبل از شروع چاپ می‌شود.


In [ ]:
# fmt: off
# ============================================================
# P5x — اجراکنندهٔ آزمایش‌های تکمیلی (همه با همان decoder/پروتکل P5؛ نام‌گذاری سازگار با P10d/P12):
#   JOBS (env RNA_XP_JOBS، پیش‌فرض "bb2_full,randlm_cap,seed3"):
#     bb2_full   : backbone دوم (P2b)، train کامل، ۳ seed                                  → xp_ckpt/b2+phys_full_s*
#     bb2_cap    : backbone دوم، cap=10 و تصادفی هم‌اندازه                                → xp_ckpt/b2+phys_c10_*_s*
#     randlm_cap : LM تصادفی (P2r) زیر cap=10 و تصادفی هم‌اندازه، ۳ seed (آزمون ضربی)       → cap_ckpt/randlm+phys_c10_*_s*
#     seed3      : seed سوم برای cap=10 (unet+phys و nolm+phys)                           → cap_ckpt/*_c10_*_s2
#     dose3 / dose50 / dose{N} : سقف N برای هر دو arm، شرط cap و rand، seedهای RNA_XP_SEEDS  → cap_ckpt/*_c{N}_*
#   runهای انجام‌شده با همان PROTO skip می‌شوند؛ برآورد زمان قبل از شروع چاپ می‌شود.
# runtime: GPU T4 | پیش‌فرض ≈ ۲ ساعت (bb2_full ≈ ۳۳ + randlm_cap ≈ ۵۰ + seed3 ≈ ۳۳ min؛ برآورد)؛ dose3 ≈ ۴۵ min، dose50 ≈ ۸۰ min
# env: RNA_XP_JOBS  RNA_XP_SEEDS="0,1"  RNA_XP_SEEDS3="0,1,2"
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
import os, hashlib
JOBS = [x.strip() for x in os.environ.get("RNA_XP_JOBS", "bb2_full,randlm_cap,seed3").split(",") if x.strip()]
_sid = "p5x_" + "-".join(JOBS) + "_" + hashlib.md5((os.environ.get("RNA_XP_SEEDS", "") + "|" + os.environ.get("RNA_XP_SEEDS3", "")).encode()).hexdigest()[:6]
_rs = ["data_paper/*.parquet", "phys_paper/phys_*.npz", "config_paper.json", "emb_paper/*.npy", "emb_paper/*_meta.npz", "emb_paper/embed_done.json",
       "artifacts_paper/cap_ckpt/*", "artifacts_paper/xp_ckpt/*"]
if any(j == "randlm_cap" for j in JOBS): _rs += ["emb_rand/*.npy", "emb_rand/*_meta.npz", "emb_rand/embed_done.json"]
if any(j.startswith("bb2") for j in JOBS): _rs += ["emb_b2/*.npy", "emb_b2/*_meta.npz", "emb_b2/embed_done.json"]
_restore = "; ".join(_rs)
try:
    with D.stage(_sid,
                 restore=_restore,
                 save="artifacts_paper/cap_ckpt/*; artifacts_paper/xp_ckpt/*"):
        import collections, json, os, re, sys, time
        import numpy as np
        import torch
        import plib as P
        importlib.reload(P)

        AR = P.AR
        STAGE = "p5x_extra"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        EPOCHS = int(os.environ.get("RNA_EPOCHS", 1 if P.QUICK else 2))
        SEEDS_X = [int(x) for x in os.environ.get("RNA_XP_SEEDS", "0" if P.QUICK else "0,1").split(",")]
        SEEDS_3 = [int(x) for x in os.environ.get("RNA_XP_SEEDS3", "0" if P.QUICK else "0,1,2").split(",")]
        OUT = AR / "artifacts_paper"; CCK = OUT / "cap_ckpt"; XCK = OUT / "xp_ckpt"
        CCK.mkdir(parents=True, exist_ok=True); XCK.mkdir(parents=True, exist_ok=True)
        EMBD = {"rinalmo": "emb_paper", "randlm": "emb_rand", "b2": "emb_b2"}
        print(f"### {STAGE} START | jobs={JOBS} seeds={SEEDS_X} seeds3={SEEDS_3} epochs={EPOCHS}", flush=True)

        # ---------------- زیرمجموعه‌ها (همان permutation و منطق P5c ⇒ نتایج cap=10 قبلی با همان PROTO skip می‌شوند)
        SPL = {}
        def splits_of(bb):
            if bb not in SPL: SPL[bb] = {n: P.Split(n, emb_dir=EMBD[bb]) for n in P.SPLITS}
            return SPL[bb]
        tr0 = splits_of("rinalmo")["train"]
        fams = np.array([str(f) for f in tr0.fam])
        perm = np.random.RandomState(123).permutation(len(tr0))


        class _Sub:
            """نمای زیرمجموعه‌ای از train (API لازم برای build_arm/train_decoder)"""
            def __init__(self, sp, ix):
                self.sp, self.ix = sp, np.asarray(ix)
                self.L = sp.L[self.ix]
                self.fam = [sp.fam[int(i)] for i in self.ix]
            def __len__(self): return len(self.ix)
            def gt_flat(self, i): return self.sp.gt_flat(int(self.ix[i]))
            def h(self, i): return self.sp.h(int(self.ix[i]))
            def onehot(self, i): return self.sp.onehot(int(self.ix[i]))
            def canon_t(self, i): return self.sp.canon_t(int(self.ix[i]))
            def phys_t(self, i): return self.sp.phys_t(int(self.ix[i]))
            def n_phys(self): return self.sp.n_phys()


        def capped_idx(cap):
            seen, keep = collections.Counter(), []
            for i in perm:
                f = fams[i]
                if seen[f] < cap: keep.append(int(i)); seen[f] += 1
            return np.array(keep)


        # ---------------- فهرست runها: (bb, arm, cond, cap, seed)  — arm: برچسب فایل؛ bb: منبع embedding
        runs = []
        def add(bb, arm, cond, cap, seeds):
            for s in seeds: runs.append(dict(bb=bb, arm=arm, cond=cond, cap=cap, seed=s))
        for job in JOBS:
            m = re.fullmatch(r"dose(\d+)", job)
            if job == "bb2_full": add("b2", "b2+phys", "full", None, SEEDS_3)
            elif job == "bb2_cap":
                for cond in ("cap", "rand"): add("b2", "b2+phys", cond, 10, SEEDS_X)
            elif job == "randlm_cap":
                for cond in ("cap", "rand"): add("randlm", "randlm+phys", cond, 10, SEEDS_3)
            elif job == "seed3":
                for cond in ("cap", "rand"):
                    for arm in ("unet+phys", "nolm+phys"): add("rinalmo", arm, cond, 10, [2])
            elif m:
                for cond in ("cap", "rand"):
                    for arm in ("unet+phys", "nolm+phys"): add("rinalmo", arm, cond, int(m.group(1)), SEEDS_X)
            else: raise ValueError(f"job ناشناخته: {job}")


        def tag_path(r):
            c, s = r["cap"], r["seed"]
            if r["bb"] == "b2":
                t = f"b2+phys_full_s{s}" if r["cond"] == "full" else f"b2+phys_c{c}_{r['cond']}_s{s}"
                return XCK / t
            return CCK / f"{r['arm']}_c{c}_{r['cond']}_s{s}"


        def proto_of(r, N):
            bbname = json.load(open(AR / EMBD["rinalmo" if r["arm"] == "nolm+phys" else r["bb"]] / "embed_done.json"))["model"]
            return {"epochs": EPOCHS, "cap": r["cap"], "cond": r["cond"], "N": int(N), "pw": P.PW, "thr_grid": P.THR_GRID, "backbone": bbname, "unet_base": P.UNET_BASE, "perm_seed": 123}


        # ---------------- برآورد زمان
        NCAP = {c: len(capped_idx(c)) for c in sorted({r["cap"] for r in runs if r["cap"]})}
        def N_of(r): return len(tr0) if r["cond"] == "full" else NCAP[r["cap"]]
        todo = []
        for r in runs:
            base = tag_path(r); ck, cz = base.with_suffix(".json"), base.with_suffix(".npz")
            done = ck.exists() and cz.exists() and json.load(open(ck)).get("_proto") == proto_of(r, N_of(r))
            if not done: todo.append(r)
        est = sum(0.0165 * N_of(r) * EPOCHS + 70 for r in todo)
        print(f"N_cap: {NCAP} | کل runها={len(runs)} | باقی‌مانده={len(todo)} | برآورد زمان ≈ {est/60:.0f} دقیقه (۰٫۰۱۶۵ s/update + ۷۰ s eval؛ تخمین)", flush=True)

        t0 = time.time()
        for r in todo:
            bb, arm, cond, cap, seed = r["bb"], r["arm"], r["cond"], r["cap"], r["seed"]
            SP = splits_of("rinalmo" if arm == "nolm+phys" else bb)
            tr = SP["train"]
            if cond == "full": sub = tr
            else: sub = _Sub(tr, capped_idx(cap) if cond == "cap" else perm[:NCAP[cap]])
            N = len(sub); spec = P.ARMS["nolm+phys" if arm == "nolm+phys" else "unet+phys"]
            base = tag_path(r); tag = base.name
            PROTO = proto_of(r, N)
            torch.manual_seed(seed); np.random.seed(seed)
            model = P.build_arm(spec, SP["train"].dim, sub)
            P.train_decoder(model, spec, sub, seed, n_seq=EPOCHS * N, accum=1, tag=tag, log_every=5000)
            res, cnt = P.evaluate_arm(model, spec, SP, with_categories=False)
            res.update({"arm": arm, "bb": bb, "seed": seed, "cap": cap, "cond": cond, "N": int(N), "_proto": PROTO, "elapsed_s": round(time.time() - t0)})
            np.savez(base.with_suffix(".npz"), **cnt); json.dump(res, open(base.with_suffix(".json"), "w"), indent=2)
            print(f"[{tag}] N={N} val={res['val_f1']} thr={res['thr']} Test={res['splits']['test']['f1']} OOD={res['ood_mean']} | {time.time()-t0:.0f}s", flush=True)
            del model
            torch.cuda.empty_cache()

        print("\nخلاصه (میانگین seedها):", flush=True)
        groups = collections.defaultdict(list)
        for r in runs:
            base = tag_path(r)
            if base.with_suffix(".json").exists(): groups[(r["bb"], r["arm"], r["cond"], r["cap"])].append(json.load(open(base.with_suffix(".json"))))
        for (bb, arm, cond, cap), rs in sorted(groups.items(), key=lambda kv: str(kv[0])):
            print(f"  {arm:<12} {cond:<5} cap={str(cap):<5} | Test {np.mean([x['splits']['test']['f1'] for x in rs]):.3f} | OOD {np.mean([x['ood_mean'] for x in rs]):.3f} ({len(rs)} seed)", flush=True)
        gate(True, "P5x کامل شد (تفسیر در P12a/P12b و P10d)", hard=False)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست


## P6L — LoRA در مقیاس کامل (۱۲۰۰۰×۳)


In [ ]:
# fmt: off
# ============================================================
# P6L — LoRA در مقیاس کامل (پل به fine-tune): همان decoder/پروتکل P5 (۱۲۰۰۰ توالی × ۳ epoch، accum=1) با LoRA r=8 روی q/v چهار لایهٔ آخر RiNALMo.
#   مقایسه با frozen = همان arm «unet+phys» از P5 (همان decoder، همان seed، همان بودجه). فقط یک seed به‌طور پیش‌فرض.
# runtime: GPU T4 | ≈ ۶۰-۷۵ min به‌ازای هر seed (برآورد از P6: ۳۰۰۰ توالی ≈ ۹ min با eval ⇒ ۳۶۰۰۰ توالی ≈ ۶۰+ min) | artifact: lora_ckpt/lora_full_s{seed}.{json,npz}
# env: RNA_LORA12K_SEEDS="0"  (برای ۲ seed: "0,1" — زمان دو برابر)
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p6l_lora_full",
                 restore="data_paper/*.parquet; emb_paper/*.npy; emb_paper/*_meta.npz; emb_paper/embed_done.json; phys_paper/phys_*.npz; config_paper.json; artifacts_paper/main_ckpt/unet+phys_s*",
                 save="artifacts_paper/lora_ckpt/lora_full_s*"):
        import json, os, re, sys, time
        import numpy as np
        import torch
        import torch.nn as nn
        import plib as P
        importlib.reload(P)
        from peft import LoraConfig, get_peft_model

        AR = P.AR
        STAGE = "p6l_lora_full"
        gate = P.gate_for(STAGE)
        C = P.cfg()
        SEEDS = [int(x) for x in os.environ.get("RNA_LORA12K_SEEDS", "0").split(",")]
        EPOCHS = int(os.environ.get("RNA_EPOCHS", 1 if P.QUICK else 2))
        RANK, LORA_LR = 8, 2e-4
        OUT = AR / "artifacts_paper"; OUT.mkdir(exist_ok=True)
        CK = OUT / "lora_ckpt"; CK.mkdir(exist_ok=True)
        print(f"### {STAGE} START | seeds={SEEDS} epochs={EPOCHS} (n_seq=EPOCHS×train، accum=1 — همان پروتکل P5) device={P.DEV}", flush=True)

        splits = {n: P.Split(n) for n in P.SPLITS}
        dim = splits["train"].dim
        spec = P.ARMS["unet+phys"]
        PROTO = {"epochs": EPOCHS, "acc": 1, "rank": RANK, "lora_lr": LORA_LR, "pw": P.PW, "n_train": len(splits["train"]), "thr_grid": P.THR_GRID,
                 "unet_base": P.UNET_BASE, "backbone": json.load(open(AR / "emb_paper" / "embed_done.json"))["model"]}
        t0 = time.time()
        for seed in SEEDS:
            ck, cz = CK / f"lora_full_s{seed}.json", CK / f"lora_full_s{seed}.npz"
            if ck.exists() and cz.exists() and json.load(open(ck)).get("_proto") == PROTO:
                print(f"[seed {seed}] checkpoint (پروتکل یکسان) — skip", flush=True); continue
            base, tok, name, d2 = P.load_backbone()
            assert d2 == dim, (d2, dim)
            lin = [n for n, m in base.named_modules() if isinstance(m, nn.Linear)]
            idx = [(n, int(re.search(r"\.(\d+)\.", n).group(1))) for n in lin if re.search(r"\.(\d+)\.", n)]
            gate(len(idx) > 0, "لایه‌های شماره‌دار پیدا شد")
            mx = max(i for _, i in idx)
            cands = [n for n, i in idx if re.search(r"(query|value|q_proj|v_proj)", n, re.I) and i >= mx - 3]
            gate(len(cands) > 0, f"{len(cands)} ماژول LoRA در لایه‌های {mx-3}..{mx}")
            torch.manual_seed(seed)
            pm = get_peft_model(base, LoraConfig(r=RANK, lora_alpha=2 * RANK, lora_dropout=0.05, target_modules=cands, bias="none")).to(P.DEV)
            lora_params = [p for p in pm.parameters() if p.requires_grad]
            print(f"  adapter: {sum(p.numel() for p in lora_params)/1e3:.1f}K پارامتر", flush=True)

            def hfn(sp, i, _pm=pm, _tok=tok):
                return P.embed_tokens(_pm, _tok, sp.seqs[i])

            pm.eval()
            with torch.no_grad():
                rel = []
                for i in range(3):
                    a, b = hfn(splits["val"], i), splits["val"].h(i)
                    rel.append(float((a - b).abs().max() / b.abs().mean().clamp(min=1e-6)))
            gate(max(rel) < 0.05, f"هم‌ارزی اولیهٔ LoRA با کش: max rel diff={max(rel):.4f}", hard=False)

            torch.manual_seed(seed); np.random.seed(seed)
            dec = P.build_arm(spec, dim, splits["train"])
            pm.train()
            diag = P.train_decoder(dec, spec, splits["train"], seed, n_seq=EPOCHS * len(splits["train"]), accum=1, tag=f"lora_full s{seed}", log_every=2000,
                                   hfn=lambda sp, i: (pm.train(), hfn(sp, i))[1], extra_groups=[{"params": lora_params, "lr": LORA_LR}])
            pm.eval()
            bnorm = float(sum(p.detach().float().norm() for n, p in pm.named_parameters() if "lora_B" in n))
            gate(diag["min_grad_norm_lora"] is not None and diag["final_grad_norm_lora"] > 0 and bnorm > 0,
                 f"LoRA واقعاً آموزش دید | |g| نهایی={diag['final_grad_norm_lora']:.2e} ‖B‖={bnorm:.3f}")
            rl, cl = P.evaluate_arm(dec, spec, splits, hfn=hfn, with_categories=False)
            rl.update({"arm": "lora_full", "seed": seed, "_proto": PROTO, "lora_diag": {**diag, "B_norm": round(bnorm, 4)}, "elapsed_s": round(time.time() - t0)})
            np.savez(cz, **cl); json.dump(rl, open(ck, "w"), indent=2)
            mf = OUT / "main_ckpt" / f"unet+phys_s{seed}.json"
            ref = json.load(open(mf)) if mf.exists() else None
            msg = f"[seed {seed}] LoRA(۱۲۰۰۰×۳): val={rl['val_f1']} thr={rl['thr']} Test={rl['splits']['test']['f1']} OOD={rl['ood_mean']}"
            if ref: msg += f" | frozen(P5): Test={ref['splits']['test']['f1']} OOD={ref['ood_mean']} | ΔTest={rl['splits']['test']['f1']-ref['splits']['test']['f1']:+.4f} ΔOOD={rl['ood_mean']-ref['ood_mean']:+.4f}"
            print(msg + f" | {time.time()-t0:.0f}s", flush=True)
            del pm, base, dec
            torch.cuda.empty_cache()
        gate(True, "LoRA در مقیاس کامل ذخیره شد (مقایسه با frozen در P12a: ستون «LoRA RiNALMo» در برابر «unet+phys»)", hard=False)
        print(f"### {STAGE} DONE", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست


## P12a — جدول‌های گزارش (وزن توالی / وزن برابر family / tierهای family / سود نسبت به nolm)


In [ ]:
# fmt: off
# ============================================================
# P12a — جدول‌های گزارش مقاله: F1 هر سامانه با (۱) وزن توالی، (۲) وزن برابر هر family/واحد، (۳) به‌تفکیک اندازهٔ family در Test،
#   و (۴) سود هر سامانه نسبت به nolm+phys (شامل تعامل Test−OOD). CI = بوت‌استرپ واحد × seed. CPU، چند دقیقه.
#   سامانه‌ها خودکار پیدا می‌شوند: RNAfold، nolm، unet، unet+phys، randlm، backbone دوم (P5x)، LoRA (P6L).
# artifact: artifacts_paper/report_tables_v5.md و analysis_v5_tables.json
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p12a_tables",
                 restore="config_paper.json; artifacts_paper/leakage.json; artifacts_paper/counts_rnafold.npz; artifacts_paper/main_ckpt/*; artifacts_paper/xp_ckpt/*; artifacts_paper/lora_ckpt/lora_full_s*; data_paper/train.parquet; data_paper/test.parquet; data_paper/levels_*.parquet",
                 save="artifacts_paper/report_tables_v5.md; artifacts_paper/analysis_v5_tables.json"):
        import json, os, pathlib, re, warnings
        import numpy as np
        import pandas as pd
        warnings.filterwarnings("ignore", category=RuntimeWarning)

        AR = pathlib.Path(os.environ.get("RNA_AR", "/content/rna_paper_v4_quick" if os.environ.get("RNA_QUICK", "") == "1" else "/content/rna_paper_v4"))
        OUT = AR / "artifacts_paper"; CK = OUT / "main_ckpt"; XCK = OUT / "xp_ckpt"; LCK = OUT / "lora_ckpt"
        OOD = ["gena", "genc", "genf"]; ALL = ["test"] + OOD
        QK = os.environ.get("RNA_QUICK", "") == "1"
        B = 300 if QK else 4000
        rng = np.random.RandomState(41)
        famc = json.load(open(AR / "config_paper.json"))["family_col"]
        leak = json.load(open(OUT / "leakage.json"))
        trf = pd.read_parquet(AR / "data_paper" / "train.parquet")[famc].astype(str).values
        tef = pd.read_parquet(AR / "data_paper" / "test.parquet")[famc].astype(str).values
        cnt = pd.Series(trf).value_counts(); fin_seq = np.array([int(cnt.get(x, 0)) for x in tef])
        TIERS = [("<10", 0, 9), ("10–49", 10, 49), ("≥50", 50, 10 ** 9)]
        UID, NU, W = {}, {}, {}
        for n in ALL:
            u = tef if n == "test" else pd.read_parquet(AR / "data_paper" / f"levels_{n}.parquet")[leak["splits"][n]["unit"]].astype(str).values
            nm = sorted(set(u)); ix = {x: i for i, x in enumerate(nm)}
            UID[n] = np.array([ix[x] for x in u]); NU[n] = len(nm); W[n] = rng.multinomial(NU[n], np.ones(NU[n]) / NU[n], size=B).astype(float)
        JB = rng.randint(0, 10 ** 6, size=(B, 16))


        def load(files, prefix=""):
            d = {}
            for f in files:
                m = re.search(r"_s(\d+)\.npz$", f.name); key = int(m.group(1)) if m else "det"; z = np.load(f)
                d[key] = {}
                for n in ALL:
                    tp, fp, fn = (z[f"{prefix}{n}_{k}"].astype(float) for k in ("tp", "fp", "fn")); has = (tp + fn) > 0
                    d[key][n] = np.where(has, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), np.nan)
            return d


        SYS = {}
        if (OUT / "counts_rnafold.npz").exists(): SYS["RNAfold-MFE"] = load([OUT / "counts_rnafold.npz"], "mfe_")
        for lab, files in [("nolm+phys", sorted(CK.glob("nolm+phys_s*.npz"))), ("unet (LM، بدون فیزیک)", sorted(CK.glob("unet_s*.npz"))),
                           ("unet+phys (RiNALMo)", sorted(CK.glob("unet+phys_s*.npz"))), ("randlm+phys (LM تصادفی)", sorted(CK.glob("randlm+phys_s*.npz"))),
                           ("b2+phys (backbone دوم)", sorted(XCK.glob("b2+phys_full_s*.npz"))), ("LoRA RiNALMo (۱۲۰۰۰×۳)", sorted(LCK.glob("lora_full_s*.npz")))]:
            if files: SYS[lab] = load(files)
        NOLM = "nolm+phys"
        print("سامانه‌ها (تعداد seed):", {k: len(v) for k, v in SYS.items()}, flush=True)
        assert NOLM in SYS, "نتایج nolm+phys پیدا نشد"


        def ci(b): return [round(float(np.nanpercentile(b, 2.5)), 4), round(float(np.nanpercentile(b, 97.5)), 4)]


        def pval(d):
            d = d[~np.isnan(d)]; return round(float(min(1.0, 2 * (1 + min((d <= 0).sum(), (d >= 0).sum())) / (len(d) + 1))), 4)


        def unit_stat(g, n, mask, equal):
            ok = (mask if mask is not None else np.ones(len(g), bool)) & ~np.isnan(g); U = NU[n]
            su = np.bincount(UID[n][ok], weights=g[ok], minlength=U); nu = np.bincount(UID[n][ok], minlength=U).astype(float)
            if not equal: den, num, pt = W[n] @ nu, W[n] @ su, su.sum() / max(nu.sum(), 1)
            else:
                m = np.where(nu > 0, su / np.maximum(nu, 1), 0.0); has = (nu > 0).astype(float)
                den, num, pt = W[n] @ has, W[n] @ m, (m[nu > 0].mean() if (nu > 0).any() else np.nan)
            return float(pt), np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)


        def combine(per):
            pts = float(np.mean([p[0] for p in per])); S = len(per)
            if S == 1: return pts, per[0][1]
            Pm = np.stack([p[1] for p in per]); return pts, np.take_along_axis(Pm, (JB[:, :S] % S).T, 0).mean(0)


        def f1_stat(sysn, n, mask=None, equal=False):
            ks = sorted(k for k in SYS[sysn] if k != "det") or ["det"]
            return combine([unit_stat(SYS[sysn][k][n], n, mask, equal) for k in ks])


        def gain_stat(A, Bn, n, mask=None, equal=False):
            ks = sorted(set(SYS[A]) & set(SYS[Bn]) - {"det"})
            return combine([unit_stat(SYS[A][k][n] - SYS[Bn][k][n], n, mask, equal) for k in ks]) if ks else (np.nan, np.full(B, np.nan))


        def ood(fn, equal):
            parts = [fn(n, equal) for n in OOD]; return float(np.mean([p[0] for p in parts])), np.mean([p[1] for p in parts], 0)


        tmask = {lab: (fin_seq >= lo) & (fin_seq <= hi) for lab, lo, hi in TIERS}
        cells = lambda pt_b: f"{pt_b[0]:.3f} {ci(pt_b[1])}"
        res = {"n_boot": B, "seeds": {k: sorted(v) for k, v in SYS.items()}}
        md = ["# جدول‌های گزارش (macro-F1؛ CI۹۵٪ = بوت‌استرپ واحد × seed)\n"]

        # ---------- جدول ۱ و ۲: F1 هر سامانه — وزن توالی و وزن برابر هر family/واحد
        for equal, ttl in [(False, "۱) F1 با وزن توالی"), (True, "۲) F1 با وزن برابر هر family/واحد (Test=family، OOD=clan/super_family)")]:
            print(f"\n=== {ttl} ===", flush=True)
            hdr = f"{'سامانه':<28}" + "".join(f"{c:>26}" for c in ["Test", "GenA", "GenC", "GenF", "OOD-mean"])
            print(hdr, flush=True); md += [f"\n## {ttl}\n", "| سامانه | Test | GenA | GenC | GenF | OOD-mean |", "|---|---|---|---|---|---|"]
            key = "equal" if equal else "seq"; res.setdefault("f1", {})[key] = {}
            for s in SYS:
                row = {n: f1_stat(s, n, None, equal) for n in ALL}; om = ood(lambda n, e, s=s: f1_stat(s, n, None, e), equal); row["ood_mean"] = om
                res["f1"][key][s] = {n: {"f1": round(v[0], 4), "ci95": ci(v[1])} for n, v in row.items()}
                print(f"{s:<28}" + "".join(f"{cells(row[n]):>26}" for n in ALL + ["ood_mean"]), flush=True)
                md.append(f"| {s} | " + " | ".join(cells(row[n]) for n in ALL + ["ood_mean"]) + " |")

        # ---------- جدول ۳: F1 در Test به‌تفکیک اندازهٔ family (تعداد نمونه در train)
        print("\n=== ۳) F1 در Test به‌تفکیک اندازهٔ family (نمونه در train) ===", flush=True)
        print(f"{'سامانه':<28}" + "".join(f"{lab + ' (' + str(int(tmask[lab].sum())) + ' seq)':>26}" for lab, _, _ in TIERS), flush=True)
        md += ["\n## ۳) F1 در Test به‌تفکیک اندازهٔ family\n", "| سامانه | " + " | ".join(l for l, _, _ in TIERS) + " |", "|---|---|---|---|"]
        res["f1_tiers"] = {}
        for s in SYS:
            row = {lab: f1_stat(s, "test", tmask[lab], False) for lab, _, _ in TIERS}
            res["f1_tiers"][s] = {lab: {"f1": round(v[0], 4), "ci95": ci(v[1])} for lab, v in row.items()}
            print(f"{s:<28}" + "".join(f"{cells(row[lab]):>26}" for lab, _, _ in TIERS), flush=True)
            md.append(f"| {s} | " + " | ".join(cells(row[lab]) for lab, _, _ in TIERS) + " |")

        # ---------- جدول ۴: سود هر سامانه نسبت به nolm+phys
        print(f"\n=== ۴) سود نسبت به {NOLM} (Δ F1) — وزن توالی/برابر، tierهای family، OOD و تعامل (Test − OOD) ===", flush=True)
        cols = ["Test(seq)", "Test(equal)"] + [f"Test {lab}" for lab, _, _ in TIERS] + ["OOD(seq)", "OOD(equal)", "تعامل(seq)"]
        print(f"{'سامانه':<28}" + "".join(f"{c:>24}" for c in cols), flush=True)
        md += ["\n## ۴) سود نسبت به nolm+phys\n", "| سامانه | " + " | ".join(cols) + " |", "|---|" + "---|" * len(cols)]
        res["gain_vs_nolm"] = {}
        for s in SYS:
            if s in (NOLM, "RNAfold-MFE"): continue
            g_t = gain_stat(s, NOLM, "test"); g_te = gain_stat(s, NOLM, "test", None, True); tiers = [gain_stat(s, NOLM, "test", tmask[lab]) for lab, _, _ in TIERS]
            g_o = ood(lambda n, e, s=s: gain_stat(s, NOLM, n, None, e), False); g_oe = ood(lambda n, e, s=s: gain_stat(s, NOLM, n, None, e), True)
            ix = g_t[1] - g_o[1]; vals = [g_t, g_te] + tiers + [g_o, g_oe, (g_t[0] - g_o[0], ix)]
            res["gain_vs_nolm"][s] = {c: {"delta": round(v[0], 4), "ci95": ci(v[1])} for c, v in zip(cols, vals)}; res["gain_vs_nolm"][s]["interaction_p"] = pval(ix)
            print(f"{s:<28}" + "".join(f"{cells(v):>24}" for v in vals), flush=True)
            md.append(f"| {s} | " + " | ".join(cells(v) for v in vals) + " |")
        (OUT / "report_tables_v5.md").write_text("\n".join(md), encoding="utf-8")
        json.dump(res, open(OUT / "analysis_v5_tables.json", "w"), indent=2, ensure_ascii=False)
        print("\nتوجه: ستون‌های «equal» و tierها مبنای گزارش مقاله‌اند؛ سامانه‌های تک‌seed (مثلاً LoRA) CI ناقص دارند (فقط واحد، نه seed).", flush=True)
        print("ذخیره شد:", OUT / "report_tables_v5.md", "و", OUT / "analysis_v5_tables.json", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست


## P12b — دوز–پاسخ (سقف‌ها) و آزمون ضربی (پیش‌آموزش × مواجهه)


In [ ]:
# fmt: off
# ============================================================
# P12b — دوز–پاسخ و آزمون ضربی: (الف) سود LM بر حسب سقف per-family (۳، ۱۰، ۵۰، full) به‌تفکیک tier و با شیب CI؛
#   (ب) فاکتوریل ۲×۲ «پیش‌آموزش (LM − LM تصادفی) × مواجهه (cap در برابر rand)» با DiD. CPU. بعد از P5x.
# artifact: artifacts_paper/analysis_v5_dose_factorial.json
# ============================================================
# ---------------- Drive (v6): بازیابی از Drive → skip اگر کامل بود → ذخیره در Drive ----------------
import sys
sys.path.insert(0, "/content/rna_plib")
import importlib
import drive as D
importlib.reload(D)
if "restore" not in __import__("inspect").signature(D.stage).parameters:
    raise SystemExit("DRIVE: drive.py قدیمی است — نوتبوک را دوباره آپلود کنید")
try:
    with D.stage("p12b_dose_factorial",
                 restore="config_paper.json; artifacts_paper/leakage.json; artifacts_paper/main_ckpt/*; artifacts_paper/cap_ckpt/*; data_paper/train.parquet; data_paper/test.parquet; data_paper/levels_*.parquet",
                 save="artifacts_paper/analysis_v5_dose_factorial.json"):
        import json, os, pathlib, re, warnings
        import numpy as np
        import pandas as pd
        warnings.filterwarnings("ignore", category=RuntimeWarning)

        AR = pathlib.Path(os.environ.get("RNA_AR", "/content/rna_paper_v4_quick" if os.environ.get("RNA_QUICK", "") == "1" else "/content/rna_paper_v4"))
        OUT = AR / "artifacts_paper"; CK = OUT / "main_ckpt"; CCK = OUT / "cap_ckpt"
        OOD = ["gena", "genc", "genf"]; ALL = ["test"] + OOD
        QK = os.environ.get("RNA_QUICK", "") == "1"
        B, MINN = (300, 3) if QK else (4000, 3)
        rng = np.random.RandomState(53)
        famc = json.load(open(AR / "config_paper.json"))["family_col"]
        leak = json.load(open(OUT / "leakage.json"))
        trf = pd.read_parquet(AR / "data_paper" / "train.parquet")[famc].astype(str).values
        tef = pd.read_parquet(AR / "data_paper" / "test.parquet")[famc].astype(str).values
        cnt_final = pd.Series(trf).value_counts(); names, inv = np.unique(tef, return_inverse=True); NUt = len(names)
        UID, NU, WB = {}, {}, {}
        for n in OOD:
            u = pd.read_parquet(AR / "data_paper" / f"levels_{n}.parquet")[leak["splits"][n]["unit"]].astype(str).values
            nm = sorted(set(u)); ix = {x: i for i, x in enumerate(nm)}
            UID[n] = np.array([ix[x] for x in u]); NU[n] = len(nm); WB[n] = rng.multinomial(NU[n], np.ones(NU[n]) / NU[n], size=B).astype(float)


        def load_set(files):
            d = {}
            for f in files:
                s = int(re.search(r"_s(\d+)\.npz$", f.name).group(1)); z = np.load(f); d[s] = {}
                for n in ALL:
                    tp, fp, fn = (z[f"{n}_{k}"].astype(float) for k in ("tp", "fp", "fn")); has = (tp + fn) > 0
                    d[s][n] = np.where(has, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), np.nan)
            return d


        def ci(b): return [round(float(np.nanpercentile(b, 2.5)), 4), round(float(np.nanpercentile(b, 97.5)), 4)]


        def pval(d):
            d = d[~np.isnan(d)]; return round(float(min(1.0, 2 * (1 + min((d <= 0).sum(), (d >= 0).sum())) / (len(d) + 1))), 4)


        def cell_files(arm, cond, cap):
            return sorted(CK.glob(f"{arm}_s*.npz")) if cond == "full" else sorted(CCK.glob(f"{arm}_c{cap}_{cond}_s*.npz"))


        def mean_f1(cell, seeds, n): return np.nanmean(np.stack([cell[s][n] for s in seeds]), 0)


        def ood_boot(arrs_a, arrs_b=None):
            """میانگین OOD (واحد-بوت‌استرپ) برای F1 (یا اختلاف F1) — arrs: {split: per-seq}"""
            pts, bs = [], []
            for n in OOD:
                g = arrs_a[n] - (arrs_b[n] if arrs_b is not None else 0); ok = ~np.isnan(g)
                su = np.bincount(UID[n][ok], weights=g[ok], minlength=NU[n]); nn = np.bincount(UID[n][ok], minlength=NU[n]).astype(float)
                pts.append(su.sum() / nn.sum()); bs.append((WB[n] @ su) / np.maximum(WB[n] @ nn, 1))
            return float(np.mean(pts)), np.mean(bs, 0)


        caps = sorted({int(re.search(r"_c(\d+)_cap_", f.name).group(1)) for f in CCK.glob("unet+phys_c*_cap_s*.npz")})
        assert caps, "cap_ckpt خالی است — P5c/P5x را اجرا کنید"
        TIERS = [("<10", 0, 9), ("10–49", 10, 49), ("≥50", 50, 10 ** 9), ("همه(وزن توالی)", 0, 10 ** 9), ("همه(وزن برابر)", 0, 10 ** 9)]
        res = {"caps": caps, "n_boot": B, "dose": {}, "factorial": {}}

        # ---- مجموعه familyهای قابل‌استفاده (≥3 توالی معتبر) و اندیس‌های بوت‌استرپ مشترک بین همهٔ capها
        SYSC = {}
        for cap in caps:
            for arm in ("unet+phys", "nolm+phys", "randlm+phys"):
                for cond in ("cap", "rand", "full"):
                    fs = cell_files(arm, cond, cap)
                    if fs: SYSC[(arm, cond, cap)] = load_set(fs)
        ref = next(iter(SYSC.values())); s0 = sorted(ref)[0]
        ok0 = ~np.isnan(ref[s0]["test"]); nval = np.bincount(inv[ok0], minlength=NUt).astype(float); keep = np.where(nval >= MINN)[0]
        w = nval[keep]; fin = np.array([int(cnt_final.get(names[f], 0)) for f in keep]); pos = {f: i for i, f in enumerate(keep)}
        masks = {lab: (fin >= lo) & (fin <= hi) for lab, lo, hi in TIERS}
        BI = {lab: rng.randint(0, int(m.sum()), (B, int(m.sum()))) for lab, m in masks.items()}
        print(f"familyهای Test با ≥{MINN} توالی: {len(keep)} | capها: {caps}", flush=True)


        def fam_vec(arr):
            ok = ~np.isnan(arr); su = np.bincount(inv[ok], weights=arr[ok], minlength=NUt); nn = np.bincount(inv[ok], minlength=NUt).astype(float)
            return (su / np.maximum(nn, 1))[keep]


        def tier_boot(v, lab):
            ix = np.where(masks[lab])[0]; ww = np.ones(len(ix)) if "برابر" in lab else w[ix]; bi = BI[lab]
            return float((ww * v[ix]).sum() / ww.sum()), (ww[bi] * v[ix][bi]).sum(1) / ww[bi].sum(1)


        fmt = lambda t: f"{t[0]:+.3f} {ci(t[1])}"

        # =============== الف) دوز–پاسخ: سود LM (unet+phys − nolm+phys) بر حسب سقف
        print("\n=== الف) دوز–پاسخ: سود LM بر حسب سقف per-family (cap) — «cap»=سقف‌دار، «rand»=تصادفی هم‌اندازه، «full»=train کامل ===", flush=True)
        curve = {}
        for lab, _, _ in TIERS:
            print(f"\n-- {lab}  (fam={int(masks[lab].sum())})", flush=True)
            print(f"   {'cap':<6}{'N_cap':>7}{'seeds':>7} | {'سود در شرط cap':<32}{'سود در شرط rand':<32}", flush=True)
            curve[lab] = []
            for cap in caps:
                need = [("unet+phys", c, cap) for c in ("cap", "rand", "full")] + [("nolm+phys", c, cap) for c in ("cap", "rand", "full")]
                if any(k not in SYSC for k in need): continue
                seeds = sorted(set.intersection(*[set(SYSC[k]) for k in need]))
                if not seeds: continue
                g = {c: fam_vec(mean_f1(SYSC[("unet+phys", c, cap)], seeds, "test")) - fam_vec(mean_f1(SYSC[("nolm+phys", c, cap)], seeds, "test")) for c in ("cap", "rand", "full")}
                tb = {c: tier_boot(g[c], lab) for c in g}
                Ncap = json.load(open(next(CCK.glob(f"unet+phys_c{cap}_cap_s*.json"))))["N"]
                print(f"   {cap:<6}{Ncap:>7}{len(seeds):>7} | {fmt(tb['cap']):<32}{fmt(tb['rand']):<32}", flush=True)
                curve[lab].append((cap, tb)); res["dose"].setdefault(lab, {})[str(cap)] = {"N_cap": int(Ncap), "seeds": seeds, **{c: {"gain": round(tb[c][0], 4), "ci95": ci(tb[c][1])} for c in tb}}
            if curve[lab]:
                tbf = curve[lab][-1][1]["full"]; print(f"   {'full':<6}{'':>7}{'':>7} | {fmt(tbf):<32}", flush=True)
                res["dose"][lab]["full"] = {"gain": round(tbf[0], 4), "ci95": ci(tbf[1])}
            if len(curve[lab]) >= 2:
                xs = np.log10([c for c, _ in curve[lab]] + [max(cnt_final.max(), 2)]); xc = xs - xs.mean()
                Ys = [tb["cap"] for _, tb in curve[lab]] + [curve[lab][-1][1]["full"]]
                sl = lambda Y: (xc[:, None] * (Y - Y.mean(0))).sum(0) / (xc ** 2).sum()
                sb = sl(np.stack([y[1] for y in Ys])); sp = float(sl(np.array([[y[0]] for y in Ys]))[0])
                res["dose"][lab]["slope_per_decade_cap"] = {"value": round(sp, 4), "ci95": ci(sb), "p": pval(sb)}
                print(f"   شیب سود به‌ازای ۱۰ برابر شدن سقف (با 'full' در log10 بیشترین مواجهه={int(cnt_final.max())}): {sp:+.4f} {ci(sb)} p={pval(sb)}", flush=True)

        # =============== ب) آزمون ضربی: پیش‌آموزش (LM − LM تصادفی) × مواجهه (cap در برابر rand)
        print("\n=== ب) آزمون ضربی (فاکتوریل 2×2): دانش پیش‌آموزش × مواجهه ===", flush=True)
        print("    PG = F1(unet+phys) − F1(randlm+phys)؛ DiD = PG(مواجههٔ بالا=rand) − PG(مواجههٔ پایین=cap). اگر DiD>0: سود پیش‌آموزش به مواجهه نیاز دارد (ضربی).", flush=True)
        for cap in caps:
            need = [(a, c, cap) for a in ("unet+phys", "randlm+phys") for c in ("cap", "rand")]
            if any(k not in SYSC for k in need): continue
            seeds = sorted(set.intersection(*[set(SYSC[k]) for k in need]))
            if not seeds: continue
            print(f"\n-- cap={cap} | seeds={seeds}", flush=True); res["factorial"][str(cap)] = {"seeds": seeds}
            V = {k[:2]: fam_vec(mean_f1(SYSC[k], seeds, "test")) for k in need}
            print(f"   {'tier':<16} | {'PG مواجههٔ بالا':<26}{'PG مواجههٔ پایین':<26}{'DiD (بالا−پایین)':<30} | {'اثر مواجهه روی LM':<24}{'اثر مواجهه روی LM تصادفی':<26}", flush=True)
            for lab, _, _ in TIERS:
                hi = tier_boot(V[("unet+phys", "rand")] - V[("randlm+phys", "rand")], lab); lo = tier_boot(V[("unet+phys", "cap")] - V[("randlm+phys", "cap")], lab)
                eA = tier_boot(V[("unet+phys", "rand")] - V[("unet+phys", "cap")], lab); eB = tier_boot(V[("randlm+phys", "rand")] - V[("randlm+phys", "cap")], lab)
                d = hi[1] - lo[1]; dd = (hi[0] - lo[0], d)
                res["factorial"][str(cap)][lab] = {"PG_high": [round(hi[0], 4), ci(hi[1])], "PG_low": [round(lo[0], 4), ci(lo[1])], "DiD": [round(dd[0], 4), ci(d), pval(d)],
                                                   "exposure_effect_LM": [round(eA[0], 4), ci(eA[1])], "exposure_effect_randLM": [round(eB[0], 4), ci(eB[1])]}
                print(f"   {lab:<16} | {fmt(hi):<26}{fmt(lo):<26}{dd[0]:+.3f} {ci(d)} p={pval(d):<7} | {fmt(eA):<24}{fmt(eB):<26}", flush=True)
            # OOD (واحد-بوت‌استرپ)
            aR, aC = {n: mean_f1(SYSC[("unet+phys", "rand", cap)], seeds, n) for n in OOD}, {n: mean_f1(SYSC[("unet+phys", "cap", cap)], seeds, n) for n in OOD}
            bR, bC = {n: mean_f1(SYSC[("randlm+phys", "rand", cap)], seeds, n) for n in OOD}, {n: mean_f1(SYSC[("randlm+phys", "cap", cap)], seeds, n) for n in OOD}
            hi, lo = ood_boot(aR, bR), ood_boot(aC, bC); d = hi[1] - lo[1]
            res["factorial"][str(cap)]["OOD"] = {"PG_high": [round(hi[0], 4), ci(hi[1])], "PG_low": [round(lo[0], 4), ci(lo[1])], "DiD": [round(hi[0] - lo[0], 4), ci(d), pval(d)]}
            print(f"   {'OOD (میانگین)':<16} | {fmt(hi):<26}{fmt(lo):<26}{hi[0] - lo[0]:+.3f} {ci(d)} p={pval(d)}", flush=True)

        print("\nراهنمای تفسیر:", flush=True)
        print(" • الف: اگر سود familyهای ≥۵۰ با بالا رفتن سقف یکنواخت بالا می‌رود (شیب مثبت معنادار) ⇒ دوز–پاسخ؛ شرط rand باید به سود full نزدیک بماند (کنترل اندازه).", flush=True)
        print(" • ب: DiD>0 معنادار و «اثر مواجهه روی LM تصادفی»≈۰ ⇒ اثر ضربی: مواجهه فقط وقتی کمک می‌کند که LM پیش‌آموزش‌دیده باشد.", flush=True)
        print(" • هشدار: tier ≥۵۰ فقط ≈۱۱ family؛ آزمایش اکتشافی؛ capهای ۳ و ۵۰ معمولاً فقط ۲ seed دارند.", flush=True)
        json.dump(res, open(OUT / "analysis_v5_dose_factorial.json", "w"), indent=2, ensure_ascii=False)
        print("ذخیره شد:", OUT / "analysis_v5_dose_factorial.json", flush=True)

except D.StageSkipped:
    pass   # مرحله از Drive بازگردانی شد؛ خطا نیست
